# Bogura drive test — complete code

One notebook for the Idle and Active report. The cells above the last one only define the program. The last cell builds the workbook.

Keep this notebook in the same folder as `Borga_Idle_V2.0.7z`, `Borga_Active_mode_11Nov26.7z`, `System DUMP.xlsb`, and the `web` folder. Start Jupyter from that folder.

```bash
pip install pandas numpy matplotlib openpyxl pillow pyxlsb cairosvg python-pptx py7zr
```

Idle stays `Borga_Idle_V2.0`. Active is Robi, M-PCell, strongest RSRP at each location, from `Borga_Active_mode_11Nov26`.


In [ ]:
# ------------------------------------------------------------------------
# 1. Libraries and folders
# Every library the four scripts use.
# In Jupyter there is no script path, so the folder is the notebook's working directory.
# ------------------------------------------------------------------------

from __future__ import annotations

import hashlib
import io
import json
import math
import re
import shutil
import subprocess
import sys
import time
import urllib.request
import zipfile
import xml.etree.ElementTree as ET
from collections import deque
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from io import BytesIO
from pathlib import Path
from shutil import which

import cairosvg
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D
from matplotlib.patches import Circle, Ellipse, Patch, PathPatch, Wedge
from matplotlib.path import Path as MPath
from matplotlib.ticker import MultipleLocator
from PIL import Image
from pptx import Presentation
from pptx.dml.color import RGBColor
from pptx.enum.shapes import MSO_SHAPE
from pptx.enum.text import MSO_ANCHOR, PP_ALIGN
from pptx.oxml.ns import qn
from pptx.util import Emu, Inches, Pt

from openpyxl import Workbook, load_workbook
from openpyxl.chart import BarChart, LineChart, Reference
from openpyxl.chart.layout import Layout, ManualLayout
from openpyxl.chart.marker import Marker
from openpyxl.chart.text import RichText
from openpyxl.drawing.image import Image as XLImage
from openpyxl.drawing.text import (
    CharacterProperties,
    Paragraph,
    ParagraphProperties,
    RichTextProperties,
)
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.worksheet import Worksheet

try:
    ROOT = Path(__file__).resolve().parent
except NameError:
    ROOT = Path.cwd()

EXTRACT_DIR = ROOT / "extracted"
OUTPUT_DIR = ROOT / "output"
PLOTS_DIR = OUTPUT_DIR / "plots"
print("Working folder:", ROOT)


## 2. Drive files, columns, and the colour bins

Idle archive, Active archive, and the system dump. Scanner columns are renamed to RSRP, RSRQ, and SINR.
Maps and bad spots use the same 50 m bins. The colour bins are the coverage legend: RSRP red is below −120 dBm.


In [ ]:
# ------------------------------------------------------------------------
# 2. Drive files, columns, and the colour bins
# Idle archive, Active archive, and the system dump. Scanner columns are renamed to RSRP, RSRQ, and SINR.
# Maps and bad spots use the same 50 m bins. The colour bins are the coverage legend: RSRP red is below −120 dBm.
# ------------------------------------------------------------------------

IDLE_ARCHIVE = ROOT / "Borga_Idle_V2.0.7z"
IDLE_DIR = EXTRACT_DIR / "Borga_Idle_V2.0"
MERGED_CSV = OUTPUT_DIR / "Bogura_merged.csv.gz"
SUMMARY_MD = OUTPUT_DIR / "analysis_summary.md"
# Cell-level system dump. One site pie is built from its Sector column, not from the
# old node-level Physical_Site_Database_V1.0.xlsb.
SITE_DB_PATH = ROOT / "System DUMP.xlsb"
# Active Mode UE log. The 11 Nov 2026 archive replaces Borga_Active_mode.csv.
# It holds one CSV per collection, the same shape as the idle export. Robi,
# Grameenphone and Banglalink are in the files; the report draws Robi only.
ACTIVE_ARCHIVE = ROOT / "Borga_Active_mode_11Nov26.7z"
ACTIVE_DIR = EXTRACT_DIR / "Borga_Active_mode_11Nov26"
ACTIVE_MERGED = OUTPUT_DIR / "Bogura_active_merged.csv.gz"
# The CSV stores a clock ("8:40:30 AM") and no calendar date. Pin that clock
# so a later rebuild does not move the timestamp onto the day the script runs.
ACTIVE_CLOCK_ANCHOR = pd.Timestamp("1970-01-01")
# Idle maps are the Robi scanner. The active file also carries a Grameenphone UE on the
# same drive; only Robi is drawn so Active sits beside Idle for the same network.
ACTIVE_OPERATOR = "Robi"
# Primary serving cell. M-SCell1 and M-SCell2 are carrier-aggregation neighbours
# and are left out of the Active maps.
ACTIVE_CELL_TYPE = "M-PCell"

SCANNER_RENAME = {
    "NB RSRP": "RSRP",
    "NB RSRQ": "RSRQ",
    "NB RS SINR": "SINR",
}

COLUMNS = [
    "Time",
    "RSRP",
    "RSRQ",
    "SINR",
    "Longitude",
    "Latitude",
    "Cell Id",
    "DL EARFCN",
]

# LTE downlink EARFCN ranges (3GPP TS 36.101).
EARFCN_BANDS = {
    1: (0, 599, "B1 2100 MHz"),
    3: (1200, 1949, "B3 1800 MHz"),
    8: (3450, 3799, "B8 900 MHz"),
    40: (38650, 39649, "B40 2300 MHz"),
    41: (39650, 41589, "B41 2500 MHz"),
}

# The all-band best-server track is ~61k points; plot it whole so no poor sample
# is dropped from a map that bad spots are drawn on.
MAP_MAX_POINTS = 150000
# Drive-test bin: maps are coloured by the median KPI of each 50 m of road, and
# bad spots are judged on the same bins, so a circle can only sit on colour that
# the map actually shows.
MAP_BIN_M = 50.0

# One scanner sweep of all bands takes ~3 s; bound it in distance too so a fast
# stretch of road does not merge separate locations into one best-server sample.
BEST_SERVER_SWEEP_S = 4.0
BEST_SERVER_SWEEP_M = 40.0

# Operator layer names used on the RSRP / RSRQ / SINR sheets.
REPORT_LAYERS = ("L900", "L1800", "L2100", "L2600")
LAYER_EARFCN_RANGES = {
    "L900": ((3450, 3799),),
    "L1800": ((1200, 1949),),
    "L2100": ((0, 599),),
    "L2600": ((2750, 3449), (39650, 41589)),  # Band 7 FDD 2600 + Band 41 TDD 2500/2600
}

# Coverage-map legend (Excel Page 1 colour table). Best bin is listed first.
# Colours match the standard Office fills used in that table.
LEGEND_BLUE = "#0070C0"
LEGEND_DARK_GREEN = "#00B050"
LEGEND_LIGHT_GREEN = "#92D050"
LEGEND_YELLOW = "#FFFF00"
LEGEND_MAGENTA = "#FF00FF"
LEGEND_RED = "#FF0000"
LEGEND_CYAN = "#00B0F0"
LEGEND_ORANGE = "#FF6600"

# Four bins from the coverage_4g_data1 RSRP legend. Best is listed first.
# A value below -150 stays in the red bin; a value above 0 stays in the dark-green bin.
MAP_RSRP_LABELS = [
    "-95 to 0",
    "-110 to -95",
    "-115 to -110",
    "-120 to -115",
    "-150 to -120",
]
# Light red includes -115 and -120. Yellow is only better than -115.
MAP_RSRP_THRESHOLDS = [-95, -110, -115, -120]
MAP_RSRP_COLORS = ["#008000", "#00FF00", "#FFFF00", "#FF6666", "#FF0000"]

# Source table listed RSRQ worst-first and wrote "X < -5" for the blue bin.
# Blue is the best colour, so that row is X >= -5; red is X < -20.
# Three bins from the coverage_4g_data1 RSRQ legend. Best is listed first.
# A value below -30 stays in the red bin; a value above 0 stays in the green bin.
MAP_RSRQ_LABELS = [
    "-10 to 0",
    "-17 to -10",
    "-30 to -17",
]
MAP_RSRQ_THRESHOLDS = [-10, -17]
MAP_RSRQ_COLORS = ["#00FF00", "#FFFF00", "#FF0000"]

MAP_SINR_LABELS = [
    "15 <= X < Max",
    "10 <= X < 15",
    "5 <= X < 10",
    "0 <= X < 5",
    "-5 <= X < 0",
    "Min < X < -5",
]
MAP_SINR_THRESHOLDS = [15, 10, 5, 0, -5]
MAP_SINR_COLORS = [
    LEGEND_BLUE,
    LEGEND_DARK_GREEN,
    LEGEND_CYAN,
    LEGEND_YELLOW,
    LEGEND_ORANGE,
    LEGEND_RED,
]

# Cover / summary tables use the same bins as the maps.
RSRP_LABELS = MAP_RSRP_LABELS
RSRP_COLORS = MAP_RSRP_COLORS


## 3. Colour a KPI into a bin

Turn an RSRP, RSRQ, or SINR value into the legend class used on the maps and the cover counts.


In [ ]:
# ------------------------------------------------------------------------
# 3. Colour a KPI into a bin
# Turn an RSRP, RSRQ, or SINR value into the legend class used on the maps and the cover counts.
# ------------------------------------------------------------------------

def _map_class(series: pd.Series, thresholds: list[float], labels: list[str]) -> pd.Series:
    """Assign legend labels; thresholds are inclusive lower bounds, best first."""
    v = pd.to_numeric(series, errors="coerce")
    classified = np.select(
        [v >= t for t in thresholds],
        labels[:-1],
        default=labels[-1],
    )
    return pd.Series(classified, index=v.index).where(v.notna(), other=pd.NA)


def rsrp_map_class(series: pd.Series) -> pd.Series:
    """Classify RSRP. Light red starts at -115 and includes -115 and -120."""
    v = pd.to_numeric(series, errors="coerce")
    classified = np.select(
        [v >= -95, v >= -110, v > -115, v >= -120],
        MAP_RSRP_LABELS[:-1],
        default=MAP_RSRP_LABELS[-1],
    )
    return pd.Series(classified, index=v.index).where(v.notna(), other=pd.NA)


def rsrq_map_class(series: pd.Series) -> pd.Series:
    return _map_class(series, MAP_RSRQ_THRESHOLDS, MAP_RSRQ_LABELS)


def sinr_map_class(series: pd.Series) -> pd.Series:
    return _map_class(series, MAP_SINR_THRESHOLDS, MAP_SINR_LABELS)


MAP_CLASSIFIERS = {
    "RSRP": rsrp_map_class,
    "RSRQ": rsrq_map_class,
    "SINR": sinr_map_class,
}


def rsrp_range_counts(series: pd.Series) -> pd.Series:
    return rsrp_map_class(series).value_counts(dropna=True).reindex(MAP_RSRP_LABELS, fill_value=0)


def hide_map_axes(ax) -> None:
    """Bare coverage map: no lon/lat ticks, matching the RSRP snapshot style."""
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(labelbottom=False, labelleft=False, length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)


## 4. Site database and the sector pies

Read System DUMP.xlsb. One site pie is one site. Each arm is one sector, using the most common azimuth of that sector.
The arms are drawn from the site centre.


In [ ]:
# ------------------------------------------------------------------------
# 4. Site database and the sector pies
# Read System DUMP.xlsb. One site pie is one site. Each arm is one sector, using the most common azimuth of that sector.
# The arms are drawn from the site centre.
# ------------------------------------------------------------------------

# RF-planner three-arm site widget: outline only, no fill inside the petals.
SITE_PIE_EDGE = "#2B2B2B"
SITE_PIE_ARM_WIDTH_FRAC = 0.16
SITE_PIE_RADIUS_KM = 0.90
_SITE_CELLS: pd.DataFrame | None = None
_SERVING_INDEX: tuple[dict, dict] | None = None
_SERVING_LIST: list[dict] | None = None
_SERVING_GRID: dict[tuple[int, int], list[dict]] | None = None


def _most_common_azimuth(series: pd.Series) -> float:
    """Mode of azimuth; ties keep the first most-frequent value."""
    s = pd.to_numeric(series, errors="coerce").dropna()
    if s.empty:
        return float("nan")
    rounded = s.round().astype(int)
    return float(rounded.value_counts().idxmax())


def load_site_sectors(path: Path | None = None, layer: str | None = None) -> pd.DataFrame:
    """One row per site+sector: most-common azimuth when a sector has many cells."""
    global _SITE_CELLS
    if _SITE_CELLS is None:
        db_path = path or SITE_DB_PATH
        if not db_path.exists():
            _SITE_CELLS = pd.DataFrame()
            return _SITE_CELLS
        if db_path.suffix.lower() == ".xlsb":
            from pyxlsb import open_workbook

            rows: list[list] = []
            cols: list[str] = []
            with open_workbook(str(db_path)) as wb:
                sheet = wb.get_sheet(wb.sheets[0])
                for i, row in enumerate(sheet.rows()):
                    vals = [c.v for c in row]
                    if i == 0:
                        cols = [str(v) for v in vals]
                        continue
                    rows.append(vals)
            cells = pd.DataFrame(rows, columns=cols)
        else:
            cells = pd.read_excel(db_path, sheet_name=0, engine="openpyxl")
        cells = cells.rename(columns={c: c.strip() for c in cells.columns})
        if "Site" in cells.columns and "SiteName" not in cells.columns:
            cells = cells.rename(columns={"Site": "SiteName"})
        for col in ("Lat", "Lon", "Azimuth", "EARFCN"):
            if col in cells.columns:
                cells[col] = pd.to_numeric(cells[col], errors="coerce")
        need = {"SiteName", "Sector", "Lat", "Lon", "Azimuth"}
        if not need.issubset(cells.columns):
            _SITE_CELLS = pd.DataFrame()
            return _SITE_CELLS
        cells = cells.dropna(subset=["SiteName", "Sector", "Lat", "Lon"])
        if "EARFCN" in cells.columns:
            cells["layer"] = cells["EARFCN"].map(earfcn_to_layer)
        else:
            cells["layer"] = ""
        _SITE_CELLS = cells
    cells = _SITE_CELLS
    if cells.empty:
        return cells
    if layer:
        cells = cells[cells["layer"] == layer]
    grouped = (
        cells.groupby(["SiteName", "Sector"], sort=False)
        .agg(
            lat=("Lat", "median"),
            lon=("Lon", "median"),
            azimuth=("Azimuth", _most_common_azimuth),
            n_cells=("Sector", "size"),
        )
        .reset_index()
    )
    grouped["n_sectors"] = grouped.groupby("SiteName")["Sector"].transform("nunique")
    return grouped.dropna(subset=["lat", "lon", "azimuth"])


def _sites_near_drive(sectors: pd.DataFrame, drive: pd.DataFrame, radius_km: float = 2.5) -> pd.DataFrame:
    """Keep sites within about radius_km of a logged drive sample."""
    if sectors.empty or drive.empty:
        return sectors
    grid = 0.01  # ~1.1 km
    lat = pd.to_numeric(drive["Latitude"], errors="coerce")
    lon = pd.to_numeric(drive["Longitude"], errors="coerce")
    mask = lat.notna() & lon.notna()
    cells = set(
        zip(
            np.floor(lat[mask] / grid).astype(int),
            np.floor(lon[mask] / grid).astype(int),
        )
    )
    steps = max(1, int(round(radius_km / (grid * 111.32))))
    expanded: set[tuple[int, int]] = set()
    for y, x in cells:
        for dy in range(-steps, steps + 1):
            for dx in range(-steps, steps + 1):
                if dy * dy + dx * dx <= steps * steps + 1:
                    expanded.add((y + dy, x + dx))
    sy = np.floor(sectors["lat"] / grid).astype(int)
    sx = np.floor(sectors["lon"] / grid).astype(int)
    keep = [(int(y), int(x)) in expanded for y, x in zip(sy, sx)]
    return sectors.loc[keep].copy()


def _arm_path(lat: float, lon: float, azimuth: float, length_km: float, width_frac: float = SITE_PIE_ARM_WIDTH_FRAC) -> MPath:
    """Hollow rectangular arm along azimuth (0 = north). Inner edge is left open so three arms form a Y."""
    r = length_km / 111.32
    half_w = r * float(width_frac) / 2.0
    az = np.radians(float(azimuth) % 360.0)
    ux, uy = np.sin(az), np.cos(az)
    px, py = np.cos(az), -np.sin(az)
    inner_l = (lon - half_w * px, lat - half_w * py)
    outer_l = (lon + r * ux - half_w * px, lat + r * uy - half_w * py)
    outer_r = (lon + r * ux + half_w * px, lat + r * uy + half_w * py)
    inner_r = (lon + half_w * px, lat + half_w * py)
    return MPath([inner_l, outer_l, outer_r, inner_r], [MPath.MOVETO, MPath.LINETO, MPath.LINETO, MPath.LINETO])


def _pie_radius_km(ax, default_km: float = SITE_PIE_RADIUS_KM) -> float:
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    span_km = min((xlim[1] - xlim[0]) * 111.32, (ylim[1] - ylim[0]) * 111.32)
    if span_km <= 0:
        return default_km
    return float(np.clip(span_km * 0.012, 0.35, 1.15))


def draw_site_pies(
    ax,
    drive: pd.DataFrame,
    radius_km: float | None = None,
    layer: str | None = None,
) -> list:
    """Draw outline-only three-arm site pies (grouped by sector name). No labels, no fill."""
    sectors = load_site_sectors(layer=layer)
    if sectors.empty:
        return []
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    vis = sectors[sectors["lat"].between(ylim[0], ylim[1]) & sectors["lon"].between(xlim[0], xlim[1])]
    vis = _sites_near_drive(vis, drive, radius_km=2.5)
    if vis.empty:
        return []
    radius = _pie_radius_km(ax) if radius_km is None else float(radius_km)
    for _site_name, part in vis.groupby("SiteName", sort=False):
        part = part.drop_duplicates(subset=["Sector"])
        lat = float(part["lat"].median())
        lon = float(part["lon"].median())
        for az in part["azimuth"].tolist():
            ax.add_patch(
                PathPatch(
                    _arm_path(lat, lon, float(az), radius),
                    facecolor="none",
                    edgecolor=SITE_PIE_EDGE,
                    linewidth=0.75,
                    joinstyle="miter",
                    zorder=3,
                )
            )
    return []


def _azimuth_to_mpl(azimuth: float) -> float:
    """Compass azimuth (0 = north, clockwise) as a matplotlib angle (0 = east, CCW)."""
    return (90.0 - float(azimuth)) % 360.0


def _bearing_deg(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Compass bearing from point 1 to point 2, degrees clockwise from north."""
    dlon = np.radians(lon2 - lon1)
    lat1r, lat2r = np.radians(lat1), np.radians(lat2)
    y = np.sin(dlon) * np.cos(lat2r)
    x = np.cos(lat1r) * np.sin(lat2r) - np.sin(lat1r) * np.cos(lat2r) * np.cos(dlon)
    return float((np.degrees(np.arctan2(y, x)) + 360.0) % 360.0)


def _ang_diff(a: float, b: float) -> float:
    return abs((float(a) - float(b) + 180.0) % 360.0 - 180.0)


def _sector_wedges(part: pd.DataFrame) -> list[tuple[str, float, float, float]]:
    """One clockwise slice per sector: name, azimuth, clockwise start, clockwise end."""
    rows = part.dropna(subset=["azimuth"]).drop_duplicates(subset=["Sector"])
    if rows.empty:
        return []
    azs = sorted((float(a) % 360.0, str(name)) for name, a in zip(rows["Sector"], rows["azimuth"]))
    n = len(azs)
    if n == 1:
        az, name = azs[0]
        return [(name, az, (az - 180.0) % 360.0, (az + 180.0) % 360.0)]
    wedges = []
    for i, (az, name) in enumerate(azs):
        prev_az = azs[(i - 1) % n][0]
        next_az = azs[(i + 1) % n][0]
        incoming = (prev_az + ((az - prev_az) % 360.0) / 2.0) % 360.0
        outgoing = (az + ((next_az - az) % 360.0) / 2.0) % 360.0
        wedges.append((name, az, incoming, outgoing))
    return wedges


# Bad-spot site mark: the same unfilled Y as a planning widget. Cyan stroke only.
BAD_SPOT_PIE_EDGE = "#00E5EE"
# The serial bad-spot maps draw a yellow line from the serving cell to each red
# bin. This gate is only the older azimuth line, used when those samples are
# not passed in. Half of a 65° sector beam. A three-sector site is always
# within 60° of some arm, so a wider gate would put a line on every nearby site.
BAD_SPOT_AIM_DEG = 32.0


## 5. Serving cell, tilt, and population

Match a Cell Id and EARFCN to the dump. Tilt text, the ground tilt, and WorldPop people per km² used when an azimuth turns.


In [ ]:
# ------------------------------------------------------------------------
# 5. Serving cell, tilt, and population
# Match a Cell Id and EARFCN to the dump. Tilt text, the ground tilt, and WorldPop people per km² used when an azimuth turns.
# ------------------------------------------------------------------------

def _serving_cell_index() -> tuple[dict, dict]:
    """Serving cells on L900, L1800, L2100 and L2600, keyed by eNB, cell and EARFCN.

    A cell with no eNB id or cell id is left out. The exact key prefers the
    EARFCN of the sample. The pair key is the fallback when the log has a Cell Id
    but the EARFCN is missing.
    """
    global _SERVING_INDEX, _SERVING_LIST
    if _SERVING_INDEX is not None:
        return _SERVING_INDEX
    load_site_sectors()
    cells = _SITE_CELLS if _SITE_CELLS is not None else pd.DataFrame()
    exact: dict[tuple[int, int, int], dict] = {}
    pair: dict[tuple[int, int], dict] = {}
    listed: list[dict] = []
    listed_pairs: set[tuple[int, int]] = set()
    if cells is not None and not cells.empty and {"eNBId", "cellId", "Lat", "Lon"}.issubset(cells.columns):
        for _, row in cells.iterrows():
            layer = str(row.get("layer") or "")
            if layer not in REPORT_LAYERS:
                continue
            enb = pd.to_numeric(row.get("eNBId"), errors="coerce")
            cid = pd.to_numeric(row.get("cellId"), errors="coerce")
            lat = pd.to_numeric(row.get("Lat"), errors="coerce")
            lon = pd.to_numeric(row.get("Lon"), errors="coerce")
            if pd.isna(enb) or pd.isna(cid) or pd.isna(lat) or pd.isna(lon):
                continue
            az = pd.to_numeric(row.get("Azimuth"), errors="coerce")
            et = pd.to_numeric(row.get("E_tilt"), errors="coerce")
            mt = pd.to_numeric(row.get("M_tilt"), errors="coerce")
            ht = pd.to_numeric(row.get("Height"), errors="coerce")
            rec = {
                "lat": float(lat),
                "lon": float(lon),
                "sector": str(row.get("Sector") or "").strip(),
                "layer": layer,
                "site": str(row.get("SiteName") or "").strip(),
                "name": str(row.get("EUtranCellId") or "").strip(),
                "azimuth": None if pd.isna(az) else float(az) % 360.0,
                "etilt": None if pd.isna(et) else float(et),
                "mtilt": None if pd.isna(mt) else float(mt),
                "height": None if pd.isna(ht) else float(ht),
            }
            key = (int(enb), int(cid))
            previous = pair.get(key)
            if previous is None or (previous.get("azimuth") is None and rec["azimuth"] is not None):
                pair[key] = rec
            ear = pd.to_numeric(row.get("EARFCN"), errors="coerce")
            if pd.notna(ear):
                exact[(*key, int(ear))] = rec
                listed.append(rec)
                listed_pairs.add(key)
    for key, rec in pair.items():
        if key not in listed_pairs:
            listed.append(rec)
    _SERVING_INDEX = (exact, pair)
    _SERVING_LIST = listed
    return _SERVING_INDEX


def _serving_cell_list() -> list[dict]:
    """One record per L900/L1800/L2100/L2600 cell that has a Cell Id in the dump."""
    _serving_cell_index()
    return _SERVING_LIST or []


def _serving_cells_near(lat: float, lon: float, km: float = 8.0) -> list[dict]:
    """Dump cells whose site is inside a box around this point."""
    global _SERVING_GRID
    if _SERVING_GRID is None:
        grid: dict[tuple[int, int], list[dict]] = {}
        for rec in _serving_cell_list():
            key = (int(rec["lat"] / 0.08), int(rec["lon"] / 0.08))
            grid.setdefault(key, []).append(rec)
        _SERVING_GRID = grid
    reach = int(km / (0.08 * 111.32)) + 1
    cy, cx = int(lat / 0.08), int(lon / 0.08)
    found: list[dict] = []
    for dy in range(-reach, reach + 1):
        for dx in range(-reach, reach + 1):
            found.extend(_SERVING_GRID.get((cy + dy, cx + dx), ()))
    return found


def _tilt_step(
    etilt: float | None, mtilt: float | None, delta: float
) -> tuple[float | None, float | None, float | None, float | None]:
    """Apply a tilt change to E_tilt first. Leftover degrees move M_tilt.

    Positive delta is downtilt. A blank tilt is not treated as 0°. E_tilt stays
    within 0 to 12 degrees and M_tilt stays within 0 to 8 degrees.
    """
    spill = delta
    new_e = etilt
    if etilt is not None:
        new_e = float(etilt) + delta
        spill = 0.0
        if new_e < 0.0:
            spill = new_e
            new_e = 0.0
        elif new_e > 12.0:
            spill = new_e - 12.0
            new_e = 12.0
    new_m = mtilt
    if mtilt is not None and abs(spill) >= 0.05:
        new_m = min(8.0, max(0.0, float(mtilt) + spill))
    return etilt, mtilt, new_e, new_m


def _deg_text(value: float | None) -> str:
    if value is None:
        return "blank"
    if abs(value - round(value)) < 0.05:
        return str(int(round(value)))
    return f"{value:.1f}"


def _tilt_now(cell: dict) -> str:
    e = "E_tilt blank" if cell.get("etilt") is None else f"E_tilt {_deg_text(cell['etilt'])}°"
    m = "M_tilt blank" if cell.get("mtilt") is None else f"M_tilt {_deg_text(cell['mtilt'])}°"
    return f"{e} and {m}"


def _tilt_bit(label: str, old: float | None, new: float | None) -> str:
    if old is None:
        return f"{label} is blank in the dump"
    if new is None or abs(new - old) < 0.05:
        return f"{label} stays {_deg_text(old)}°"
    return f"{label} {_deg_text(old)}° to {_deg_text(new)}°"


def _tilt_line(cell: dict, delta: float, action: str) -> str:
    """One solution row. delta > 0 downtilt, delta < 0 uptilt."""
    old_e, old_m, new_e, new_m = _tilt_step(cell.get("etilt"), cell.get("mtilt"), delta)
    name = f"{cell.get('sector') or cell.get('name') or 'cell'} {cell.get('layer') or ''}".strip()
    e_same = old_e is None or new_e is None or abs(new_e - old_e) < 0.05
    m_same = old_m is None or new_m is None or abs(new_m - old_m) < 0.05
    if e_same and m_same:
        return (
            f"{name}: {_tilt_bit('E_tilt', old_e, new_e)}. {_tilt_bit('M_tilt', old_m, new_m)}. "
            f"There is no further {action} left inside 0–12° E_tilt and 0–8° M_tilt."
        )
    return (
        f"{name}: {_tilt_bit('E_tilt', old_e, new_e)}. {_tilt_bit('M_tilt', old_m, new_m)}. "
        f"{action.capitalize()}."
    )


# L900 planning reach. Higher bands shorten in proportion to frequency, which is
# the free-space rule (distance for the same loss scales as 1/frequency).
L900_REACH_KM = 8.0
_LAYER_FREQ_MHZ = {"L900": 900.0, "L1800": 1800.0, "L2100": 2100.0, "L2600": 2600.0}
_POP_GRID: tuple | None = None


def _layer_reach_km(layer: str) -> float:
    freq = _LAYER_FREQ_MHZ.get(layer or "", 1800.0)
    return L900_REACH_KM * 900.0 / freq


def _population_grid() -> tuple[np.ndarray, float, float, float, float]:
    """WorldPop 2020 people per km² for the Bogura drive, about 1 km cells."""
    global _POP_GRID
    if _POP_GRID is None:
        path = ROOT / "data" / "bogura_worldpop_2020_1km.npz"
        loaded = np.load(path)
        grid = loaded["pop"].astype(np.float32)
        grid = np.where(np.isfinite(grid) & (grid < 1.0e6), grid, 0.0)
        _POP_GRID = (
            grid,
            float(loaded["xmin"]),
            float(loaded["ymin"]),
            float(loaded["xmax"]),
            float(loaded["ymax"]),
        )
    return _POP_GRID


def population_per_km2(lat: float, lon: float) -> float:
    grid, xmin, ymin, xmax, ymax = _population_grid()
    if not (xmin <= lon <= xmax and ymin <= lat <= ymax):
        return 0.0
    height, width = grid.shape
    col = int(np.clip((lon - xmin) / (xmax - xmin) * (width - 1), 0, width - 1))
    row = int(np.clip((ymax - lat) / (ymax - ymin) * (height - 1), 0, height - 1))
    return float(grid[row, col])


def _people(value: float) -> str:
    return f"{value:,.0f}/km²"


def _destination(lat: float, lon: float, bearing: float, dist_km: float) -> tuple[float, float]:
    br = math.radians(bearing % 360.0)
    lat2 = lat + (dist_km / 111.32) * math.cos(br)
    lon2 = lon + (dist_km / max(_km_per_deg_lon(lat), 0.2)) * math.sin(br)
    return lat2, lon2


def _signed_deg(azimuth: float, bearing: float) -> float:
    """Bearing relative to azimuth, degrees, negative to the left."""
    return (float(bearing) - float(azimuth) + 180.0) % 360.0 - 180.0


def _ground_tilt_deg(height_m: float | None, dist_km: float) -> float:
    height = 30.0 if height_m is None else max(10.0, float(height_m))
    return math.degrees(math.atan(height / max(dist_km, 0.05) / 1000.0))


def _wedge_population(lat: float, lon: float, azimuth: float) -> float:
    """People per km² along a sector bearing, averaged at 1.5, 3 and 4.5 km."""
    values = []
    for dist in (1.5, 3.0, 4.5):
        lat2, lon2 = _destination(lat, lon, azimuth, dist)
        values.append(population_per_km2(lat2, lon2))
    return float(sum(values) / len(values))


def _sectors_near(lat: float, lon: float, km: float = 8.5) -> list[dict]:
    """One record per sector around a point. L900 is kept when the sector has it."""
    rank = {"L900": 0, "L1800": 1, "L2100": 2, "L2600": 3}
    best: dict[tuple, dict] = {}
    for rec in _serving_cells_near(lat, lon, km):
        if rec.get("azimuth") is None:
            continue
        away = float(_equirect_km(rec["lat"], rec["lon"], lat, lon))
        if away > km:
            continue
        key = (rec.get("sector"), round(float(rec["lat"]), 4), round(float(rec["lon"]), 4))
        current = best.get(key)
        if current is None or rank.get(rec.get("layer"), 9) < rank.get(current.get("layer"), 9):
            best[key] = rec
    return list(best.values())


## 6. Choose the solution

Three solutions, in this order.
Tilt and azimuth when a sector inside 2.5 km can aim at the red road. An azimuth turn is kept when the population on the new bearing still serves the customers. Downtilt of an existing sector does not increase by more than 2°.
A 4th L900 sector when the spot is in a null inside 1.5 km, when one or two sites inside 2 km can take it, when the needed downtilt would pass 2°, or when a host inside 3 km can aim at the spot. Existing sectors are not moved.
A new 3-sector L900 site only when the nearest existing site is farther than 3 km.


In [ ]:
# ------------------------------------------------------------------------
# 6. Choose the solution
# Three solutions, in this order.
# Tilt and azimuth when a sector inside 2.5 km can aim at the red road. An azimuth turn is kept when the population on the new bearing still serves the customers. Downtilt of an existing sector does not increase by more than 2°.
# A 4th L900 sector when the spot is in a null inside 1.5 km, when one or two sites inside 2 km can take it, when the needed downtilt would pass 2°, or when a host inside 3 km can aim at the spot. Existing sectors are not moved.
# A new 3-sector L900 site only when the nearest existing site is farther than 3 km.
# ------------------------------------------------------------------------

def _blank_segments(samples: pd.DataFrame, spot: pd.Series) -> list[dict]:
    """Driven stretches inside the circle where the map has no plot.

    A normal sample is a few metres from the next one. A jump of 80 m to 1.6 km
    at driving speed, on the same log, is a blank: the road was driven and
    nothing was plotted. Those metres are part of this bad spot.
    """
    if samples is None or samples.empty or "outline_width_km" not in getattr(spot, "index", []):
        return []
    if pd.isna(spot.get("outline_width_km")):
        return []
    need = ["Latitude", "Longitude", "Time"]
    if any(col not in samples.columns for col in need):
        return []
    work = samples.dropna(subset=need).sort_values("Time")
    if len(work) < 2:
        return []
    lat = work["Latitude"].to_numpy(dtype=float)
    lon = work["Longitude"].to_numpy(dtype=float)
    time_s = pd.to_datetime(work["Time"]).to_numpy(dtype="datetime64[ns]").astype("int64") / 1e9
    source = work["Source"].to_numpy() if "Source" in work.columns else np.zeros(len(work))
    lat0 = float(spot["outline_lat"])
    lon0 = float(spot["outline_lon"])
    half_w = float(spot["outline_width_km"]) / _km_per_deg_lon(lat0) / 2.0
    half_h = float(spot["outline_height_km"]) / 111.32 / 2.0
    if half_w <= 0 or half_h <= 0:
        return []
    same = source[:-1] == source[1:]
    dt = time_s[1:] - time_s[:-1]
    dist_m = _equirect_km(lat[:-1], lon[:-1], lat[1:], lon[1:]) * 1000.0
    speed = dist_m / np.maximum(dt, 1e-6) * 3.6
    mid_lat = (lat[:-1] + lat[1:]) / 2.0
    mid_lon = (lon[:-1] + lon[1:]) / 2.0
    inside = ((mid_lon - lon0) / half_w) ** 2 + ((mid_lat - lat0) / half_h) ** 2 <= 1.0
    keep = (
        same
        & (dt >= 8.0)
        & (dt <= 150.0)
        & (dist_m >= 80.0)
        & (dist_m <= 1600.0)
        & (speed >= 15.0)
        & (speed <= 120.0)
        & inside
    )
    idx = np.flatnonzero(keep)
    order = idx[np.argsort(-dist_m[idx])]
    return [
        {
            "lat": float(mid_lat[i]),
            "lon": float(mid_lon[i]),
            "lat1": float(lat[i]),
            "lon1": float(lon[i]),
            "lat2": float(lat[i + 1]),
            "lon2": float(lon[i + 1]),
            "length_m": float(dist_m[i]),
        }
        for i in order
    ]


def _not_required(reason: str) -> str:
    return "Not required. " + reason


# Azimuth and tilt are used when a sector is within about 2 to 2.5 km.
# A 4th sector is required in the null inside 1.5 km. It also replaces a new site
# when a host inside 3 km can aim at the bad spot. Downtilt is limited to 2°.
TUNE_KM = 2.5
FOURTH_MUST_KM = 1.5
FOURTH_RISK_KM = 2.0
FOURTH_SAVE_KM = 3.0
DOWNTILT_MAX_DEG = 2.0


def _sites_around(sectors: list[dict], lat: float, lon: float) -> list[dict]:
    """One record per site, with the distance and whether the spot sits in a null."""
    groups: dict[tuple, list[dict]] = {}
    for rec in sectors:
        if rec.get("azimuth") is None:
            continue
        key = (
            rec.get("site") or rec.get("sector") or "",
            round(float(rec["lat"]), 4),
            round(float(rec["lon"]), 4),
        )
        groups.setdefault(key, []).append(rec)
    sites = []
    for (name, _lat, _lon), recs in groups.items():
        dist = float(_equirect_km(recs[0]["lat"], recs[0]["lon"], lat, lon))
        bearing = _bearing_deg(recs[0]["lat"], recs[0]["lon"], lat, lon)
        azimuths = [float(rec["azimuth"]) for rec in recs]
        nearest_off = min(abs(_signed_deg(az, bearing)) for az in azimuths)
        sites.append({
            "name": name or str(recs[0].get("sector") or "site"),
            "lat": float(recs[0]["lat"]),
            "lon": float(recs[0]["lon"]),
            "dist": dist,
            "bearing": bearing,
            "nearest_off": nearest_off,
            "in_null": len(azimuths) >= 2 and 30.0 < nearest_off <= 70.0,
            "recs": recs,
            "height": next((rec.get("height") for rec in recs if rec.get("height")), None),
        })
    sites.sort(key=lambda item: item["dist"])
    return sites


def _tune_candidate(rec: dict, dist: float, bearing: float, spot_pop: float) -> dict | None:
    """Azimuth and tilt for one sector within 2.5 km, or nothing if it cannot cover."""
    reach = _layer_reach_km(rec.get("layer") or "")
    if dist > min(TUNE_KM, reach + 0.3):
        return None
    signed = _signed_deg(float(rec["azimuth"]), bearing)
    off = abs(signed)
    if off > 50.0:
        return None
    desired = _ground_tilt_deg(rec.get("height"), dist)
    current = 0.0 if rec.get("etilt") is None else float(rec["etilt"])
    raw = desired - current
    # A downtilt of more than 2° shrinks the existing footprint, so it is not proposed.
    overshoot = raw > DOWNTILT_MAX_DEG
    if raw > DOWNTILT_MAX_DEG:
        delta = DOWNTILT_MAX_DEG
    elif raw < -6.0:
        delta = -6.0
    else:
        delta = raw
    tilt_change = abs(delta) >= 0.8
    turn = 0.0
    old_pop = new_pop = None
    if off > 12.0:
        turn = math.copysign(min(max(off - 8.0, 0.0), 25.0), signed)
    if abs(turn) >= 8.0:
        new_az = (float(rec["azimuth"]) + turn) % 360.0
        old_pop = _wedge_population(rec["lat"], rec["lon"], rec["azimuth"])
        new_pop = _wedge_population(rec["lat"], rec["lon"], new_az)
        if new_pop + 0.5 * spot_pop < old_pop * 0.90:
            turn = 0.0
            old_pop = new_pop = None
    if off > 32.0 and abs(turn) < 8.0:
        return None
    if not tilt_change and abs(turn) < 8.0:
        return None
    return {
        "rec": rec,
        "dist": dist,
        "off": off,
        "delta": delta if tilt_change else 0.0,
        "turn": turn,
        "old_pop": old_pop,
        "new_pop": new_pop,
        "overshoot": overshoot,
    }


def _proposal_draw(kind: str, lat: float, lon: float, azimuths: list[float], target_lat: float, target_lon: float, label: str) -> dict:
    """Where to draw a proposed site or 4th sector on the bad-spot map."""
    return {
        "kind": kind,
        "lat": lat,
        "lon": lon,
        "azimuths": azimuths,
        "target_lat": target_lat,
        "target_lon": target_lon,
        "label": label,
    }


def _fourth_text(site: dict, reason: str = "null") -> str:
    tilt = int(round(min(10.0, max(0.0, _ground_tilt_deg(site.get("height"), site["dist"])))))
    base = (
        f"New L900 sector on {site['name']}, azimuth {site['bearing']:.0f}°, "
        f"E_tilt {tilt}°, M_tilt 0°. The site is {site['dist']:.1f} km away and the spot "
        f"is {site['nearest_off']:.0f}° off the nearest existing sector. "
    )
    if reason == "downtilt":
        why = (
            "A downtilt of more than 2° would shrink the existing coverage, so this sector "
            "is aimed at the bad spot from a site within 1 km. Existing sectors are not moved."
        )
    elif reason == "save":
        why = (
            "A new site is not added. This sector is aimed at the bad spot, and at the blank "
            "inside the circle, from the existing site. Existing sectors are not moved."
        )
    else:
        why = "This sector sits in that null. Existing sectors are not moved."
    return base + why


def _ep_num(value: float | None) -> float | None:
    if value is None:
        return None
    number = float(value)
    if abs(number - round(number)) < 0.05:
        return float(int(round(number)))
    return round(number, 1)


def _fourth_action(site: dict) -> dict:
    """One new L900 sector on an existing site, for the engineering-parameter sheet."""
    tilt = int(round(min(10.0, max(0.0, _ground_tilt_deg(site.get("height"), site["dist"])))))
    height = site.get("height")
    return {
        "site": site["name"],
        "sector": "New L900",
        "cell": "New L900",
        "height": None,
        "azimuth": None,
        "et": None,
        "mt": None,
        "p_height": _ep_num(height if height else 30.0),
        "p_azimuth": _ep_num(float(site["bearing"])),
        "p_et": float(tilt),
        "p_mt": 0.0,
        "action": "4th sector, L900",
    }


def _tune_action(item: dict) -> dict:
    """Existing cell, with the proposed azimuth and tilt beside the current values."""
    rec = item["rec"]
    azimuth = float(rec["azimuth"])
    new_az = azimuth
    if abs(item["turn"]) >= 8.0:
        new_az = (azimuth + item["turn"]) % 360.0
    _old_e, _old_m, new_e, new_m = _tilt_step(rec.get("etilt"), rec.get("mtilt"), item["delta"])
    if abs(item["delta"]) < 0.8:
        new_e, new_m = rec.get("etilt"), rec.get("mtilt")
    cell = str(rec.get("name") or "").strip() or f"{rec.get('sector')} {rec.get('layer')}".strip()
    return {
        "site": item["site"],
        "sector": str(rec.get("sector") or "").strip(),
        "cell": cell,
        "height": _ep_num(rec.get("height")),
        "azimuth": _ep_num(azimuth),
        "et": _ep_num(rec.get("etilt")),
        "mt": _ep_num(rec.get("mtilt")),
        "p_height": _ep_num(rec.get("height")),
        "p_azimuth": _ep_num(new_az),
        "p_et": _ep_num(new_e),
        "p_mt": _ep_num(new_m),
        "action": "Tilt and azimuth",
    }


def _new_site_text(lat: float, lon: float, red_lat: float, red_lon: float, sectors: list[dict]) -> tuple[str, str]:
    site_lat = lat + 0.18 / 111.32
    site_lon = lon
    face = _bearing_deg(site_lat, site_lon, red_lat, red_lon)
    azimuths = [face % 360.0, (face + 120.0) % 360.0, (face + 240.0) % 360.0]
    near = min(
        (float(_equirect_km(rec["lat"], rec["lon"], site_lat, site_lon)) for rec in sectors),
        default=5.0,
    )
    face_tilt = int(round(min(10.0, max(2.0, _ground_tilt_deg(30.0, 0.25)))))
    outer_tilt = int(round(min(8.0, max(2.0, _ground_tilt_deg(30.0, max(near / 2.0, 0.6))))))
    arms = ", ".join(
        f"{az:.0f}° E_tilt {face_tilt if i == 0 else outer_tilt}° M_tilt 0°"
        for i, az in enumerate(azimuths)
    )
    text = (
        f"New 3-sector L900 site at {site_lat:.5f}, {site_lon:.5f}. "
        f"Sectors {arms}. The nearest existing site is {near:.1f} km away, "
        f"and the outer tilts land short of it."
    )
    where = f"New site {site_lat:.5f}, {site_lon:.5f}"
    actions = []
    for index, az in enumerate(azimuths):
        actions.append({
            "site": where,
            "sector": f"S{index + 1}",
            "cell": "New L900",
            "height": None,
            "azimuth": None,
            "et": None,
            "mt": None,
            "p_height": 30.0,
            "p_azimuth": _ep_num(az),
            "p_et": float(face_tilt if index == 0 else outer_tilt),
            "p_mt": 0.0,
            "action": "New 3-sector site",
        })
    return "New site", text, _proposal_draw(
        "new", site_lat, site_lon, azimuths, red_lat, red_lon, "New site",
    ), actions


def _choose_coverage_fix(
    sites: list[dict],
    sectors: list[dict],
    spot_pop: float,
    target_lat: float,
    target_lon: float,
    red_lat: float,
    red_lon: float,
) -> dict:
    """Pick one final solution from the distance rules.

    A null inside 1.5 km is always a 4th L900 sector. Downtilt is limited to 2°.
    When the spot needs more downtilt, a 4th sector within 1 km is used instead.
    A 4th sector also replaces a new site when a host inside 3 km can aim at the
    bad spot. A new 3-sector site is kept only when no such host exists.
    """
    nearby = [site for site in sites if site["dist"] <= TUNE_KM]
    nulls = [site for site in sites if site["in_null"]]
    close_null = next((site for site in nulls if site["dist"] <= FOURTH_MUST_KM), None)
    usable_null = next((site for site in nulls if site["dist"] <= FOURTH_RISK_KM), None)
    nearest = sites[0] if sites else None

    tunes = []
    for site in nearby:
        bearing = site["bearing"]
        for rec in site["recs"]:
            hit = _tune_candidate(rec, site["dist"], bearing, spot_pop)
            if hit is not None:
                hit["site"] = site["name"]
                tunes.append(hit)
    tunes.sort(key=lambda item: (
        0 if item["rec"].get("layer") == "L900" else 1,
        item["dist"],
        item["off"],
    ))

    def pack(kind: str, site: str, final: str, steps: list[dict], draw: dict | None = None, actions: list[dict] | None = None) -> dict:
        return {"kind": kind, "site": site, "final": final, "steps": steps, "draw": draw, "actions": actions or []}

    def steps_for(tilt: str, azimuth: str, fourth: str, new_site: str) -> list[dict]:
        return [
            {"step": "1", "action": "E-tilt / M-tilt", "proposal": tilt},
            {"step": "2", "action": "Azimuth", "proposal": azimuth},
            {"step": "3", "action": "4th sector, L900", "proposal": fourth},
            {"step": "4", "action": "New 3-sector site", "proposal": new_site},
        ]

    def tune_lines(chosen: list[dict]) -> tuple[str, str, str, str]:
        tilt_bits = []
        az_bits = []
        for item in chosen:
            rec = item["rec"]
            name = f"{rec.get('sector')} {rec.get('layer')}".strip()
            if abs(item["delta"]) >= 0.8:
                action = "uptilt so the beam reaches the spot" if item["delta"] < 0 else "downtilt so the beam lands on the spot"
                line = f"From {item['dist']:.1f} km, {item['off']:.0f}° off azimuth. {_tilt_line(rec, item['delta'], action)}"
                if item.get("overshoot"):
                    line += " Downtilt is limited to 2° so the existing coverage is not reduced."
                tilt_bits.append(line)
            else:
                tilt_bits.append(
                    f"{name} already has the tilt for {item['dist']:.1f} km (E_tilt {_deg_text(rec.get('etilt'))}°)."
                )
            if abs(item["turn"]) >= 8.0:
                new_az = (float(rec["azimuth"]) + item["turn"]) % 360.0
                direction = "clockwise" if item["turn"] > 0 else "counter-clockwise"
                pop_line = ""
                if item["old_pop"] is not None:
                    pop_line = (
                        f" Population on the current bearing is {_people(item['old_pop'])}, "
                        f"on the new bearing {_people(item['new_pop'])}, and on the spot {_people(spot_pop)}."
                    )
                az_bits.append(
                    f"{name}: azimuth {float(rec['azimuth']):.0f}° to {new_az:.0f}° "
                    f"({direction} {abs(item['turn']):.0f}°).{pop_line}"
                )
            else:
                az_bits.append(f"{name}: azimuth stays {float(rec['azimuth']):.0f}°. It already faces the spot.")
        site_names = []
        for item in chosen:
            if item["site"] not in site_names:
                site_names.append(item["site"])
        final_bits = []
        for item in chosen:
            rec = item["rec"]
            name = f"{rec.get('sector')} {rec.get('layer')}".strip()
            parts = [f"{name} at {item['dist']:.1f} km"]
            if abs(item["delta"]) >= 0.8:
                _old_e, _old_m, new_e, _new_m = _tilt_step(rec.get("etilt"), rec.get("mtilt"), item["delta"])
                parts.append(f"E_tilt {_deg_text(rec.get('etilt'))}° to {_deg_text(new_e)}°")
            if abs(item["turn"]) >= 8.0:
                new_az = (float(rec["azimuth"]) + item["turn"]) % 360.0
                parts.append(f"azimuth {float(rec['azimuth']):.0f}° to {new_az:.0f}°")
            else:
                parts.append(f"azimuth stays {float(rec['azimuth']):.0f}°")
            final_bits.append(", ".join(parts))
        return (
            " ".join(tilt_bits),
            " ".join(az_bits),
            ", ".join(site_names),
            "; ".join(final_bits) + ".",
        )

    def fourth_draw(site: dict) -> dict:
        return _proposal_draw(
            "fourth", site["lat"], site["lon"], [float(site["bearing"])],
            target_lat, target_lon, "4th L900",
        )

    def fourth_pack(site: dict, reason: str, tilt_why: str, azimuth_why: str, new_why: str) -> dict:
        text = _fourth_text(site, reason)
        return pack("fourth", site["name"], text, steps_for(
            _not_required(tilt_why),
            _not_required(azimuth_why),
            "Final. " + text,
            _not_required(new_why),
        ), fourth_draw(site), [_fourth_action(site)])

    def near_fourth_host() -> dict | None:
        """A site close enough that a 4th sector can replace a heavy downtilt."""
        band = [site for site in sites if 0.5 <= site["dist"] <= 1.0]
        if not band:
            band = [site for site in sites if site["dist"] <= 1.0]
        return band[0] if band else None

    if close_null is not None:
        text = _fourth_text(close_null)
        return pack("fourth", close_null["name"], text, steps_for(
            _not_required(
                f"The spot is in the null of {close_null['name']}, {close_null['dist']:.1f} km away, "
                "inside 1.5 km. A 4th sector is required here, so the existing tilts stay."
            ),
            _not_required(
                "Turning an existing azimuth would move that beam off the customers it already serves. "
                f"Population on the spot is {_people(spot_pop)}."
            ),
            "Final. " + text,
            _not_required("The 4th sector is inside 1.5 km, so a new site is not required."),
        ), fourth_draw(close_null), [_fourth_action(close_null)])

    few_sites = len(nearby) <= 2
    if nearby and few_sites and usable_null is not None and usable_null["dist"] <= FOURTH_RISK_KM:
        # One or two sites, and the spot is in their null inside 2 km.
        if not tunes or tunes[0]["off"] > 25.0:
            text = _fourth_text(usable_null)
            return pack("fourth", usable_null["name"], text, steps_for(
                _not_required(
                    f"Only {len(nearby)} site(s) are inside 2.5 km. "
                    "A tilt change on those few sectors would have to cover the null as well as their own roads."
                ),
                _not_required(
                    f"Only {len(nearby)} site(s) are nearby, so a 4th sector is used instead of turning their only beams. "
                    f"Population on the spot is {_people(spot_pop)}."
                ),
                "Final. " + text,
                _not_required(f"The 4th sector is {usable_null['dist']:.1f} km away, inside 2 km."),
            ), fourth_draw(usable_null), [_fourth_action(usable_null)])

    if tunes:
        chosen = []
        seen = set()
        for item in tunes:
            if item["site"] in seen:
                continue
            chosen.append(item)
            seen.add(item["site"])
            if len(chosen) == 2:
                break
        # Several sites do not force a tune. Use it when a sector already faces the spot.
        # Otherwise a 4th sector inside 2 km is the better fill for a null.
        if any(item.get("overshoot") for item in chosen):
            host = near_fourth_host()
            if host is not None:
                return fourth_pack(
                    host, "downtilt",
                    "The spot needs more than 2° of downtilt. That would shrink the existing coverage, "
                    f"so the tilts stay and a 4th sector within 1 km is used. Population on the spot is {_people(spot_pop)}.",
                    "An azimuth turn is not used. The 4th sector carries the new beam.",
                    f"The 4th sector is on {host['name']}, {host['dist']:.1f} km away, so a new site is not required.",
                )
        faces = chosen[0]["off"] <= 32.0
        if not faces and usable_null is not None and not few_sites:
            text = _fourth_text(usable_null)
            return pack("fourth", usable_null["name"], text, steps_for(
                _not_required(
                    f"{len(nearby)} sites are inside 2.5 km, but none faces the spot. "
                    "Tilting them would not put the beam on it."
                ),
                _not_required(
                    "An azimuth turn was not the better change. "
                    f"A 4th sector at {usable_null['name']} fills the null without moving the busier beams. "
                    f"Population on the spot is {_people(spot_pop)}."
                ),
                "Final. " + text,
                _not_required(f"The 4th sector is {usable_null['dist']:.1f} km away, inside 2 km."),
            ), fourth_draw(usable_null), [_fourth_action(usable_null)])
        tilt, azimuth, site, final = tune_lines(chosen)
        return pack("tune", site, final, steps_for(
            "Final. " + tilt,
            "Final. " + azimuth,
            _not_required(
                "The spot is inside 2.5 km of a sector that can be tuned, and it is not in a null inside 1.5 km. "
                "A 4th sector is not added on top of that beam."
            ),
            _not_required("Azimuth and tilt cover this spot, so a new site is not required."),
        ), None, [_tune_action(item) for item in chosen])

    saver = next((site for site in sites if site["dist"] <= FOURTH_SAVE_KM), None)
    if saver is not None:
        return fourth_pack(
            saver, "save",
            f"The nearest site is {saver['name']} at {saver['dist']:.1f} km. "
            "Azimuth and tilt of the existing sectors are not used for this spot. "
            "A 4th sector aimed at the bad spot is used instead of a new site.",
            f"Population on the spot is {_people(spot_pop)}. The new sector takes the bearing, so existing azimuths stay.",
            f"{saver['name']} is inside 3 km, so a new 3-sector site is not added.",
        )

    new_site, new_text, new_draw, new_actions = _new_site_text(target_lat, target_lon, red_lat, red_lon, sectors)
    if nearest is None:
        far = "No sector was found inside the L900 reach."
    elif nearest["dist"] > TUNE_KM:
        far = (
            f"The nearest sector is {nearest['name']} at {nearest['dist']:.1f} km. "
            "Azimuth and tilt are used only inside about 2.5 km."
        )
    else:
        far = (
            f"{len(nearby)} site(s) are inside 2.5 km, and a tilt or azimuth change "
            "would miss the spot or leave the busier side."
        )
    if usable_null is None:
        fourth = (
            "No existing site inside 3 km can take a sector aimed at this spot. "
            + (f"The nearest site is {nearest['name']} at {nearest['dist']:.1f} km." if nearest else "No site was found inside the L900 reach.")
        )
    else:
        fourth = f"A 4th sector was not the chosen step. {usable_null['name']} is {usable_null['dist']:.1f} km away."
    return pack("new", new_site, new_text, steps_for(
        _not_required(far),
        _not_required(far + f" Population on the spot is {_people(spot_pop)}."),
        _not_required(fourth),
        "Final. " + new_text,
    ), new_draw, new_actions)


def bad_spot_advice(samples: pd.DataFrame, column: str, spot: pd.Series) -> dict:
    """Finding and one final solution for a bad spot, including a blank inside the circle.

    Inside about 2.5 km the final step can retune azimuth and tilt. Downtilt is
    limited to 2°. A larger downtilt becomes a 4th L900 sector on a site within
    1 km. A 4th sector also replaces a new site when a host inside 3 km can aim
    at the bad spot. A new 3-sector site is kept only past that. The side table
    shows all four steps. The final one is marked.
    """
    blanks = _blank_segments(samples, spot)
    blank_m = sum(item["length_m"] for item in blanks)
    bins = _red_bins_inside(samples, column, spot)
    runs = _red_runs(bins) if bins else []
    main: list[dict] = []
    if runs:
        # The sector is aimed at the poor road, not at a blank that can sit on the other side of a site.
        main = max(runs, key=_run_length_m)
        red_lat = sum(item["lat"] for item in main) / len(main)
        red_lon = sum(item["lon"] for item in main) / len(main)
    elif bins:
        red_lat = sum(item["lat"] for item in bins) / len(bins)
        red_lon = sum(item["lon"] for item in bins) / len(bins)
    else:
        red_lat = float(spot["lat"])
        red_lon = float(spot["lon"])
    target_lat, target_lon = red_lat, red_lon
    target_name = "the red road"
    spot_pop = population_per_km2(target_lat, target_lon)
    sectors = _sectors_near(target_lat, target_lon, 8.5)
    finding_lines = []
    if blank_m >= 80:
        finding_lines.append(
            f"Blank, no plot, {blank_m:,.0f} m inside this circle is counted as part of this bad spot."
        )
    else:
        finding_lines.append("No driven blank inside this circle. The poor road is the red plot.")
    def serving_bits(chosen: list[dict]) -> list[str]:
        served: dict[tuple, dict] = {}
        for item in chosen:
            for end in item.get("ends") or ():
                cell = _resolve_serving_cell(end.get("eci"), end.get("earfcn"))
                if cell is None or cell.get("azimuth") is None:
                    continue
                key = (cell.get("sector"), cell.get("layer"))
                slot = served.setdefault(key, {"cell": cell, "n": 0, "lat": 0.0, "lon": 0.0})
                slot["n"] += 1
                slot["lat"] += float(end["lat"])
                slot["lon"] += float(end["lon"])
        bits = []
        for slot in sorted(served.values(), key=lambda item: -item["n"])[:4]:
            cell = slot["cell"]
            slat = slot["lat"] / slot["n"]
            slon = slot["lon"] / slot["n"]
            dist = float(_equirect_km(cell["lat"], cell["lon"], slat, slon))
            off = abs(_signed_deg(cell["azimuth"], _bearing_deg(cell["lat"], cell["lon"], slat, slon)))
            who = f"{cell.get('sector')} {cell.get('layer')}".strip()
            noun = "bin" if slot["n"] == 1 else "bins"
            bits.append(f"{who} ({slot['n']} {noun}, {dist:.1f} km, {off:.0f}° off)")
        return bits

    road_bits = serving_bits(main or bins)
    if road_bits:
        finding_lines.append("Serving cells on this red road: " + "; ".join(road_bits) + ".")
    other = [item for item in bins if main and item not in main]
    other_bits = serving_bits(other)
    if other_bits:
        finding_lines.append("Other red samples in this circle: " + "; ".join(other_bits) + ".")
    finding_lines.append(
        f"WorldPop 2020 on {target_name} is {_people(spot_pop)}. "
        f"L900 reach is {L900_REACH_KM:.0f} km, L1800 {_layer_reach_km('L1800'):.1f} km, "
        f"L2100 {_layer_reach_km('L2100'):.1f} km, L2600 {_layer_reach_km('L2600'):.1f} km."
    )

    sites = _sites_around(sectors, target_lat, target_lon)
    if sites:
        finding_lines.append(
            f"Nearest sector is {sites[0]['name']} at {sites[0]['dist']:.1f} km. "
            f"{sum(site['dist'] <= TUNE_KM for site in sites)} site(s) are inside 2.5 km."
        )
    choice = _choose_coverage_fix(
        sites, sectors, spot_pop, target_lat, target_lon, red_lat, red_lon,
    )
    final = choice["final"]
    if blank_m >= 80:
        final = f"{final} This covers the red road and the {blank_m:,.0f} m blank."
    return {
        "finding": " ".join(finding_lines),
        "blank_m": blank_m,
        "steps": choice["steps"],
        "kind": choice["kind"],
        "site": choice["site"],
        "population": _people(spot_pop),
        "final": final,
        "draw": choice.get("draw"),
        "actions": choice.get("actions") or [],
    }


## 7. Draw one bad-spot map

The yellow dashed line leaves the tip of the sector that served the red samples. It is not the sector the plan turns.
An orange arm is a 4th L900 sector. Purple arms are a new 3-sector site. A blank inside the orange circle is counted with the spot.


In [ ]:
# ------------------------------------------------------------------------
# 7. Draw one bad-spot map
# The yellow dashed line leaves the tip of the sector that served the red samples. It is not the sector the plan turns.
# An orange arm is a 4th L900 sector. Purple arms are a new 3-sector site. A blank inside the orange circle is counted with the spot.
# ------------------------------------------------------------------------

def _resolve_serving_cell(eci: object, earfcn: object) -> dict | None:
    """Site position of the cell that served this sample. Requires a Cell Id."""
    exact, pair = _serving_cell_index()
    number = pd.to_numeric(pd.Series([eci]), errors="coerce").iloc[0]
    if pd.isna(number):
        return None
    enb, cid = divmod(int(number), 256)
    ear = pd.to_numeric(pd.Series([earfcn]), errors="coerce").iloc[0]
    if pd.notna(ear):
        hit = exact.get((enb, cid, int(ear)))
        if hit is not None:
            return hit
    return pair.get((enb, cid))


def _samples_inside_outline(samples: pd.DataFrame, spot: pd.Series) -> pd.DataFrame:
    """Rows that sit inside this bad spot's orange circle."""
    if samples is None or samples.empty:
        return samples
    if "outline_width_km" not in spot.index or pd.isna(spot.get("outline_width_km")):
        return samples
    lon0 = float(spot["outline_lon"])
    lat0 = float(spot["outline_lat"])
    half_w = float(spot["outline_width_km"]) / _km_per_deg_lon(lat0) / 2.0
    half_h = float(spot["outline_height_km"]) / 111.32 / 2.0
    if half_w <= 0 or half_h <= 0:
        return samples
    lon = pd.to_numeric(samples["Longitude"], errors="coerce")
    lat = pd.to_numeric(samples["Latitude"], errors="coerce")
    inside = ((lon - lon0) / half_w) ** 2 + ((lat - lat0) / half_h) ** 2 <= 1.0
    return samples.loc[inside.fillna(False)]


def _red_bins_inside(samples: pd.DataFrame, column: str, spot: pd.Series) -> list[dict]:
    """50 m bins inside the orange circle whose median is poor, with the serving Cell Id.

    The colour on the map is that median, so the line ends on the red piece the
    reader sees. A bin with no Cell Id is kept for the length label and gets no line.
    """
    work = _samples_inside_outline(samples, spot)
    if work is None or work.empty or column not in work.columns or "Cell Id" not in work.columns:
        return []
    work = work.dropna(subset=["Longitude", "Latitude", column]).copy()
    if work.empty:
        return []
    rule = BAD_SPOT_RULES[column]
    threshold = float(rule["threshold"])
    strict = bool(rule.get("strict_less"))
    lat = pd.to_numeric(work["Latitude"], errors="coerce")
    lon = pd.to_numeric(work["Longitude"], errors="coerce")
    dlat, dlon = _bin_steps(lat, MAP_BIN_M)
    work["_by"] = np.floor(lat / dlat).astype("int64")
    work["_bx"] = np.floor(lon / dlon).astype("int64")
    bins: list[dict] = []
    for _, part in work.groupby(["_by", "_bx"], sort=False):
        values = pd.to_numeric(part[column], errors="coerce")
        if values.notna().sum() == 0:
            continue
        median = float(values.median())
        poor_bin = median < threshold if strict else median <= threshold
        if not poor_bin:
            continue
        poor_rows = part.loc[_poor_mask(part[column], threshold, strict_less=strict)]
        use = poor_rows if not poor_rows.empty else part
        cells = pd.to_numeric(use["Cell Id"], errors="coerce")
        identified = use.loc[cells.notna()]
        ends: list[dict] = []
        if not identified.empty:
            identified = identified.copy()
            identified["_eci"] = cells.loc[identified.index].astype("int64")
            for eci_val, grp in identified.groupby("_eci", sort=False):
                earfcn = None
                if "DL EARFCN" in grp.columns:
                    ears = pd.to_numeric(grp["DL EARFCN"], errors="coerce").dropna()
                    if not ears.empty:
                        earfcn = int(ears.mode().iloc[0])
                ends.append({
                    "lat": float(pd.to_numeric(grp["Latitude"], errors="coerce").mean()),
                    "lon": float(pd.to_numeric(grp["Longitude"], errors="coerce").mean()),
                    "eci": int(eci_val),
                    "earfcn": earfcn,
                })
        when = pd.to_datetime(part["Time"], errors="coerce").min() if "Time" in part.columns else pd.NaT
        bins.append({
            "lat": float(pd.to_numeric(part["Latitude"], errors="coerce").mean()),
            "lon": float(pd.to_numeric(part["Longitude"], errors="coerce").mean()),
            "eci": ends[0]["eci"] if ends else None,
            "earfcn": ends[0]["earfcn"] if ends else None,
            "ends": ends,
            "time": when,
        })
    bins.sort(key=lambda item: (pd.Timestamp.min if pd.isna(item["time"]) else item["time"], item["lat"], item["lon"]))
    return bins


def _red_runs(bins: list[dict]) -> list[list[dict]]:
    """Group red bins that touch the same stretch of road, whatever the time order."""
    n = len(bins)
    if n == 0:
        return []
    parent = list(range(n))

    def find(i: int) -> int:
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    # A short stretch of good coverage still belongs to the same red spot.
    limit_m = 400.0
    for i in range(n):
        for j in range(i + 1, n):
            gap_m = float(_equirect_km(bins[i]["lat"], bins[i]["lon"], bins[j]["lat"], bins[j]["lon"])) * 1000.0
            if gap_m <= limit_m:
                parent[find(j)] = find(i)
    groups: dict[int, list[dict]] = {}
    for i, item in enumerate(bins):
        groups.setdefault(find(i), []).append(item)
    return list(groups.values())


def _run_length_m(run: list[dict]) -> float:
    """Road length along the red bins, walking nearest neighbours so a bend is not a shortcut."""
    if len(run) <= 1:
        return float(MAP_BIN_M)
    pending = list(run)
    ordered = [pending.pop(0)]
    while pending:
        last = ordered[-1]
        nxt = min(
            pending,
            key=lambda item: float(_equirect_km(last["lat"], last["lon"], item["lat"], item["lon"])),
        )
        pending.remove(nxt)
        ordered.append(nxt)
    total = 0.0
    for left, right in zip(ordered, ordered[1:]):
        total += float(_equirect_km(left["lat"], left["lon"], right["lat"], right["lon"])) * 1000.0
    # Path from the first red bin to the last, plus one bin so both ends count.
    # A short good gap that joined the bins is included, so the label is the
    # full stretch rather than 50 m on every piece.
    return total + float(MAP_BIN_M)


def _view_arm_deg(ax, spot_lat: float) -> float:
    """Same arm length the cyan sector mark uses, in degrees of latitude."""
    xlim, ylim = ax.get_xlim(), ax.get_ylim()
    span_km = min((xlim[1] - xlim[0]) * _km_per_deg_lon(spot_lat), (ylim[1] - ylim[0]) * 111.32)
    radius_km = float(np.clip(span_km * 0.07, 0.30, 0.85))
    return radius_km / 111.32


def _draw_serving_spider(ax, samples: pd.DataFrame, column: str, spot: pd.Series) -> None:
    """Dashed yellow line from the serving sector arm to each red bin.

    The line starts at the end of that cell's sector, not at the middle of the
    site, so the sector that served the sample is the one the line leaves.
    Every Cell Id on the red samples is drawn when it is an L900, L1800, L2100
    or L2600 cell in the system dump. The length label is the full red stretch.
    """
    bins = _red_bins_inside(samples, column, spot)
    arm = _view_arm_deg(ax, float(spot["lat"]))
    segments = []
    labeled: dict[tuple, dict] = {}
    for item in bins:
        for end in item.get("ends") or ():
            cell = _resolve_serving_cell(end["eci"], end["earfcn"])
            if cell is None:
                continue
            dist_km = float(_equirect_km(cell["lat"], cell["lon"], end["lat"], end["lon"]))
            if dist_km < 0.03:
                continue
            lat0 = cell["lat"]
            lon0 = cell["lon"]
            az = cell.get("azimuth")
            if az is not None:
                rad = np.radians(float(az) % 360.0)
                lon0 = lon0 + arm * np.sin(rad)
                lat0 = lat0 + arm * np.cos(rad)
            segments.append([(lon0, lat0), (end["lon"], end["lat"])])
            key = (cell.get("sector"), cell.get("layer"), round(float(cell["lat"]), 4), round(float(cell["lon"]), 4))
            # The number on the line is the length of that line, from the sector arm to the red bin.
            dist_m = float(_equirect_km(lat0, lon0, end["lat"], end["lon"])) * 1000.0
            current = labeled.get(key)
            if current is None or dist_m > current["dist_m"]:
                labeled[key] = {
                    "lon0": lon0, "lat0": lat0,
                    "lon1": float(end["lon"]), "lat1": float(end["lat"]),
                    "dist_m": dist_m,
                }
    if segments:
        ax.add_collection(LineCollection(
            segments,
            colors="#F5C400",
            linewidths=1.15,
            linestyles=(0, (6, 3.5)),
            capstyle="butt",
            zorder=23,
            alpha=0.95,
        ))
    for item in labeled.values():
        if item["dist_m"] < 80:
            continue
        mx = item["lon0"] + 0.55 * (item["lon1"] - item["lon0"])
        my = item["lat0"] + 0.55 * (item["lat1"] - item["lat0"])
        ax.text(
            mx, my, f"{item['dist_m']:,.0f} m",
            ha="center", va="center", fontsize=8, color="#4A235A", fontweight="bold",
            zorder=34, clip_on=True,
            bbox={"boxstyle": "round,pad=0.16", "fc": "#F5EEF8", "ec": "#6C3483", "lw": 0.7, "alpha": 0.94},
        )
    for run in _red_runs(bins):
        mid = run[len(run) // 2]
        ax.text(
            mid["lon"],
            mid["lat"],
            f"{_run_length_m(run):.0f} m",
            ha="center",
            va="bottom",
            fontsize=7,
            color="#9B1B1B",
            fontweight="bold",
            zorder=30,
            clip_on=True,
            bbox={"boxstyle": "square,pad=0.16", "fc": "white", "ec": "#C0392B", "lw": 0.7, "alpha": 0.94},
        )


def _proposal_arm_km(ax, lat: float, draw: dict, azimuth: float, facing: bool) -> float:
    """Arm length for a proposed sector. The arm toward the bad spot reaches past it."""
    del ax, lat, azimuth
    side_km = 1.35 if draw.get("kind") == "new" else 0.9
    if not facing:
        return side_km
    reach = float(_equirect_km(draw["lat"], draw["lon"], draw["target_lat"], draw["target_lon"]))
    # A new site sits on the spot, so the facing arm is drawn long enough to cross it.
    # A 4th sector starts at the existing site, so its arm lands on the spot and a little past it.
    return max(side_km, reach * 1.25)


def _extent_with_proposal(extent: tuple[float, float, float, float], draw: dict) -> tuple[float, float, float, float]:
    """Widen a bad-spot view just enough to keep the proposed site and its arms."""
    lat = float(draw["lat"])
    reach = float(_equirect_km(draw["lat"], draw["lon"], draw["target_lat"], draw["target_lon"]))
    # New site: three arms of 1.35 km. 4th sector: one arm back toward the spot, so only a small margin past the site.
    margin_km = 1.7 if draw.get("kind") == "new" else 0.55
    if draw.get("kind") != "new":
        margin_km = max(margin_km, 0.35)
    pad_lon = margin_km / _km_per_deg_lon(lat)
    pad_lat = margin_km / 111.32
    # The facing arm may extend past the target. Keep that tip inside the frame.
    overshoot_km = max(0.0, max(1.35, reach * 1.25) - reach) if draw.get("kind") == "new" else max(0.0, reach * 0.25)
    tip_pad_lon = overshoot_km / _km_per_deg_lon(lat)
    tip_pad_lat = overshoot_km / 111.32
    lons = [float(draw["lon"]), float(draw["target_lon"])]
    lats = [lat, float(draw["target_lat"])]
    return (
        min(extent[0], min(lons) - pad_lon, min(lons) - tip_pad_lon),
        max(extent[1], max(lons) + pad_lon, max(lons) + tip_pad_lon),
        min(extent[2], min(lats) - pad_lat, min(lats) - tip_pad_lat),
        max(extent[3], max(lats) + pad_lat, max(lats) + tip_pad_lat),
    )


def _draw_proposal(ax, draw: dict | None) -> None:
    """Draw a proposed new site, or one new 4th sector, long enough to reach the spot."""
    if not draw:
        return
    lat = float(draw["lat"])
    lon = float(draw["lon"])
    color = "#6C3483" if draw.get("kind") == "new" else "#D35400"
    azimuths = [float(az) for az in draw.get("azimuths") or []]
    for index, az in enumerate(azimuths):
        length = _proposal_arm_km(ax, lat, draw, az, index == 0)
        rad = np.radians(az % 360.0)
        x1 = lon + (length / _km_per_deg_lon(lat)) * np.sin(rad)
        y1 = lat + (length / 111.32) * np.cos(rad)
        ax.plot([lon, x1], [lat, y1], color="white", linewidth=6.0, solid_capstyle="round", zorder=31, clip_on=True)
        ax.plot([lon, x1], [lat, y1], color=color, linewidth=3.2, solid_capstyle="round", zorder=32, clip_on=True)
        if index == 0:
            ax.text(
                x1, y1, f"{az:.0f}°",
                ha="center", va="center", fontsize=7, color=color, fontweight="bold",
                zorder=35, clip_on=True,
                bbox={"boxstyle": "round,pad=0.12", "fc": "white", "ec": color, "lw": 0.6, "alpha": 0.94},
            )
    ax.scatter([lon], [lat], s=42, c=color, marker="o", zorder=36, edgecolors="white", linewidths=0.9)
    ax.text(
        lon, lat, str(draw.get("label") or "Proposed"),
        ha="center", va="bottom", fontsize=8, color=color, fontweight="bold",
        zorder=37, clip_on=False,
        bbox={"boxstyle": "round,pad=0.18", "fc": "white", "ec": color, "lw": 0.8, "alpha": 0.95},
    )


def draw_bad_spot_sector_pies(
    ax,
    spot: pd.Series,
    samples: pd.DataFrame | None = None,
    column: str | None = None,
) -> None:
    """Transparent sector marks for every site that touches this bad-spot view.

    One cyan line per sector, with no fill. If any sector of a site falls inside
    the view, every sector of that site is drawn. When the samples under the map
    are passed in, a yellow line runs from the serving cell to each red bin
    inside the orange circle. Without those samples, the older azimuth line to
    the middle of the spot is kept.
    """
    sectors = load_site_sectors()
    if sectors.empty:
        return
    xlim, ylim = ax.get_xlim(), ax.get_ylim()
    spot_lat = float(spot["lat"])
    spot_lon = float(spot["lon"])
    inside = sectors["lat"].between(ylim[0], ylim[1]) & sectors["lon"].between(xlim[0], xlim[1])
    if not inside.any():
        if samples is not None and column:
            _draw_serving_spider(ax, samples, column, spot)
        return
    present = sectors.loc[inside, "SiteName"].unique()
    chosen = sectors[sectors["SiteName"].isin(present)]
    span_km = min((xlim[1] - xlim[0]) * _km_per_deg_lon(spot_lat), (ylim[1] - ylim[0]) * 111.32)
    radius_km = float(np.clip(span_km * 0.07, 0.30, 0.85))
    r_deg = radius_km / 111.32
    for site_name, part in chosen.groupby("SiteName", sort=False):
        part = part.drop_duplicates(subset=["Sector"]).dropna(subset=["azimuth"])
        if part.empty:
            continue
        here = part[inside.reindex(part.index).fillna(False).to_numpy()]
        if here.empty:
            continue
        lat = float(here["lat"].median())
        lon = float(here["lon"].median())
        bearing = _bearing_deg(lat, lon, spot_lat, spot_lon)
        facing: list[float] = []
        for _, sec in part.iterrows():
            az = float(sec["azimuth"]) % 360.0
            rad = np.radians(az)
            if _ang_diff(az, bearing) <= BAD_SPOT_AIM_DEG:
                facing.append(az)
            ax.plot(
                [lon, lon + r_deg * np.sin(rad)],
                [lat, lat + r_deg * np.cos(rad)],
                color="#8FD8E0",
                linewidth=3.0,
                solid_capstyle="round",
                zorder=24,
                clip_on=True,
            )
            ax.plot(
                [lon, lon + r_deg * np.sin(rad)],
                [lat, lat + r_deg * np.cos(rad)],
                color="#E7FBFD",
                linewidth=1.5,
                solid_capstyle="round",
                zorder=25,
                clip_on=True,
            )
            name = str(sec["Sector"]).strip()
            if not name or name.lower() == "nan":
                continue
            ax.text(
                lon + r_deg * 1.28 * np.sin(rad),
                lat + r_deg * 1.28 * np.cos(rad),
                name,
                ha="center",
                va="center",
                fontsize=4.8,
                color="#F7FEFF",
                zorder=29,
                clip_on=True,
            )
        if samples is not None and column:
            continue
        if not facing:
            continue
        dist_km = float(_equirect_km(lat, lon, spot_lat, spot_lon))
        if dist_km < radius_km * 1.8:
            continue
        text = f"{dist_km:.1f} kM" if dist_km < 10 else f"{dist_km:.0f} kM"
        for az in facing:
            rad = np.radians(az)
            x0 = lon + r_deg * np.sin(rad)
            y0 = lat + r_deg * np.cos(rad)
            ax.annotate(
                "",
                xy=(spot_lon, spot_lat),
                xytext=(x0, y0),
                arrowprops={"arrowstyle": "-|>", "color": "#F5C400", "lw": 1.4, "mutation_scale": 7},
                zorder=26,
                clip_on=True,
            )
            mx = (x0 + spot_lon) / 2.0
            my = (y0 + spot_lat) / 2.0
            ax.text(
                mx, my, text, ha="center", va="center", fontsize=6, color="#111111",
                fontweight="bold", zorder=28, clip_on=True,
                bbox={"boxstyle": "round,pad=0.12", "fc": "#FFE14A", "ec": "none", "alpha": 0.92},
            )
    if samples is not None and column:
        _draw_serving_spider(ax, samples, column, spot)


## 8. Coverage maps and the satellite background

RSRP, RSRQ, and SINR maps, the road map, and the satellite tiles behind them. Poor bins are drawn on top of good bins.


In [ ]:
# ------------------------------------------------------------------------
# 8. Coverage maps and the satellite background
# RSRP, RSRQ, and SINR maps, the road map, and the satellite tiles behind them. Poor bins are drawn on top of good bins.
# ------------------------------------------------------------------------

def _coverage_extent_from_geo(geo: pd.DataFrame) -> tuple[float, float, float, float]:
    lon_min, lon_max = float(geo["Longitude"].min()), float(geo["Longitude"].max())
    lat_min, lat_max = float(geo["Latitude"].min()), float(geo["Latitude"].max())
    pad_lon = 0.02 * (lon_max - lon_min)
    pad_lat = 0.02 * (lat_max - lat_min)
    return lon_min - pad_lon, lon_max + pad_lon, lat_min - pad_lat, lat_max + pad_lat


def _bin_steps(lat: pd.Series, bin_m: float) -> tuple[float, float]:
    """Bin grid in degrees. The reference latitude is rounded so that maps and bad
    spots land on the same grid even when built from slightly different subsets."""
    ref_lat = round(float(lat.median()), 1)
    return bin_m / 1000.0 / 111.32, bin_m / 1000.0 / _km_per_deg_lon(ref_lat)


def _bin_route_values(
    geo: pd.DataFrame,
    values: pd.Series,
    classifier,
    bin_m: float,
) -> tuple[pd.DataFrame, pd.Series]:
    """Reduce the route to one point per bin, coloured by that bin's median KPI.

    A city-scale map is about 200 m per pixel, so a raw scatter just shows
    whichever class happens to be drawn last over a whole segment of road.
    Binning makes the colour of a road mean the coverage measured on it.
    """
    lat = pd.to_numeric(geo["Latitude"], errors="coerce")
    lon = pd.to_numeric(geo["Longitude"], errors="coerce")
    val = pd.to_numeric(values, errors="coerce")
    ok = lat.notna() & lon.notna() & val.notna()
    if not ok.any():
        return geo.iloc[0:0], pd.Series(dtype=object)
    lat, lon, val = lat[ok], lon[ok], val[ok]
    dlat, dlon = _bin_steps(lat, bin_m)
    key = pd.MultiIndex.from_arrays(
        [np.floor(lat / dlat).astype(np.int64), np.floor(lon / dlon).astype(np.int64)]
    )
    binned = (
        pd.DataFrame(
            {"Latitude": lat.to_numpy(), "Longitude": lon.to_numpy(), "value": val.to_numpy()},
            index=key,
        )
        .groupby(level=[0, 1])
        .agg(Latitude=("Latitude", "mean"), Longitude=("Longitude", "mean"), value=("value", "median"))
        .reset_index(drop=True)
    )
    return binned, classifier(binned["value"])


# A drive-test export draws the route as a solid trace, not a cloud of
# sub-pixel dots. At city scale a 50 m bin is smaller than one pixel, so a
# scatter turns the two greens into one blurry line.
MAP_DPI = 200
MAP_ROUTE_LW = 3.2
_TILE_CACHE = Path("/tmp/bogura_sat_tiles")
_LIGHT_BASE_CACHE = Path("/tmp/bogura_light_base")
_BOUNDARY_CACHE = Path("/tmp/bogura_boundary_tiles")
_LIGHT_BASE_URL = (
    "https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}"
)
_BOUNDARY_URL = (
    "https://server.arcgisonline.com/ArcGIS/rest/services/Reference/World_Boundaries_and_Places/MapServer/tile/{z}/{y}/{x}"
)
_BASEMAP_CACHE: dict[tuple, object] = {}
# Circle geometries for the report, in pixels of the saved PNG (origin top-left).
# Each circle becomes its own Excel shape so it can be dragged on its own.
MAP_CIRCLE_PIXELS: dict[str, dict] = {}


def _merc_lat(y: float, n: float) -> float:
    merc = math.pi * (1.0 - 2.0 * y / n)
    return math.degrees(math.atan(math.sinh(merc)))


def _tile_bounds(x: int, y: int, z: int) -> tuple[float, float, float, float]:
    """West, east, south, north of one Web-Mercator tile."""
    n = 2**z
    west = x / n * 360.0 - 180.0
    east = (x + 1) / n * 360.0 - 180.0
    north = _merc_lat(y, n)
    south = _merc_lat(y + 1, n)
    return west, east, south, north


def _choose_zoom(span_deg: float) -> int:
    """Zoom where the satellite view stays within about twenty tiles on a side."""
    best = 10
    for z in range(9, 17):
        tiles = span_deg / (360.0 / 2**z)
        if tiles <= 20:
            best = z
        else:
            break
    return best


def _fetch_cached_tile(cache: Path, url: str, z: int, x: int, y: int):
    """Download one map tile into its own cache so styles never share a filename."""
    from PIL import Image

    for ext in (".jpg", ".png"):
        path = cache / f"{z}_{x}_{y}{ext}"
        if path.exists() and path.stat().st_size > 80:
            return Image.open(path)
    req = urllib.request.Request(url, headers={"User-Agent": "BoguraDriveTestReport/1.47 (coverage map export)"})
    cache.mkdir(parents=True, exist_ok=True)
    import time
    last_error: Exception | None = None
    data = b""
    for attempt in range(3):
        try:
            with urllib.request.urlopen(req, timeout=20) as resp:
                data = resp.read()
            break
        except Exception as exc:
            last_error = exc
            time.sleep(1.5 * (attempt + 1))
    else:
        raise last_error if last_error else RuntimeError(f"tile {z}/{x}/{y}")
    ext = ".jpg" if data[:3] == b"\xff\xd8\xff" else ".png"
    path = cache / f"{z}_{x}_{y}{ext}"
    path.write_bytes(data)
    return Image.open(path)


def _fetch_tile(z: int, x: int, y: int):
    # Esri World Imagery is z/y/x. x is the column, y is the row.
    url = (
        "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/"
        f"{z}/{y}/{x}"
    )
    return _fetch_cached_tile(_TILE_CACHE, url, z, x, y).convert("RGB")


def _add_basemap(ax) -> None:
    """Satellite image under the route."""
    try:
        _add_basemap_image(ax)
    except Exception as exc:
        print(f"Basemap skipped: {exc}")


def _add_basemap_image(ax) -> None:
    from PIL import Image

    west, east = ax.get_xlim()
    south, north = ax.get_ylim()
    if east <= west or north <= south:
        return
    z = _choose_zoom(max(east - west, north - south))
    n = 2**z
    x0 = max(0, int((west + 180.0) / 360.0 * n))
    x1 = min(n - 1, int((east + 180.0) / 360.0 * n))
    def tile_y(lat: float) -> int:
        lat_r = math.radians(max(-85.0, min(85.0, lat)))
        return int((1.0 - math.log(math.tan(lat_r) + 1.0 / math.cos(lat_r)) / math.pi) / 2.0 * n)
    y0 = max(0, min(tile_y(north), n - 1))
    y1 = max(0, min(tile_y(south), n - 1))
    if y1 < y0:
        y0, y1 = y1, y0
    key = (z, x0, x1, y0, y1)
    image = _BASEMAP_CACHE.get(key)
    if image is None:
        from concurrent.futures import ThreadPoolExecutor

        width = (x1 - x0 + 1) * 256
        height = (y1 - y0 + 1) * 256
        canvas = Image.new("RGB", (width, height), (40, 48, 36))
        coords = [(x, y) for y in range(y0, y1 + 1) for x in range(x0, x1 + 1)]

        def _load(xy: tuple[int, int]):
            x, y = xy
            try:
                return xy, _fetch_tile(z, x, y)
            except Exception:
                return xy, None

        with ThreadPoolExecutor(max_workers=8) as pool:
            fetched = list(pool.map(_load, coords))
        for (x, y), tile in fetched:
            if tile is None:
                continue
            canvas.paste(tile, ((x - x0) * 256, (y - y0) * 256))
        image = np.asarray(canvas)
        _BASEMAP_CACHE[key] = image
    west_m, _, _, north_m = _tile_bounds(x0, y0, z)
    _, east_m, south_m, _ = _tile_bounds(x1, y1, z)
    ax.imshow(
        image, extent=(west_m, east_m, south_m, north_m),
        origin="upper", interpolation="bilinear", zorder=0, aspect="auto",
    )
    ax.set_xlim(west, east)
    ax.set_ylim(south, north)
    ax.set_aspect("equal", adjustable="box")
    ax.text(
        0.01, 0.01, "Satellite © Esri",
        transform=ax.transAxes, fontsize=6, color="#FFFFFF", ha="left", va="bottom", zorder=6,
    )


def _compose_tiles(z: int, x0: int, x1: int, y0: int, y1: int, *, kind: str):
    """Stitch one basemap. Boundary maps are the light canvas plus admin lines."""
    from PIL import Image
    from concurrent.futures import ThreadPoolExecutor

    width = (x1 - x0 + 1) * 256
    height = (y1 - y0 + 1) * 256
    if kind == "boundary":
        canvas = Image.new("RGBA", (width, height), (245, 245, 245, 255))
        services = (
            (_LIGHT_BASE_CACHE, _LIGHT_BASE_URL, "RGB"),
            (_BOUNDARY_CACHE, _BOUNDARY_URL, "RGBA"),
        )
    else:
        canvas = Image.new("RGB", (width, height), (40, 48, 36))
        services = ((_TILE_CACHE, (
            "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}"
        ), "RGB"),)
    coords = [(x, y) for y in range(y0, y1 + 1) for x in range(x0, x1 + 1)]

    def _load(item):
        cache, template, _mode = item[0]
        x, y = item[1]
        try:
            tile = _fetch_cached_tile(cache, template.format(z=z, y=y, x=x), z, x, y)
            return (x, y), tile
        except Exception:
            return (x, y), None

    # Fetch each service on its own so a missing boundary tile still leaves the canvas.
    for service in services:
        with ThreadPoolExecutor(max_workers=8) as pool:
            fetched = list(pool.map(_load, [(service, xy) for xy in coords]))
        for (x, y), tile in fetched:
            if tile is None:
                continue
            if tile.mode == "RGBA":
                canvas.paste(tile, ((x - x0) * 256, (y - y0) * 256), tile)
            else:
                canvas.paste(tile.convert("RGB"), ((x - x0) * 256, (y - y0) * 256))
    return np.asarray(canvas.convert("RGB"))


def _add_boundary_basemap(ax) -> None:
    """White admin-boundary map. The coloured KPI trace is the only line drawn on it."""
    try:
        west, east = ax.get_xlim()
        south, north = ax.get_ylim()
        if east <= west or north <= south:
            return
        z = _choose_zoom(max(east - west, north - south))
        n = 2**z
        x0 = max(0, int((west + 180.0) / 360.0 * n))
        x1 = min(n - 1, int((east + 180.0) / 360.0 * n))

        def tile_y(lat: float) -> int:
            lat_r = math.radians(max(-85.0, min(85.0, lat)))
            return int((1.0 - math.log(math.tan(lat_r) + 1.0 / math.cos(lat_r)) / math.pi) / 2.0 * n)

        y0 = max(0, min(tile_y(north), n - 1))
        y1 = max(0, min(tile_y(south), n - 1))
        if y1 < y0:
            y0, y1 = y1, y0
        key = ("boundary", z, x0, x1, y0, y1)
        image = _BASEMAP_CACHE.get(key)
        if image is None:
            image = _compose_tiles(z, x0, x1, y0, y1, kind="boundary")
            _BASEMAP_CACHE[key] = image
        west_m, _, _, north_m = _tile_bounds(x0, y0, z)
        _, east_m, south_m, _ = _tile_bounds(x1, y1, z)
        ax.imshow(
            image, extent=(west_m, east_m, south_m, north_m),
            origin="upper", interpolation="bilinear", zorder=0, aspect="auto",
        )
        ax.set_xlim(west, east)
        ax.set_ylim(south, north)
        ax.set_aspect("equal", adjustable="box")
        ax.text(
            0.99, 0.01, "Boundaries © Esri",
            transform=ax.transAxes, fontsize=6, color="#666666", ha="right", va="bottom", zorder=6,
        )
    except Exception as exc:
        print(f"Boundary basemap skipped: {exc}")


def _route_pieces(geo: pd.DataFrame, column: str, bin_m: float) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Time-ordered 50 m vertices, coloured by that bin's median, split on gaps."""
    work = geo.dropna(subset=["Longitude", "Latitude", column]).copy()
    if "Time" in work.columns:
        sort_cols = ["Source", "Time"] if "Source" in work.columns else ["Time"]
        work = work.sort_values(sort_cols)
    lat = pd.to_numeric(work["Latitude"], errors="coerce")
    lon = pd.to_numeric(work["Longitude"], errors="coerce")
    val = pd.to_numeric(work[column], errors="coerce")
    ok = lat.notna() & lon.notna() & val.notna()
    work = work.loc[ok]
    if work.empty:
        return np.empty((0, 2)), np.empty(0, dtype=object), np.empty(0, dtype=bool)
    lat = lat[ok]
    lon = lon[ok]
    val = val[ok]
    dlat, dlon = _bin_steps(lat, bin_m)
    key = pd.MultiIndex.from_arrays(
        [np.floor(lat / dlat).astype(np.int64), np.floor(lon / dlon).astype(np.int64)]
    )
    median = val.groupby(key).transform("median")
    classes = MAP_CLASSIFIERS[column](median)
    # One vertex per visit to a bin, in drive order, so the trace follows the road.
    changed = np.empty(len(work), dtype=bool)
    changed[0] = True
    keys = key.to_numpy()
    changed[1:] = keys[1:] != keys[:-1]
    step = np.zeros(len(work))
    step[1:] = _equirect_km(lat.to_numpy()[:-1], lon.to_numpy()[:-1], lat.to_numpy()[1:], lon.to_numpy()[1:]) * 1000.0
    source = work["Source"].to_numpy() if "Source" in work.columns else np.zeros(len(work))
    new_pass = np.zeros(len(work), dtype=bool)
    new_pass[1:] = (source[1:] != source[:-1]) | (step[1:] > 120.0)
    keep = changed | new_pass
    pts = np.column_stack([lon.to_numpy()[keep], lat.to_numpy()[keep]])
    return pts, classes.to_numpy()[keep], new_pass[keep]


def _draw_coverage_points(
    ax,
    geo: pd.DataFrame,
    class_series: pd.Series,
    labels: list[str],
    colors: list[str],
    max_points: int = MAP_MAX_POINTS,
    *,
    poor_on_top: bool = True,
    bin_m: float | None = None,
    column: str | None = None,
    route_casing: bool = True,
) -> None:
    """Solid drive trace, worst colour on top, same 50 m bins as the bad spots.

    A dark casing under the colour keeps yellow and the two greens readable on
    the street map, the way a log-export route sits on the road.
    """
    if geo.empty:
        return
    classifier = MAP_CLASSIFIERS.get(column or "")
    if not (bin_m and classifier is not None and column in geo.columns):
        sample = downsample(geo, max_points=max_points)
        classes = class_series.reindex(sample.index)
        for label, color in zip(labels, colors):
            part = sample[classes == label]
            if part.empty:
                continue
            ax.scatter(part["Longitude"], part["Latitude"], s=12, c=color, linewidths=0, zorder=3)
        return
    pts, classes, new_pass = _route_pieces(geo, column, bin_m)
    if len(pts) == 0:
        return
    color_of = dict(zip(labels, colors))
    order = list(labels) if poor_on_top else list(reversed(labels))
    # Consecutive segments of one class become one polyline, so a vector export
    # has clean joins and a few thousand paths rather than one per 50 m.
    segments: dict[str, list] = {lab: [] for lab in labels}
    casing: list = []
    run_start = 0
    breaks = list(np.flatnonzero(new_pass)) + [len(pts)]
    alone: dict[str, list] = {lab: [] for lab in labels}
    for end in breaks:
        if end - run_start == 1:
            lab = classes[run_start]
            if lab in alone:
                alone[lab].append(pts[run_start])
        elif end - run_start > 1:
            casing.append(pts[run_start:end])
            i = run_start
            while i < end - 1:
                lab = classes[i]
                j = i
                while j + 1 < end - 1 and classes[j + 1] == lab:
                    j += 1
                if lab in segments:
                    segments[lab].append(pts[i:j + 2])
                i = j + 1
        run_start = end
    # A white casing keeps the route readable on satellite imagery.
    # Bad-spot boundary maps leave it off: the coloured KPI trace is the chart.
    if casing and route_casing:
        ax.add_collection(LineCollection(
            casing, colors="#FFFFFF", linewidths=MAP_ROUTE_LW + 2.4,
            capstyle="round", joinstyle="round", zorder=2,
        ))
    for z, lab in enumerate(order):
        segs = segments.get(lab) or []
        if not segs:
            continue
        ax.add_collection(LineCollection(
            segs, colors=color_of[lab], linewidths=MAP_ROUTE_LW,
            capstyle="round", joinstyle="round", zorder=3 + z,
        ))
        if alone[lab]:
            xy = np.vstack(alone[lab])
            ax.scatter(xy[:, 0], xy[:, 1], s=18, c=color_of[lab], linewidths=0, zorder=3 + z)


def _legend_sample_counts(df: pd.DataFrame, column: str, classifier, labels: list[str]) -> list[int]:
    """Sample counts for the rows actually drawn, one number per legend bin."""
    classes = classifier(df[column])
    counts = classes.value_counts(dropna=True)
    return [int(counts.get(lab, 0)) for lab in labels]


def _count_legend_handles(labels: list[str], colors: list[str], counts: list[int]) -> list:
    """Square swatches with the bin count in parentheses, matching the coverage legend."""
    from matplotlib.patches import Patch

    return [
        Patch(
            facecolor=color,
            edgecolor="#333333",
            linewidth=0.6,
            label=f"{label} ({count})",
        )
        for label, color, count in zip(labels, colors, counts)
    ]


def _place_count_legend(ax, handles: list, title: str):
    """Legend box inside the map, on the left, as in the coverage export."""
    legend = ax.legend(
        handles=handles,
        title=title,
        loc="upper left",
        bbox_to_anchor=(0.015, 0.985),
        frameon=True,
        fontsize=8,
        title_fontsize=9,
        borderpad=0.7,
        labelspacing=0.4,
        handlelength=1.15,
        handleheight=1.05,
        borderaxespad=0.2,
    )
    frame = legend.get_frame()
    frame.set_edgecolor("#000000")
    frame.set_linewidth(0.9)
    frame.set_facecolor("#FFFFFF")
    frame.set_alpha(1.0)
    legend.get_title().set_fontweight("bold")
    # The poorest colour is drawn above the other bins, so the box has to sit higher
    # than that trace. The legend then covers the road the way the reference export does.
    legend.set_zorder(20)
    return legend


def _coverage_legend_handles(labels: list[str], colors: list[str]) -> list:
    return [
        Line2D(
            [0],
            [0],
            marker="o",
            color="none",
            markerfacecolor=c,
            markeredgecolor="#333333",
            markeredgewidth=0.4,
            markersize=8,
            label=lab,
        )
        for lab, c in zip(labels, colors)
    ]


def plot_discrete_coverage_map(
    df: pd.DataFrame,
    path: Path,
    column: str,
    class_series: pd.Series,
    labels: list[str],
    colors: list[str],
    title: str,
    legend_title: str,
    max_points: int = MAP_MAX_POINTS,
    extent: tuple[float, float, float, float] | None = None,
    layer: str | None = None,
    empty_message: str = "No IDLE samples on this band",
) -> Path:
    """Drive route colored by discrete KPI ranges, with an RSRP-style side legend."""
    PLOTS_DIR.mkdir(parents=True, exist_ok=True)
    geo = df.dropna(subset=["Longitude", "Latitude", column]).copy()
    fig, ax = plt.subplots(figsize=(8.0, 9.2))
    _draw_coverage_points(
        ax, geo, class_series, labels, colors, max_points=max_points, bin_m=MAP_BIN_M, column=column
    )
    if extent is not None:
        ax.set_xlim(extent[0], extent[1])
        ax.set_ylim(extent[2], extent[3])
    elif not geo.empty:
        lon_min, lon_max, lat_min, lat_max = _coverage_extent_from_geo(geo)
        ax.set_xlim(lon_min, lon_max)
        ax.set_ylim(lat_min, lat_max)
    if geo.empty:
        ax.text(
            0.5,
            0.5,
            empty_message,
            transform=ax.transAxes,
            ha="center",
            va="center",
            color="#7F8C8D",
            fontsize=12,
        )
    ax.legend(
        handles=_coverage_legend_handles(labels, colors),
        title=legend_title,
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),
        frameon=True,
        fontsize=8,
        title_fontsize=10,
    )
    ax.set_aspect("equal", adjustable="box")
    _add_basemap(ax)
    add_map_scale_bar(ax)
    ax.set_title(title)
    hide_map_axes(ax)
    fig.tight_layout()
    fig.savefig(path, dpi=MAP_DPI, facecolor="white", bbox_inches="tight")
    plt.close(fig)
    return path


def plot_road_map(
    df: pd.DataFrame,
    path: Path,
    extent: tuple[float, float, float, float] | None = None,
) -> Path:
    """Satellite trace of the drive. One road colour, no legend and no mode title."""
    PLOTS_DIR.mkdir(parents=True, exist_ok=True)
    geo = df.dropna(subset=["Longitude", "Latitude", "RSRP"]).copy()
    fig, ax = plt.subplots(figsize=(7.4, 9.2))
    if geo.empty:
        ax.text(
            0.5, 0.5, "No route", transform=ax.transAxes,
            ha="center", va="center", color="#7F8C8D", fontsize=12,
        )
    else:
        pts, _classes, new_pass = _route_pieces(geo, "RSRP", MAP_BIN_M)
        segments: list = []
        run = 0
        breaks = list(np.flatnonzero(new_pass)) + [len(pts)]
        for end in breaks:
            if end - run > 1:
                segments.append(pts[run:end])
            run = end
        if segments:
            ax.add_collection(LineCollection(
                segments, colors="#1B4F72", linewidths=MAP_ROUTE_LW + 2.6,
                capstyle="round", joinstyle="round", zorder=2,
            ))
            ax.add_collection(LineCollection(
                segments, colors="#F4F7FB", linewidths=MAP_ROUTE_LW,
                capstyle="round", joinstyle="round", zorder=3,
            ))
        if extent is not None:
            ax.set_xlim(extent[0], extent[1])
            ax.set_ylim(extent[2], extent[3])
        else:
            lon_min, lon_max, lat_min, lat_max = _coverage_extent_from_geo(geo)
            ax.set_xlim(lon_min, lon_max)
            ax.set_ylim(lat_min, lat_max)
    ax.set_aspect("equal", adjustable="box")
    _add_basemap(ax)
    add_map_scale_bar(ax)
    hide_map_axes(ax)
    fig.tight_layout()
    fig.savefig(path, dpi=MAP_DPI, facecolor="white", bbox_inches="tight")
    plt.close(fig)
    return path


def plot_rsrp_coverage_map(
    df: pd.DataFrame,
    path: Path,
    max_points: int = MAP_MAX_POINTS,
    *,
    layer: str | None = None,
    extent: tuple[float, float, float, float] | None = None,
    mode_label: str | None = None,
) -> Path:
    """Discrete RSRP coverage map with a range legend (no lon/lat chrome)."""
    title = "Bogura coverage map — RSRP"
    if mode_label:
        title = f"{title} — {mode_label}"
    if layer:
        title = f"{title} — {layer}"
    empty = f"No {mode_label} samples on this band" if mode_label else "No IDLE samples on this band"
    return plot_discrete_coverage_map(
        df,
        path,
        column="RSRP",
        class_series=rsrp_map_class(df["RSRP"]),
        labels=MAP_RSRP_LABELS,
        colors=MAP_RSRP_COLORS,
        title=title,
        legend_title="RSRP (dBm)",
        max_points=max_points,
        extent=extent,
        layer=layer,
        empty_message=empty,
    )


def plot_rsrq_coverage_map(
    df: pd.DataFrame,
    path: Path,
    max_points: int = MAP_MAX_POINTS,
    *,
    layer: str | None = None,
    extent: tuple[float, float, float, float] | None = None,
    mode_label: str | None = None,
) -> Path:
    title = "Bogura coverage map — RSRQ"
    if mode_label:
        title = f"{title} — {mode_label}"
    if layer:
        title = f"{title} — {layer}"
    empty = f"No {mode_label} samples on this band" if mode_label else "No IDLE samples on this band"
    return plot_discrete_coverage_map(
        df,
        path,
        column="RSRQ",
        class_series=rsrq_map_class(df["RSRQ"]),
        labels=MAP_RSRQ_LABELS,
        colors=MAP_RSRQ_COLORS,
        title=title,
        legend_title="RSRQ (dB)",
        max_points=max_points,
        extent=extent,
        layer=layer,
        empty_message=empty,
    )


def plot_sinr_coverage_map(
    df: pd.DataFrame,
    path: Path,
    max_points: int = MAP_MAX_POINTS,
    *,
    layer: str | None = None,
    extent: tuple[float, float, float, float] | None = None,
    mode_label: str | None = None,
) -> Path:
    title = "Bogura coverage map — SINR"
    if mode_label:
        title = f"{title} — {mode_label}"
    if layer:
        title = f"{title} — {layer}"
    empty = f"No {mode_label} samples on this band" if mode_label else "No IDLE samples on this band"
    return plot_discrete_coverage_map(
        df,
        path,
        column="SINR",
        class_series=sinr_map_class(df["SINR"]),
        labels=MAP_SINR_LABELS,
        colors=MAP_SINR_COLORS,
        title=title,
        legend_title="SINR (dB)",
        max_points=max_points,
        extent=extent,
        layer=layer,
        empty_message=empty,
    )


## 9. Find the bad spots

A 50 m bin is poor when its median RSRP is below −120 dBm. RSRQ and SINR still colour their own maps.
Poor bins join one run across a gap of at most 150 m and 60 s. Runs within 2 km join one circle while the poor road still spans at most 7 km.
This drive keeps 26 idle RSRP circles and 99 active RSRP circles.


In [ ]:
# ------------------------------------------------------------------------
# 9. Find the bad spots
# A 50 m bin is poor when its median RSRP is below −120 dBm. RSRQ and SINR still colour their own maps.
# Poor bins join one run across a gap of at most 150 m and 60 s. Runs within 2 km join one circle while the poor road still spans at most 7 km.
# This drive keeps 26 idle RSRP circles and 99 active RSRP circles.
# ------------------------------------------------------------------------

# Bad-spot contract from report v1.35.
# 1) Poor is RSRP < -120 dBm (red bin only; -120 through -115 is light red),
#    RSRQ <= -20 dB (inside the red bin), SINR <= 0 dB (orange and red).
# 2) A bin of road is 50 m. Its colour and its poor/good call both use the median
#    of the samples in it, and poor bins are drawn on top of good ones.
# 3) Every poor run is covered. A run stays one run across a gap of at most 150 m
#    and 60 s on the same pass. Runs within 2 km of each other join one circle
#    when the poor road inside that circle still spans at most 7 km, so the drawn
#    circle stays within 10 km and covers the road. A longer poor road is split
#    into several circles. Pockets more than 2 km apart stay separate circles
#    even when both sit inside the same 10 km neighbourhood, and each circle is
#    its own shape so it can be moved on its own.
# 4) Leftover poor bins that cover >= 1 km² stay a discrete (dotted) circle.
#    With rule 3 taking every run, that count is 0 on this drive; the rule stays.
# Idle best server is the strongest band in each scanner sweep (4 s or 40 m).
# Active best server is the strongest Robi carrier sharing one timestamp.
BAD_SPOT_MIN_CONSEC_M = 200.0
BAD_SPOT_DISCRETE_AREA_KM2 = 1.0
# Best-server samples are one scanner sweep apart (<= 40 m), so a poor stretch may
# skip a sweep or two and still be the same hole.
BAD_SPOT_MAX_SAMPLE_GAP_M = 150.0
BAD_SPOT_MAX_SAMPLE_GAP_S = 60.0
BAD_SPOT_GRID_DEG = 0.009  # ~1.0 km cells for discrete-area clustering
BAD_SPOT_MAX_CONSEC = 30
BAD_SPOT_MAX_DISCRETE = 15
BAD_SPOT_DISCRETE_MIN_POOR = 20
BAD_SPOT_DISCRETE_MAX_SPAN_KM = 3.2
# Calling a whole km² bad means the roads inside it are bad, not that a few
# scattered samples in it are.
BAD_SPOT_DISCRETE_MIN_SHARE = 0.50
# Poor-road span that still fits inside a drawn circle of BAD_SPOT_CIRCLE_MAX_KM.
BAD_SPOT_GROUP_SPAN_KM = 7.0
BAD_SPOT_CIRCLE_MAX_KM = 10.0
BAD_SPOT_CIRCLE_MIN_KM = 2.5
BAD_SPOT_MERGE_GAP_KM = 2.0
BAD_SPOT_CONSEC_MAX_SPAN_KM = BAD_SPOT_GROUP_SPAN_KM
BAD_SPOT_ZOOM_PAD_KM = 0.70
BAD_SPOT_ZOOM_MIN_KM = 1.2
BAD_SPOT_OUTLINE_PAD = 1.45  # with sqrt(2), oval covers bbox corners / red tails
BAD_SPOT_OUTLINE_MIN_MINOR_KM = 0.50
BAD_SPOT_OVERVIEW_MIN_KM = 2.6  # city-scale combined map must show a visible circle
BAD_SPOT_OVERVIEW_MAX_KM = 4.5
BAD_SPOT_OVERVIEW_LW = 2.15
# A few spots can each carry a big haloed ring. Hundreds cannot: the rings would
# cover the colour they are marking, so they shrink and thin as the count grows.
BAD_SPOT_OVERVIEW_TIERS = (
    (40, 2.6, 2.15, True),
    (250, 1.4, 0.70, False),
    (10**9, 0.8, 0.50, False),
)
BAD_SPOT_ZOOM_COLS = 4
BAD_SPOT_MAX_ZOOMS = 300
BAD_SPOT_MAX = BAD_SPOT_MAX_CONSEC + BAD_SPOT_MAX_DISCRETE
BAD_SPOT_RULES = {
    # strict_less: a value equal to the threshold is not poor. -120 is the top of the red bin.
    "RSRP": {"threshold": -120.0, "unit": "dBm", "min_consecutive_m": 0.0, "max_consecutive": 2000, "strict_less": True},
    "RSRQ": {"threshold": -20.0, "unit": "dB", "min_consecutive_m": 0.0, "max_consecutive": 2000, "strict_less": False},
    "SINR": {"threshold": 0.0, "unit": "dB", "min_consecutive_m": 0.0, "max_consecutive": 2000, "strict_less": False},
}


def min_consecutive_m(column: str) -> float:
    return float(BAD_SPOT_RULES[column].get("min_consecutive_m", BAD_SPOT_MIN_CONSEC_M))


# Expected values for the contract above. Kept separate from the live constants so
# a later edit that retunes a rule fails the report check instead of passing quietly.
# Spot counts are circles and discrete areas per sheet.
# Idle stays on the published Borga_Idle_V2.0 result. Active follows
# Borga_Active_mode_11Nov26.7z: Robi M-PCell only, strongest RSRP at each location.
LOGIC_LOCK = {
    "active_operator": "Robi",
    "sweep_s": 4.0,
    "sweep_m": 40.0,
    "bin_m": 50.0,
    "gap_m": 150.0,
    "gap_s": 60.0,
    "span_km": 7.0,
    "circle_max_km": 10.0,
    "circle_min_km": 2.5,
    "merge_gap_km": 2.0,
    "discrete_km2": 1.0,
    "discrete_share": 0.50,
    "poor": {"RSRP": -120.0, "RSRQ": -20.0, "SINR": 0.0},
    "poor_strict": {"RSRP": True, "RSRQ": False, "SINR": False},
    "min_consecutive_m": {"RSRP": 0.0, "RSRQ": 0.0, "SINR": 0.0},
    "rsrp_greens": ("#008000", "#00FF00"),
    "ring_tiers": ((40, 2.6, 2.15, True), (250, 1.4, 0.70, False), (10**9, 0.8, 0.50, False)),
    # RSRQ and SINR bad-spot sheets are not in the workbook. The RSRP sheets
    # keep these circle counts. RSRQ and SINR rules still colour the KPI maps.
    "counts": {
        "Bad Spot RSRP": (26, 0),
        "Bad Spot RSRP_Active": (99, 0),
    },
}


def logic_drift() -> list[str]:
    """Report any live constant that no longer matches LOGIC_LOCK."""
    problems = []
    if ACTIVE_OPERATOR != LOGIC_LOCK["active_operator"]:
        problems.append(f"active operator is {ACTIVE_OPERATOR}")
    live_windows = (
        BEST_SERVER_SWEEP_S, BEST_SERVER_SWEEP_M, MAP_BIN_M,
        BAD_SPOT_MAX_SAMPLE_GAP_M, BAD_SPOT_MAX_SAMPLE_GAP_S, BAD_SPOT_CONSEC_MAX_SPAN_KM,
        BAD_SPOT_CIRCLE_MAX_KM, BAD_SPOT_CIRCLE_MIN_KM, BAD_SPOT_MERGE_GAP_KM,
        BAD_SPOT_DISCRETE_AREA_KM2, BAD_SPOT_DISCRETE_MIN_SHARE,
    )
    locked_windows = (
        LOGIC_LOCK["sweep_s"], LOGIC_LOCK["sweep_m"], LOGIC_LOCK["bin_m"],
        LOGIC_LOCK["gap_m"], LOGIC_LOCK["gap_s"], LOGIC_LOCK["span_km"],
        LOGIC_LOCK["circle_max_km"], LOGIC_LOCK["circle_min_km"], LOGIC_LOCK["merge_gap_km"],
        LOGIC_LOCK["discrete_km2"], LOGIC_LOCK["discrete_share"],
    )
    if live_windows != locked_windows:
        problems.append(f"distance/time windows {live_windows} != {locked_windows}")
    for kpi, threshold in LOGIC_LOCK["poor"].items():
        rule = BAD_SPOT_RULES[kpi]
        if float(rule["threshold"]) != threshold or float(rule["min_consecutive_m"]) != LOGIC_LOCK["min_consecutive_m"][kpi]:
            problems.append(f"{kpi} rule {rule['threshold']}, min {rule['min_consecutive_m']}")
        if bool(rule.get("strict_less")) != bool(LOGIC_LOCK["poor_strict"][kpi]):
            problems.append(f"{kpi} strict_less is {bool(rule.get('strict_less'))}")
    if tuple(MAP_RSRP_COLORS[:2]) != LOGIC_LOCK["rsrp_greens"]:
        problems.append(f"RSRP greens are {MAP_RSRP_COLORS[:2]}")
    if BAD_SPOT_OVERVIEW_TIERS != LOGIC_LOCK["ring_tiers"]:
        problems.append("overview ring tiers changed")
    return problems


def consecutive_label(column: str) -> str:
    """Legend / table wording for the consecutive rule actually applied to this KPI."""
    minimum = min_consecutive_m(column)
    if minimum <= 0:
        return "Poor run (any length)"
    return f"Consecutive \u2265 {minimum:,.0f} m"


def _km_per_deg_lon(lat: float) -> float:
    return 111.32 * float(np.cos(np.radians(lat)))


def _equirect_km(lat1, lon1, lat2, lon2) -> np.ndarray:
    mid = np.radians((np.asarray(lat1) + np.asarray(lat2)) / 2.0)
    dlat = (np.asarray(lat2) - np.asarray(lat1)) * 111.32
    dlon = (np.asarray(lon2) - np.asarray(lon1)) * 111.32 * np.cos(mid)
    return np.sqrt(dlat**2 + dlon**2)


def _poor_mask(values: pd.Series, threshold: float, *, strict_less: bool = False) -> pd.Series:
    """Poor samples. RSRP is strictly below -120. Light red is not poor. RSRQ and SINR are at or below their threshold."""
    v = pd.to_numeric(values, errors="coerce")
    poor = v < threshold if strict_less else v <= threshold
    return v.notna() & poor


def _prepare_track(df: pd.DataFrame, column: str) -> pd.DataFrame:
    g = df.dropna(subset=["Time", "Longitude", "Latitude", column]).copy()
    g[column] = pd.to_numeric(g[column], errors="coerce")
    g["Longitude"] = pd.to_numeric(g["Longitude"], errors="coerce")
    g["Latitude"] = pd.to_numeric(g["Latitude"], errors="coerce")
    g = g.dropna(subset=[column, "Longitude", "Latitude", "Time"])
    sort_cols = ["Source", "Time"] if "Source" in g.columns else ["Time"]
    return g.sort_values(sort_cols).reset_index(drop=True)


def _convex_hull_xy(points: np.ndarray) -> np.ndarray:
    """Monotone-chain convex hull for (n, 2) points."""
    pts = np.asarray(points, dtype=float)
    if len(pts) <= 1:
        return pts
    pts = pts[np.lexsort((pts[:, 1], pts[:, 0]))]
    uniq = np.concatenate([pts[0:1], pts[1:][np.any(np.diff(pts, axis=0) != 0, axis=1)]])
    if len(uniq) <= 2:
        return uniq

    def cross(o, a, b) -> float:
        return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])

    lower: list[np.ndarray] = []
    for p in uniq:
        while len(lower) >= 2 and cross(lower[-2], lower[-1], p) <= 0:
            lower.pop()
        lower.append(p)
    upper: list[np.ndarray] = []
    for p in uniq[::-1]:
        while len(upper) >= 2 and cross(upper[-2], upper[-1], p) <= 0:
            upper.pop()
        upper.append(p)
    return np.vstack(lower[:-1] + upper[:-1])


def _hull_area_km2(lons: np.ndarray, lats: np.ndarray) -> float:
    if len(lons) < 3:
        return 0.0
    lat0 = float(np.mean(lats))
    lon0 = float(np.mean(lons))
    xy = np.column_stack(((lons - lon0) * _km_per_deg_lon(lat0), (lats - lat0) * 111.32))
    hull = _convex_hull_xy(xy)
    if len(hull) < 3:
        return 0.0
    x, y = hull[:, 0], hull[:, 1]
    return float(abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1))) / 2.0)


def _path_length_m(lats: np.ndarray, lons: np.ndarray) -> float:
    if len(lats) < 2:
        return 0.0
    return float(np.sum(_equirect_km(lats[:-1], lons[:-1], lats[1:], lons[1:])) * 1000.0)


def _covering_ellipse(
    lons: np.ndarray,
    lats: np.ndarray,
    *,
    pad: float = BAD_SPOT_OUTLINE_PAD,
    min_minor_km: float = BAD_SPOT_OUTLINE_MIN_MINOR_KM,
    min_major_km: float = 0.55,
) -> tuple[float, float, float, float, float]:
    """Axis-aligned oval that contains the whole poor stretch, including corners and red tails."""
    lons = np.asarray(lons, dtype=float)
    lats = np.asarray(lats, dtype=float)
    lat0 = float((np.min(lats) + np.max(lats)) / 2.0)
    lon0 = float((np.min(lons) + np.max(lons)) / 2.0)
    km_lon = _km_per_deg_lon(lat0)
    span_lon_km = (float(np.max(lons)) - float(np.min(lons))) * km_lon
    span_lat_km = (float(np.max(lats)) - float(np.min(lats))) * 111.32
    # Inscribed ellipse (width=bbox) leaves diagonal tails outside. Scale so corners stay in.
    cover = float(np.sqrt(2.0)) * pad / 1.15
    width_km = max(span_lon_km * cover, min_major_km)
    height_km = max(span_lat_km * cover, min_minor_km)
    return lon0, lat0, width_km, height_km, 0.0


def _outline_fields(lons: np.ndarray, lats: np.ndarray, kind: str) -> dict:
    lon0, lat0, width_km, height_km, angle = _covering_ellipse(lons, lats)
    width_km = min(BAD_SPOT_CIRCLE_MAX_KM, max(width_km, BAD_SPOT_CIRCLE_MIN_KM))
    height_km = min(BAD_SPOT_CIRCLE_MAX_KM, max(height_km, BAD_SPOT_CIRCLE_MIN_KM))
    if kind == "discrete":
        diameter_km = min(BAD_SPOT_CIRCLE_MAX_KM, max(width_km, height_km))
        width_km = height_km = diameter_km
        angle = 0.0
    return {
        "outline_lon": lon0,
        "outline_lat": lat0,
        "outline_width_km": float(width_km),
        "outline_height_km": float(height_km),
        "outline_angle": float(angle),
    }


def _spot_from_points(
    part: pd.DataFrame,
    column: str,
    threshold: float,
    kind: str,
    length_m: float,
    area_km2: float,
) -> dict:
    lat = float(part["Latitude"].mean())
    lon = float(part["Longitude"].mean())
    lat_min = float(part["Latitude"].min())
    lat_max = float(part["Latitude"].max())
    lon_min = float(part["Longitude"].min())
    lon_max = float(part["Longitude"].max())
    vals = pd.to_numeric(part[column], errors="coerce")
    n = int(vals.notna().sum())
    n_poor = int(_poor_mask(vals, threshold, strict_less=bool(BAD_SPOT_RULES[column].get("strict_less"))).sum())
    mean = float(vals.mean()) if n else np.nan
    return {
        "kind": kind,
        "n": n,
        "n_poor": n_poor,
        "poor_pct": n_poor / max(n, 1),
        "mean": mean,
        "lat": lat,
        "lon": lon,
        "lat_min": lat_min,
        "lat_max": lat_max,
        "lon_min": lon_min,
        "lon_max": lon_max,
        "span_lat_km": (lat_max - lat_min) * 111.32,
        "span_lon_km": (lon_max - lon_min) * _km_per_deg_lon(lat),
        "length_m": float(length_m),
        "area_km2": float(area_km2),
        "score": (length_m if kind == "consecutive" else area_km2 * 1000.0),
        **_outline_fields(part["Longitude"].to_numpy(), part["Latitude"].to_numpy(), kind),
    }


def _span_km(lats: np.ndarray, lons: np.ndarray) -> float:
    if len(lats) == 0:
        return 0.0
    lat0 = float(np.mean(lats))
    return max(
        (float(np.max(lats)) - float(np.min(lats))) * 111.32,
        (float(np.max(lons)) - float(np.min(lons))) * _km_per_deg_lon(lat0),
    )


def binned_kpi(geo: pd.DataFrame, column: str, bin_m: float = MAP_BIN_M) -> pd.Series:
    """Median KPI of each bin of road, spread back over the samples in that bin.

    Bad spots are judged on this, not on single samples, and the maps are drawn
    from the same bins: a stretch is circled only where the map shows it poor.
    """
    lat = pd.to_numeric(geo["Latitude"], errors="coerce")
    lon = pd.to_numeric(geo["Longitude"], errors="coerce")
    val = pd.to_numeric(geo[column], errors="coerce")
    if val.notna().sum() == 0:
        return val
    dlat, dlon = _bin_steps(lat, bin_m)
    key = pd.MultiIndex.from_arrays(
        [np.floor(lat / dlat).astype("Int64"), np.floor(lon / dlon).astype("Int64")]
    )
    return val.groupby(key).transform("median")


def _poor_runs(
    poor_idx: np.ndarray,
    same_pass: np.ndarray,
    cum_m: np.ndarray,
    time_s: np.ndarray,
) -> list[list[int]]:
    """Chain poor samples of one pass into runs, tolerating a short good gap."""
    runs: list[list[int]] = []
    if len(poor_idx) == 0:
        return runs
    current = [int(poor_idx[0])]
    for prev, cur in zip(poor_idx[:-1], poor_idx[1:]):
        gap_m = float(cum_m[cur] - cum_m[prev])
        gap_s = float(time_s[cur] - time_s[prev])
        if (
            same_pass[prev] == same_pass[cur]
            and gap_m <= BAD_SPOT_MAX_SAMPLE_GAP_M
            and gap_s <= BAD_SPOT_MAX_SAMPLE_GAP_S
        ):
            current.append(int(cur))
        else:
            runs.append(current)
            current = [int(cur)]
    runs.append(current)
    return runs


def _split_by_span(run: list[int], lat: np.ndarray, lon: np.ndarray) -> list[list[int]]:
    """Cut a run that wanders further than one oval should cover."""
    pieces: list[list[int]] = []
    current: list[int] = []
    for i in run:
        trial = current + [i]
        if len(trial) > 1 and _span_km(lat[trial], lon[trial]) > BAD_SPOT_CONSEC_MAX_SPAN_KM:
            pieces.append(current)
            current = [i]
        else:
            current = trial
    if current:
        pieces.append(current)
    return pieces


def _bbox_of(idx: list[int], lat: np.ndarray, lon: np.ndarray) -> tuple[float, float, float, float]:
    sl = lat[idx]
    so = lon[idx]
    return float(sl.min()), float(sl.max()), float(so.min()), float(so.max())


def _bbox_gap_km(a: tuple[float, float, float, float], b: tuple[float, float, float, float]) -> float:
    """Distance between two lat/lon boxes. Zero when they touch or overlap."""
    lat_gap = max(0.0, b[0] - a[1], a[0] - b[1])
    lon_gap = max(0.0, b[2] - a[3], a[2] - b[3])
    lat0 = (a[0] + a[1] + b[0] + b[1]) / 4.0
    return float(math.hypot(lat_gap * 111.32, lon_gap * _km_per_deg_lon(lat0)))


def _union_span_km(a: tuple[float, float, float, float], b: tuple[float, float, float, float]) -> float:
    lat_min, lat_max = min(a[0], b[0]), max(a[1], b[1])
    lon_min, lon_max = min(a[2], b[2]), max(a[3], b[3])
    lat0 = (lat_min + lat_max) / 2.0
    return max((lat_max - lat_min) * 111.32, (lon_max - lon_min) * _km_per_deg_lon(lat0))


def _merge_circle_groups(
    pieces: list[list[int]],
    lengths: list[float],
    lat: np.ndarray,
    lon: np.ndarray,
) -> list[dict]:
    """Join poor runs that sit within BAD_SPOT_MERGE_GAP_KM when the union still fits one circle.

    Runs farther apart stay as their own circles, including several circles inside
    one 10 km neighbourhood.
    """
    groups = [
        {"idx": list(idx), "length": length, "bbox": _bbox_of(idx, lat, lon)}
        for idx, length in zip(pieces, lengths)
        if idx
    ]
    changed = True
    while changed and len(groups) > 1:
        changed = False
        best: tuple[tuple[float, float], int, int] | None = None
        for i in range(len(groups)):
            for j in range(i + 1, len(groups)):
                gap = _bbox_gap_km(groups[i]["bbox"], groups[j]["bbox"])
                if gap > BAD_SPOT_MERGE_GAP_KM:
                    continue
                span = _union_span_km(groups[i]["bbox"], groups[j]["bbox"])
                if span > BAD_SPOT_GROUP_SPAN_KM:
                    continue
                key = (gap, span)
                if best is None or key < best[0]:
                    best = (key, i, j)
        if best is None:
            break
        _, i, j = best
        groups[i]["idx"].extend(groups[j]["idx"])
        groups[i]["length"] += groups[j]["length"]
        groups[i]["bbox"] = (
            min(groups[i]["bbox"][0], groups[j]["bbox"][0]),
            max(groups[i]["bbox"][1], groups[j]["bbox"][1]),
            min(groups[i]["bbox"][2], groups[j]["bbox"][2]),
            max(groups[i]["bbox"][3], groups[j]["bbox"][3]),
        )
        groups.pop(j)
        changed = True
    return groups


def _consecutive_bad_spots(
    track: pd.DataFrame,
    column: str,
    threshold: float,
    detect_column: str | None = None,
    min_length_m: float = BAD_SPOT_MIN_CONSEC_M,
) -> tuple[pd.DataFrame, np.ndarray]:
    """Rule 2: a run of consecutive poor road of at least min_length_m.

    Returns (spots, used-row mask).
    """
    detect_column = detect_column or column
    n = len(track)
    used = np.zeros(n, dtype=bool)
    if n == 0:
        return pd.DataFrame(), used
    lat = track["Latitude"].to_numpy(dtype=float)
    lon = track["Longitude"].to_numpy(dtype=float)
    time_s = pd.to_datetime(track["Time"]).to_numpy(dtype="datetime64[ns]").astype("int64") / 1e9
    same_pass = (
        track["Source"].to_numpy()
        if "Source" in track.columns
        else np.zeros(n, dtype=np.int64)
    )
    poor = _poor_mask(
        track[detect_column], threshold, strict_less=bool(BAD_SPOT_RULES[column].get("strict_less"))
    ).to_numpy()
    step_m = np.zeros(n)
    step_m[1:] = _equirect_km(lat[:-1], lon[:-1], lat[1:], lon[1:]) * 1000.0
    cum_m = np.cumsum(step_m)

    rows: list[dict] = []
    longest_short_m = 0.0
    pieces: list[list[int]] = []
    piece_lengths: list[float] = []
    for run in _poor_runs(np.flatnonzero(poor), same_pass, cum_m, time_s):
        for piece in _split_by_span(run, lat, lon):
            if not piece:
                continue
            # A lone poor sample still stands for the 50 m bin it was judged on.
            length_m = max(float(cum_m[piece[-1]] - cum_m[piece[0]]), MAP_BIN_M)
            if length_m < min_length_m:
                longest_short_m = max(longest_short_m, length_m)
                continue
            pieces.append(piece)
            piece_lengths.append(length_m)
            used[piece] = True
    for group in _merge_circle_groups(pieces, piece_lengths, lat, lon):
        chunk = track.iloc[sorted(group["idx"])]
        rows.append(
            _spot_from_points(
                chunk,
                column,
                threshold,
                kind="consecutive",
                length_m=float(group["length"]),
                area_km2=_hull_area_km2(chunk["Longitude"].to_numpy(), chunk["Latitude"].to_numpy()),
            )
        )
    spots = pd.DataFrame(rows)
    if not spots.empty:
        spots = spots.sort_values("length_m", ascending=False).reset_index(drop=True)
    spots.attrs["longest_short_m"] = longest_short_m
    return spots, used


def _discrete_area_bad_spots(leftover: pd.DataFrame, column: str, threshold: float) -> pd.DataFrame:
    """Rule 4: leftover poor bins covering >= 1 km² that never joined a poor run.

    Cluster on a ~1 km grid and only keep compact patches (not a thin road network).
    """
    if leftover.empty:
        return pd.DataFrame()
    g = leftover.reset_index(drop=True)
    latb = np.floor(g["Latitude"].to_numpy(dtype=float) / BAD_SPOT_GRID_DEG).astype(int)
    lonb = np.floor(g["Longitude"].to_numpy(dtype=float) / BAD_SPOT_GRID_DEG).astype(int)
    keys = list(zip(latb.tolist(), lonb.tolist()))
    cell_idx: dict[tuple[int, int], list[int]] = {}
    for i, cell in enumerate(keys):
        cell_idx.setdefault(cell, []).append(i)
    cells = set(cell_idx)
    neigh = ((1, 0), (-1, 0), (0, 1), (0, -1))
    seen: set[tuple[int, int]] = set()
    rows = []
    for cell in cells:
        if cell in seen:
            continue
        queue = deque([cell])
        seen.add(cell)
        group = [cell]
        while queue:
            y, x = queue.popleft()
            for dy, dx in neigh:
                nb = (y + dy, x + dx)
                if nb in cells and nb not in seen:
                    trial = group + [nb]
                    idx_trial = [i for c in trial for i in cell_idx[c]]
                    trial_part = g.iloc[idx_trial]
                    span_lat_km = (
                        float(trial_part["Latitude"].max()) - float(trial_part["Latitude"].min())
                    ) * 111.32
                    span_lon_km = (
                        float(trial_part["Longitude"].max()) - float(trial_part["Longitude"].min())
                    ) * _km_per_deg_lon(float(trial_part["Latitude"].mean()))
                    if max(span_lat_km, span_lon_km) > BAD_SPOT_DISCRETE_MAX_SPAN_KM:
                        continue
                    seen.add(nb)
                    queue.append(nb)
                    group.append(nb)
        idx = [i for c in group for i in cell_idx[c]]
        part = g.iloc[idx]
        if len(part) < BAD_SPOT_DISCRETE_MIN_POOR:
            continue
        area = _hull_area_km2(part["Longitude"].to_numpy(), part["Latitude"].to_numpy())
        span_lat_km = (float(part["Latitude"].max()) - float(part["Latitude"].min())) * 111.32
        span_lon_km = (float(part["Longitude"].max()) - float(part["Longitude"].min())) * _km_per_deg_lon(float(part["Latitude"].mean()))
        bbox_area = span_lat_km * span_lon_km
        # A thin road of leftover holes is not a 1 km² area.
        if min(span_lat_km, span_lon_km) < 0.45:
            continue
        if area < BAD_SPOT_DISCRETE_AREA_KM2 and bbox_area < BAD_SPOT_DISCRETE_AREA_KM2:
            continue
        rows.append(
            _spot_from_points(
                part,
                column,
                threshold,
                kind="discrete",
                length_m=np.nan,
                area_km2=max(area, bbox_area),
            )
        )
    spots = pd.DataFrame(rows)
    if spots.empty:
        return spots
    return spots.sort_values("area_km2", ascending=False).head(BAD_SPOT_MAX_DISCRETE).reset_index(drop=True)


def _spot_extra_stats(df: pd.DataFrame, spot: pd.Series) -> dict:
    box = df[
        df["Latitude"].between(spot["lat_min"], spot["lat_max"])
        & df["Longitude"].between(spot["lon_min"], spot["lon_max"])
    ]
    top_cell = pd.NA
    if "Cell Id" in box.columns and box["Cell Id"].notna().any():
        top_cell = int(box["Cell Id"].dropna().astype("int64").value_counts().idxmax())
    return {
        "mean_rsrp": float(box["RSRP"].mean()) if "RSRP" in box and box["RSRP"].notna().any() else np.nan,
        "mean_rsrq": float(box["RSRQ"].mean()) if "RSRQ" in box and box["RSRQ"].notna().any() else np.nan,
        "mean_sinr": float(box["SINR"].mean()) if "SINR" in box and box["SINR"].notna().any() else np.nan,
        "top_cell": top_cell,
    }


def find_bad_spots(df: pd.DataFrame, column: str, max_spots: int | None = None) -> pd.DataFrame:
    """Bad spots on the same best-server all-band track used by the first coverage map."""
    rule = BAD_SPOT_RULES[column]
    threshold = rule["threshold"]
    if max_spots is None:
        max_spots = int(rule.get("max_consecutive", BAD_SPOT_MAX_CONSEC)) + BAD_SPOT_MAX_DISCRETE
    track = _prepare_track(df, column)
    binned_col = f"{column}__binned"
    track[binned_col] = binned_kpi(track, column)
    consec, used = _consecutive_bad_spots(
        track,
        column,
        threshold,
        detect_column=binned_col,
        min_length_m=float(rule.get("min_consecutive_m", BAD_SPOT_MIN_CONSEC_M)),
    )
    longest_short_m = float(consec.attrs.get("longest_short_m", 0.0))
    total_consec = len(consec)
    max_consec = int(rule.get("max_consecutive", BAD_SPOT_MAX_CONSEC))
    if total_consec > max_consec:
        consec = consec.head(max_consec).copy()
    strict_less = bool(rule.get("strict_less"))
    leftover = track.iloc[np.flatnonzero(_poor_mask(track[binned_col], threshold, strict_less=strict_less).to_numpy() & ~used)]
    discrete = _discrete_area_bad_spots(leftover, column, threshold)
    if not discrete.empty:
        dense = [
            _map_poor_share(track, row, column, threshold) >= BAD_SPOT_DISCRETE_MIN_SHARE
            for _, row in discrete.iterrows()
        ]
        discrete = discrete.loc[dense].reset_index(drop=True)
    total_discrete = len(discrete)
    spots = pd.concat([consec, discrete], ignore_index=True)
    if spots.empty:
        return spots
    if len(spots) > max_spots:
        spots = spots.head(max_spots).copy()
    extras = [_spot_extra_stats(df, row) for _, row in spots.iterrows()]
    spots = spots.copy()
    spots["spot"] = np.arange(1, len(spots) + 1)
    spots["kpi"] = column
    spots["threshold"] = threshold
    spots["unit"] = rule["unit"]
    spots["mean_rsrp"] = [e["mean_rsrp"] for e in extras]
    spots["mean_rsrq"] = [e["mean_rsrq"] for e in extras]
    spots["mean_sinr"] = [e["mean_sinr"] for e in extras]
    spots["top_cell"] = [e["top_cell"] for e in extras]
    # Reported, not filtered on: every sample below the threshold is painted on top
    # of the good ones, so a stretch that is poor on one pass out of five is still
    # visible on the map and must still be circled.
    spots["map_poor_share"] = [_map_poor_share(track, row, column, threshold) for _, row in spots.iterrows()]
    spots.attrs["total_consecutive"] = total_consec
    spots.attrs["total_discrete"] = total_discrete
    spots.attrs["longest_short_m"] = longest_short_m
    return spots


def _map_poor_share(df: pd.DataFrame, spot: pd.Series, column: str, threshold: float) -> float:
    """Share of all-band best-server samples in the (padded) spot box that are poor."""
    lat_min = float(spot["lat_min"])
    lat_max = float(spot["lat_max"])
    lon_min = float(spot["lon_min"])
    lon_max = float(spot["lon_max"])
    dlat = max(lat_max - lat_min, 1e-5)
    dlon = max(lon_max - lon_min, 1e-5)
    box = df[
        df["Latitude"].between(lat_min - 0.15 * dlat, lat_max + 0.15 * dlat)
        & df["Longitude"].between(lon_min - 0.15 * dlon, lon_max + 0.15 * dlon)
    ]
    vals = pd.to_numeric(box[column], errors="coerce")
    mask = _poor_mask(vals, threshold, strict_less=bool(BAD_SPOT_RULES[column].get("strict_less")))
    n = int(vals.notna().sum())
    if n == 0:
        return 0.0
    return float(mask.sum() / n)


def _ellipse_from_points(
    lons: np.ndarray,
    lats: np.ndarray,
    min_km: float = 1.6,
    max_km: float = 6.0,
    length_km: float | None = None,
) -> tuple[float, float, float, float, float]:
    """Oval around a consecutive poor stretch; sized from the actual span/length."""
    lon0 = float(np.mean(lons))
    lat0 = float(np.mean(lats))
    km_lon = _km_per_deg_lon(lat0)
    span_lon_km = float(np.max(lons) - np.min(lons)) * km_lon if len(lons) else 0.0
    span_lat_km = float(np.max(lats) - np.min(lats)) * 111.32 if len(lats) else 0.0
    angle = 0.0
    if len(lons) >= 8:
        x = (lons - lon0) * km_lon
        y = (lats - lat0) * 111.32
        cov = np.cov(np.vstack([x, y]))
        eigvals, eigvecs = np.linalg.eigh(cov)
        order = np.argsort(eigvals)[::-1]
        eigvecs = eigvecs[:, order]
        angle = float(np.degrees(np.arctan2(eigvecs[1, 0], eigvecs[0, 0])))
    width_km = max(span_lon_km * 1.4, min_km)
    height_km = max(span_lat_km * 1.4, min_km * 0.7)
    if length_km:
        major = max(width_km, height_km, float(length_km) * 1.15)
        if width_km >= height_km:
            width_km = major
        else:
            height_km = major
    width_km = min(width_km, max_km)
    height_km = min(height_km, max_km)
    return lon0, lat0, width_km / km_lon, height_km / 111.32, angle


def _circle_from_bbox(lon_min: float, lon_max: float, lat_min: float, lat_max: float, min_diameter_km: float = 2.8, max_diameter_km: float = 5.5) -> tuple[float, float, float, float]:
    """Large circle covering a discrete poor area (>= 1 km²)."""
    lat0 = (lat_min + lat_max) / 2.0
    lon0 = (lon_min + lon_max) / 2.0
    span_lon_km = (lon_max - lon_min) * _km_per_deg_lon(lat0)
    span_lat_km = (lat_max - lat_min) * 111.32
    diameter_km = float(np.clip(max(span_lon_km, span_lat_km) * 1.25, min_diameter_km, max_diameter_km))
    return lon0, lat0, diameter_km / _km_per_deg_lon(lat0), diameter_km / 111.32


def add_map_scale_bar(ax, km: float | None = None, corner: str = "upper left") -> None:
    """Scale bar on a map. The length is a round fraction of the view."""
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    if xlim[1] <= xlim[0] or ylim[1] <= ylim[0]:
        return
    lat_mid = (ylim[0] + ylim[1]) / 2.0
    span_km = (xlim[1] - xlim[0]) * _km_per_deg_lon(lat_mid)
    if km is None:
        raw = span_km / 5.0
        km = 0.2
        for choice in (0.2, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0, 40.0):
            if choice <= raw * 1.35:
                km = choice
    if corner == "lower left":
        lat = ylim[0] + 0.08 * (ylim[1] - ylim[0])
    else:
        lat = ylim[1] - 0.09 * (ylim[1] - ylim[0])
    lon0 = xlim[0] + 0.045 * (xlim[1] - xlim[0])
    width = km / _km_per_deg_lon(lat)
    tick = 0.012 * (ylim[1] - ylim[0])
    ax.plot([lon0, lon0 + width], [lat, lat], color="white", lw=5.5, solid_capstyle="butt", zorder=40, clip_on=True)
    ax.plot([lon0, lon0 + width], [lat, lat], color="#1B1B1B", lw=2.2, solid_capstyle="butt", zorder=41, clip_on=True)
    ax.plot([lon0, lon0], [lat - tick, lat + tick], color="#1B1B1B", lw=1.3, zorder=41, clip_on=True)
    ax.plot([lon0 + width, lon0 + width], [lat - tick, lat + tick], color="#1B1B1B", lw=1.3, zorder=41, clip_on=True)
    label = f"{km:g} km" if km >= 1 else f"{int(round(km * 1000))} m"
    ax.text(
        lon0 + width / 2.0, lat + tick, label,
        ha="center", va="bottom", fontsize=8, color="#1B1B1B", fontweight="bold", zorder=42, clip_on=True,
        bbox={"boxstyle": "round,pad=0.15", "fc": "white", "ec": "#1B1B1B", "lw": 0.6, "alpha": 0.94},
    )


def _spot_outline(spot: pd.Series) -> tuple[float, float, float, float, float, tuple]:
    """Oval/circle that encloses the whole poor stretch, including red tails."""
    kind = str(spot.get("kind", "consecutive"))
    style = (0, (1.0, 1.4)) if kind == "discrete" else (0, (3.0, 2.0))
    lat0 = float(spot["outline_lat"]) if "outline_lat" in spot.index and pd.notna(spot.get("outline_lat")) else (float(spot["lat_min"]) + float(spot["lat_max"])) / 2.0
    lon0 = float(spot["outline_lon"]) if "outline_lon" in spot.index and pd.notna(spot.get("outline_lon")) else (float(spot["lon_min"]) + float(spot["lon_max"])) / 2.0
    km_lon = _km_per_deg_lon(lat0)
    if "outline_width_km" in spot.index and pd.notna(spot.get("outline_width_km")):
        width_km = float(spot["outline_width_km"])
        height_km = float(spot["outline_height_km"])
        angle = float(spot.get("outline_angle", 0.0) or 0.0)
        return lon0, lat0, width_km / km_lon, height_km / 111.32, angle, style
    span_lon_km = max((float(spot["lon_max"]) - float(spot["lon_min"])) * km_lon, 0.2)
    span_lat_km = max((float(spot["lat_max"]) - float(spot["lat_min"])) * 111.32, 0.2)
    cover = float(np.sqrt(2.0)) * BAD_SPOT_OUTLINE_PAD / 1.15
    width_km = max(span_lon_km * cover, 0.45)
    height_km = max(span_lat_km * cover, 0.38)
    if kind == "discrete":
        diameter_km = min(max(width_km, height_km, 1.15), 4.5)
        return lon0, lat0, diameter_km / km_lon, diameter_km / 111.32, 0.0, style
    return lon0, lat0, width_km / km_lon, height_km / 111.32, 0.0, style


def overview_ring_style(spot_count: int) -> tuple[float, float, bool]:
    """Ring floor (km), line width and halo flag for this many spots on one map."""
    for limit, min_km, lw, halo in BAD_SPOT_OVERVIEW_TIERS:
        if spot_count <= limit:
            return min_km, lw, halo
    return BAD_SPOT_OVERVIEW_TIERS[-1][1:]


def _add_spot_outline(
    ax,
    spot: pd.Series,
    *,
    overview: bool = False,
    style_tier: tuple[float, float, bool] = (BAD_SPOT_OVERVIEW_MIN_KM, BAD_SPOT_OVERVIEW_LW, True),
) -> None:
    cx, cy, w, h, angle, style = _spot_outline(spot)
    lat0 = cy
    km_lon = _km_per_deg_lon(lat0)
    if overview:
        min_diameter_km, lw, halo = style_tier
        width_km = abs(w) * km_lon
        height_km = abs(h) * 111.32
        diameter_km = float(np.clip(max(width_km, height_km, min_diameter_km), min_diameter_km, BAD_SPOT_OVERVIEW_MAX_KM))
        w = diameter_km / km_lon
        h = diameter_km / 111.32
        angle = 0.0
        if halo:
            ax.add_patch(
                Ellipse(
                    (cx, cy),
                    width=w,
                    height=h,
                    angle=angle,
                    fill=False,
                    edgecolor="#FFFFFF",
                    linestyle="solid",
                    linewidth=lw + 2.0,
                    zorder=19,
                )
            )
    else:
        lw = 1.0
    # Black under the orange stroke so the circle shows on both dark water and pale roofs.
    ax.add_patch(
        Ellipse(
            (cx, cy),
            width=w,
            height=h,
            angle=angle,
            fill=False,
            edgecolor="#1B1B1B",
            linestyle=style,
            linewidth=lw + 1.6,
            zorder=19,
        )
    )
    ax.add_patch(
        Ellipse(
            (cx, cy),
            width=w,
            height=h,
            angle=angle,
            fill=False,
            edgecolor="#FF6600",
            linestyle=style,
            linewidth=lw,
            zorder=20,
        )
    )


def _zoom_extent(spot: pd.Series) -> tuple[float, float, float, float]:
    lat0 = float(spot["outline_lat"]) if "outline_lat" in getattr(spot, "index", []) and pd.notna(spot.get("outline_lat")) else float(spot["lat"])
    lon0 = float(spot["outline_lon"]) if "outline_lon" in getattr(spot, "index", []) and pd.notna(spot.get("outline_lon")) else float(spot["lon"])
    km_lon = _km_per_deg_lon(lat0)
    span_lat_km = max((float(spot["lat_max"]) - float(spot["lat_min"])) * 111.32, 0.2)
    span_lon_km = max((float(spot["lon_max"]) - float(spot["lon_min"])) * km_lon, 0.2)
    if "outline_width_km" in getattr(spot, "index", []) and pd.notna(spot.get("outline_width_km")):
        span_lon_km = max(span_lon_km, float(spot["outline_width_km"]))
        span_lat_km = max(span_lat_km, float(spot["outline_height_km"]))
    half = max(BAD_SPOT_ZOOM_MIN_KM, span_lat_km, span_lon_km) / 2.0 + BAD_SPOT_ZOOM_PAD_KM
    return (
        lon0 - half / km_lon,
        lon0 + half / km_lon,
        lat0 - half / 111.32,
        lat0 + half / 111.32,
    )


def plot_bad_spot_map(
    df: pd.DataFrame,
    path: Path,
    column: str,
    class_series: pd.Series,
    labels: list[str],
    colors: list[str],
    spots: pd.DataFrame,
    title: str,
    legend_title: str,
    max_points: int = MAP_MAX_POINTS,
    extent: tuple[float, float, float, float] | None = None,
    basemap: bool = True,
    shapes: bool = False,
    outlines: bool = False,
    basemap_style: str = "satellite",
    route_casing: bool = True,
    legend_inside: bool = False,
    legend_counts: list[int] | None = None,
) -> Path:
    """First all-band coverage map.

    A .svg path writes a pure vector file: no basemap, transparent background,
    and text kept as text. shapes=True leaves the circles out of the picture and
    records one geometry per circle so the workbook can place a separate shape
    for each of them. outlines=True paints the circles into the picture. The
    combined view uses neither: its circles live only on the individual bad-spot maps.
    """
    PLOTS_DIR.mkdir(parents=True, exist_ok=True)
    vector = path.suffix.lower() == ".svg"
    if vector:
        basemap = False
    geo = df.dropna(subset=["Longitude", "Latitude", column]).copy()
    fig, ax = plt.subplots(figsize=(8.0, 9.2))
    _draw_coverage_points(
        ax, geo, class_series, labels, colors, max_points=max_points,
        bin_m=MAP_BIN_M, column=column, route_casing=route_casing,
    )
    if extent is not None:
        ax.set_xlim(extent[0], extent[1])
        ax.set_ylim(extent[2], extent[3])
    elif not geo.empty:
        lon_min, lon_max, lat_min, lat_max = _coverage_extent_from_geo(geo)
        ax.set_xlim(lon_min, lon_max)
        ax.set_ylim(lat_min, lat_max)
    ax.set_aspect("equal", adjustable="box")
    if basemap and basemap_style == "boundary":
        _add_boundary_basemap(ax)
    elif basemap:
        _add_basemap(ax)
    if not vector:
        add_map_scale_bar(ax, corner="lower left" if legend_inside else "upper left")
    if legend_inside:
        counts = legend_counts if legend_counts is not None else [0] * len(labels)
        legend = _place_count_legend(ax, _count_legend_handles(labels, colors, counts), legend_title)
    else:
        handles = _coverage_legend_handles(labels, colors)
        handles.extend(
            [
                Line2D([0], [0], color="#FF6600", lw=2.2, linestyle=(0, (3, 2)), label=consecutive_label(column)),
                Line2D([0], [0], color="#FF6600", lw=2.2, linestyle=(0, (1, 1.4)), label="Discrete area ≥ 1 km²"),
            ]
        )
        legend = ax.legend(
            handles=handles,
            title=legend_title,
            loc="center left",
            bbox_to_anchor=(1.02, 0.5),
            frameon=True,
            fontsize=8,
            title_fontsize=10,
        )
    if spots is not None and not spots.empty and outlines and not shapes:
        for _, spot in spots.iterrows():
            _add_spot_outline(ax, spot, overview=False)
    if not legend_inside:
        ax.set_title(title)
    hide_map_axes(ax)
    fig.tight_layout()
    _save_map(fig, path, legends=[legend], ax=ax if shapes else None, spots=spots if shapes else None)
    return path


def _record_circle_pixels(
    fig, ax, path: Path, spots: pd.DataFrame, tight, axes=None, name_start: int = 1,
) -> None:
    """Store each circle's box in pixels of the PNG that savefig is about to write.

    axes, when set, is one axes per spot, in the same order. name_start keeps a
    single-spot map named Circle 7 when it is bad spot 7.
    """
    dpi = float(fig.dpi)
    save_dpi = float(MAP_DPI)
    circles = []
    for i, (_, spot) in enumerate(spots.iterrows()):
        n = name_start + i
        use_ax = axes[i] if axes is not None else ax
        cx, cy, w, h, _angle, _style = _spot_outline(spot)
        x0, y_lo = use_ax.transData.transform((cx - w / 2.0, cy - h / 2.0))
        x1, y_hi = use_ax.transData.transform((cx + w / 2.0, cy + h / 2.0))

        def to_saved_x(px: float) -> float:
            return (px / dpi - tight.x0) * save_dpi

        def to_saved_y_top(py: float) -> float:
            return (tight.y1 - py / dpi) * save_dpi

        left, right = to_saved_x(x0), to_saved_x(x1)
        top, bottom = to_saved_y_top(y_hi), to_saved_y_top(y_lo)
        circles.append({
            "name": f"Circle {n}",
            "x": float(min(left, right)),
            "y": float(min(top, bottom)),
            "w": float(abs(right - left)),
            "h": float(abs(bottom - top)),
            "dotted": str(spot.get("kind", "consecutive")) == "discrete",
        })
    MAP_CIRCLE_PIXELS[str(path.resolve())] = {
        "size": (tight.width * save_dpi, tight.height * save_dpi),
        "circles": circles,
    }


def _save_map(
    fig, path: Path, legends=(), ax=None, spots: pd.DataFrame | None = None, axes=None, name_start: int = 1,
) -> None:
    """PNG at MAP_DPI on white, or for .svg a transparent vector file with text kept as text.

    When spots is set, each circle is recorded against the PNG and is not drawn
    into it. The workbook turns those records into separate shapes. axes is one
    axes per spot for a grid of zoom panels.
    """
    pad = 0.15
    if path.suffix.lower() == ".svg":
        fig.patch.set_alpha(0.0)
        for axis in fig.axes:
            axis.patch.set_alpha(0.0)
        for legend in legends:
            legend.get_frame().set_facecolor("none")
        with plt.rc_context({"svg.fonttype": "none", "svg.hashsalt": "bogura"}):
            fig.savefig(path, format="svg", transparent=True, bbox_inches="tight", pad_inches=pad)
    else:
        bbox_arg: object = "tight"
        if spots is not None and not spots.empty and (ax is not None or axes is not None):
            fig.canvas.draw()
            tight = fig.get_tightbbox(fig.canvas.get_renderer()).padded(pad)
            _record_circle_pixels(fig, ax, path, spots, tight, axes=axes, name_start=name_start)
            bbox_arg = tight
            pad = 0.0
        fig.savefig(path, dpi=MAP_DPI, facecolor="white", bbox_inches=bbox_arg, pad_inches=pad)
    plt.close(fig)


def plot_bad_spot_zoom_grid(
    df: pd.DataFrame,
    spots: pd.DataFrame,
    path: Path,
    column: str,
    class_series: pd.Series,
    labels: list[str],
    colors: list[str],
    title: str,
    shapes: bool = False,
) -> Path | None:
    """Zoomed all-band coverage around each identified spot, with no numbers.

    A .svg path writes a pure vector file without the basemap, like plot_bad_spot_map.
    shapes=True leaves each circle out of the picture and records it so the workbook
    can place a separate shape on that panel.
    """
    if spots is None or spots.empty:
        return None
    PLOTS_DIR.mkdir(parents=True, exist_ok=True)
    vector = path.suffix.lower() == ".svg"
    work = spots.head(BAD_SPOT_MAX_ZOOMS)
    n = len(work)
    cols = BAD_SPOT_ZOOM_COLS
    rows = max(1, (n + cols - 1) // cols)
    fig, axes = plt.subplots(rows, cols, figsize=(3.15 * cols, 3.35 * rows))
    axes_flat = np.atleast_1d(axes).ravel()
    geo = df.dropna(subset=["Longitude", "Latitude", column]).copy()
    panel_axes = []
    for i, ax in enumerate(axes_flat):
        if i >= n:
            ax.set_axis_off()
            continue
        spot = work.iloc[i]
        zext = _zoom_extent(spot)
        local = geo[
            geo["Longitude"].between(zext[0], zext[1])
            & geo["Latitude"].between(zext[2], zext[3])
        ]
        _draw_coverage_points(
            ax,
            local,
            class_series.reindex(local.index),
            labels,
            colors,
            max_points=8000,
            poor_on_top=True,
            bin_m=MAP_BIN_M,
            column=column,
        )
        ax.set_xlim(zext[0], zext[1])
        ax.set_ylim(zext[2], zext[3])
        ax.set_aspect("equal", adjustable="box")
        if not vector:
            _add_basemap(ax)
            add_map_scale_bar(ax)
        if not vector:
            draw_bad_spot_sector_pies(ax, spot)
        if shapes:
            panel_axes.append(ax)
        else:
            _add_spot_outline(ax, spot)
        hide_map_axes(ax)
        kind = str(spot.get("kind", "consecutive"))
        if kind == "discrete":
            caption = f"Discrete {float(spot['area_km2']):.1f} km²"
        else:
            word = "Poor run" if min_consecutive_m(column) <= 0 else "Consecutive"
            caption = f"{word} {float(spot['length_m']):.0f} m"
        ax.set_title(caption, fontsize=8)
    fig.suptitle(title, fontsize=11, fontweight="bold")
    fig.tight_layout()
    if shapes and panel_axes:
        _save_map(fig, path, axes=panel_axes, spots=work.reset_index(drop=True))
    else:
        _save_map(fig, path)
    return path


def plot_bad_spot_serial_maps(
    df: pd.DataFrame,
    spots: pd.DataFrame,
    directory: Path,
    stem: str,
    column: str,
    class_series: pd.Series,
    labels: list[str],
    colors: list[str],
    *,
    basemap_style: str = "satellite",
    route_casing: bool = True,
) -> list[Path]:
    """One large map per bad spot, longest first.

    The circle is recorded and not painted, so the workbook can place one movable
    shape on that map. Sector pies and distance lines stay in the picture. The
    combined view does not use this function.
    """
    if spots is None or spots.empty:
        return []
    directory.mkdir(parents=True, exist_ok=True)
    work = spots.head(BAD_SPOT_MAX_ZOOMS)
    geo = df.dropna(subset=["Longitude", "Latitude", column]).copy()
    paths: list[Path] = []
    total = len(work)
    for i, (_, spot) in enumerate(work.iterrows(), start=1):
        path = directory / f"{stem}_{i:02d}.png"
        print(f"  {stem} bad spot {i}/{total}", flush=True)
        fig, ax = plt.subplots(figsize=(9.4, 8.6))
        zext = _zoom_extent(spot)
        # Room for every sector arm when the site sits near the edge of the spot.
        arm_km = 0.9
        mid_lat = (zext[2] + zext[3]) / 2.0
        arm_lon = arm_km / _km_per_deg_lon(mid_lat)
        arm_lat = arm_km / 111.32
        zext = (zext[0] - arm_lon, zext[1] + arm_lon, zext[2] - arm_lat, zext[3] + arm_lat)
        proposal = bad_spot_advice(df, column, spot).get("draw")
        if proposal:
            zext = _extent_with_proposal(zext, proposal)
        local = geo[
            geo["Longitude"].between(zext[0], zext[1])
            & geo["Latitude"].between(zext[2], zext[3])
        ]
        _draw_coverage_points(
            ax,
            local,
            class_series.reindex(local.index),
            labels,
            colors,
            max_points=8000,
            poor_on_top=True,
            bin_m=MAP_BIN_M,
            column=column,
            route_casing=route_casing,
        )
        ax.set_xlim(zext[0], zext[1])
        ax.set_ylim(zext[2], zext[3])
        ax.set_aspect("equal", adjustable="box")
        if basemap_style == "boundary":
            _add_boundary_basemap(ax)
        else:
            _add_basemap(ax)
        draw_bad_spot_sector_pies(ax, spot, local, column)
        _draw_proposal(ax, proposal)
        blanks = _blank_segments(df, spot)
        for gap in blanks:
            ax.plot(
                [gap["lon1"], gap["lon2"]],
                [gap["lat1"], gap["lat2"]],
                color="#111111",
                linewidth=3.2,
                linestyle=(0, (1.2, 1.4)),
                solid_capstyle="round",
                zorder=24,
            )
            ax.plot(
                [gap["lon1"], gap["lon2"]],
                [gap["lat1"], gap["lat2"]],
                color="#FFFFFF",
                linewidth=1.5,
                linestyle=(0, (1.2, 1.4)),
                solid_capstyle="round",
                zorder=25,
            )
        add_map_scale_bar(ax)
        hide_map_axes(ax)
        kind = str(spot.get("kind", "consecutive"))
        blank_m = sum(gap["length_m"] for gap in blanks)
        if kind == "discrete":
            caption = f"Discrete {float(spot['area_km2']):.1f} km²"
        else:
            word = "Poor run" if min_consecutive_m(column) <= 0 else "Consecutive"
            caption = f"{word} {float(spot['length_m']):.0f} m"
        if blank_m >= 80:
            caption = f"{caption}  +  blank {blank_m:,.0f} m"
        ax.set_title(caption, fontsize=14, fontweight="bold")
        fig.tight_layout()
        one = work.iloc[[i - 1]].reset_index(drop=True)
        _save_map(fig, path, ax=ax, spots=one, name_start=i)
        paths.append(path)
    return paths


def plot_rsrp_bad_spot_map(
    df: pd.DataFrame,
    spots: pd.DataFrame,
    path: Path,
    *,
    extent: tuple[float, float, float, float] | None = None,
    title: str = "Combined view — RSRP",
    shapes: bool = False,
    outlines: bool = False,
    legend_title: str = "Idle by RSRP",
) -> Path:
    """Satellite map coloured by the four-bin RSRP legend, counts on the left.

    The basemap is the same Esri imagery as the RSRP sheet. The coloured trace
    is the KPI, with the same white casing the coverage maps use.
    """
    counts = _legend_sample_counts(df, "RSRP", rsrp_map_class, MAP_RSRP_LABELS)
    return plot_bad_spot_map(
        df,
        path,
        column="RSRP",
        class_series=rsrp_map_class(df["RSRP"]),
        labels=MAP_RSRP_LABELS,
        colors=MAP_RSRP_COLORS,
        spots=spots,
        title=title,
        legend_title=legend_title,
        extent=extent,
        shapes=shapes,
        outlines=outlines,
        basemap_style="satellite",
        route_casing=True,
        legend_inside=True,
        legend_counts=counts,
    )


def export_rsrp_map_svg(df: pd.DataFrame, path: Path | None = None) -> Path:
    """Vector copy of the RSRP sheet's combined Idle map, from the same track and spots."""
    track = best_server(df)
    spots = find_bad_spots(track, "RSRP")
    out = path or (PLOTS_DIR / "svg" / "rsrp_combined_idle.svg")
    out.parent.mkdir(parents=True, exist_ok=True)
    return plot_rsrp_bad_spot_map(track, spots, out, extent=coverage_extent(track))


def plot_rsrq_bad_spot_map(
    df: pd.DataFrame,
    spots: pd.DataFrame,
    path: Path,
    *,
    extent: tuple[float, float, float, float] | None = None,
    title: str = "Combined view — RSRQ",
    shapes: bool = False,
    outlines: bool = False,
    legend_title: str = "Idle by RSRQ",
) -> Path:
    """Satellite map coloured by the three-bin RSRQ legend, counts on the left.

    The basemap is the same Esri imagery as the RSRP sheet. The coloured trace
    is the KPI, with the same white casing the coverage maps use.
    """
    counts = _legend_sample_counts(df, "RSRQ", rsrq_map_class, MAP_RSRQ_LABELS)
    return plot_bad_spot_map(
        df,
        path,
        column="RSRQ",
        class_series=rsrq_map_class(df["RSRQ"]),
        labels=MAP_RSRQ_LABELS,
        colors=MAP_RSRQ_COLORS,
        spots=spots,
        title=title,
        legend_title=legend_title,
        extent=extent,
        shapes=shapes,
        outlines=outlines,
        basemap_style="satellite",
        route_casing=True,
        legend_inside=True,
        legend_counts=counts,
    )


def plot_sinr_bad_spot_map(
    df: pd.DataFrame,
    spots: pd.DataFrame,
    path: Path,
    *,
    extent: tuple[float, float, float, float] | None = None,
    title: str = "Combined view — SINR",
    shapes: bool = False,
    outlines: bool = False,
) -> Path:
    return plot_bad_spot_map(
        df,
        path,
        column="SINR",
        class_series=sinr_map_class(df["SINR"]),
        labels=MAP_SINR_LABELS,
        colors=MAP_SINR_COLORS,
        spots=spots,
        title=title,
        legend_title="SINR (dB)",
        extent=extent,
        shapes=shapes,
        outlines=outlines,
    )


## 10. Read the logs and merge them

Extract the 7z archives, keep Robi M-PCell on the Active log, and keep the strongest RSRP at each location.
Idle still keeps the strongest band in each scanner sweep, at most 4 s and 40 m.


In [ ]:
# ------------------------------------------------------------------------
# 10. Read the logs and merge them
# Extract the 7z archives, keep Robi M-PCell on the Active log, and keep the strongest RSRP at each location.
# Idle still keeps the strongest band in each scanner sweep, at most 4 s and 40 m.
# ------------------------------------------------------------------------

def extract_7z(archive: Path, dest: Path) -> None:
    """Extract a 7z archive. Uses the 7z binary when it is installed, else py7zr."""
    from shutil import which

    dest.mkdir(parents=True, exist_ok=True)
    exe = which("7z") or which("7za")
    if exe:
        subprocess.run([exe, "x", f"-o{dest}", str(archive), "-y"], check=True)
        return
    import py7zr

    with py7zr.SevenZipFile(archive) as zf:
        zf.extractall(path=dest)


def _numbered_csvs(directory: Path) -> list[Path]:
    if not directory.exists():
        return []
    return sorted(directory.glob("*.csv"), key=lambda p: int(p.stem) if p.stem.isdigit() else p.stem)


def idle_csv_files() -> list[Path]:
    return _numbered_csvs(IDLE_DIR)


def active_csv_files() -> list[Path]:
    return _numbered_csvs(ACTIVE_DIR)


def extract_archives() -> None:
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    if not IDLE_ARCHIVE.exists():
        sys.exit(f"Missing idle archive: {IDLE_ARCHIVE.name}")
    print(f"Extracting {IDLE_ARCHIVE.name} -> {IDLE_DIR}")
    extract_7z(IDLE_ARCHIVE, EXTRACT_DIR)
    files = idle_csv_files()
    if not files:
        sys.exit(f"Extraction finished but no CSV files were found in {IDLE_DIR}")
    print(f"  {len(files)} CSV files")


def _cgi_eci(value: object) -> object:
    """Last number of a CGI such as 470-2-168962068, which is the E-UTRAN cell identity."""
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return pd.NA
    text = str(value).strip()
    if not text or text.lower() in {"nan", "<na>", "none"}:
        return pd.NA
    token = text.replace(" ", "").split("-")[-1]
    if not token.isdigit():
        return pd.NA
    return int(token)


def _parse_active_time(series: pd.Series) -> pd.Series:
    """Parse either a full timestamp or a clock with no date."""
    raw = series.astype("string").str.strip()
    clock = raw.str.match(r"^\d{1,2}:\d{2}:\d{2}\s*[AaPp][Mm]$", na=False)
    if float(clock.mean()) > 0.9:
        parsed = pd.to_datetime(raw.where(clock), format="%I:%M:%S %p", errors="coerce")
        return ACTIVE_CLOCK_ANCHOR + (parsed - parsed.dt.normalize())
    return pd.to_datetime(series, errors="coerce")


def earfcn_to_layer(earfcn: object) -> str:
    """Map DL EARFCN to L900 / L1800 / L2100 / L2600. Unknown EARFCNs return ''."""
    if pd.isna(earfcn):
        return ""
    try:
        value = int(earfcn)
    except (TypeError, ValueError):
        return ""
    for layer, ranges in LAYER_EARFCN_RANGES.items():
        for lo, hi in ranges:
            if lo <= value <= hi:
                return layer
    return ""


def filter_by_layer(df: pd.DataFrame, layer: str) -> pd.DataFrame:
    if "Layer" in df.columns:
        return df.loc[df["Layer"] == layer].copy()
    earfcn = pd.to_numeric(df["DL EARFCN"], errors="coerce")
    return df.loc[earfcn.map(earfcn_to_layer) == layer].copy()


def _sweep_group_ids(
    source: np.ndarray,
    time_s: np.ndarray,
    lat: np.ndarray,
    lon: np.ndarray,
    max_s: float = BEST_SERVER_SWEEP_S,
    max_m: float = BEST_SERVER_SWEEP_M,
) -> np.ndarray:
    """Split a pass into scanner sweeps: a new group every max_s seconds or max_m metres."""
    n = len(time_s)
    ids = np.empty(n, dtype=np.int64)
    if n == 0:
        return ids
    step_m = np.zeros(n)
    step_m[1:] = _equirect_km(lat[:-1], lon[:-1], lat[1:], lon[1:]) * 1000.0
    cum_m = np.cumsum(step_m)
    gid = 0
    t0 = time_s[0]
    d0 = cum_m[0]
    src0 = source[0]
    for i in range(n):
        if source[i] != src0 or time_s[i] - t0 > max_s or cum_m[i] - d0 > max_m:
            gid += 1
            t0 = time_s[i]
            d0 = cum_m[i]
            src0 = source[i]
        ids[i] = gid
    return ids


def strongest_fix(df: pd.DataFrame) -> pd.DataFrame:
    """One row per latitude, longitude and timestamp: the strongest RSRP.

    RSRQ and SINR stay on that same row. Any other row at that fix is dropped.
    """
    work = df.dropna(subset=["RSRP", "Time", "Latitude", "Longitude"]).copy()
    if work.empty:
        return work
    work = work.reset_index(drop=True)
    work["RSRP"] = pd.to_numeric(work["RSRP"], errors="coerce")
    work["Latitude"] = pd.to_numeric(work["Latitude"], errors="coerce")
    work["Longitude"] = pd.to_numeric(work["Longitude"], errors="coerce")
    work["Time"] = pd.to_datetime(work["Time"], errors="coerce")
    work = work.dropna(subset=["RSRP", "Latitude", "Longitude", "Time"])
    if work.empty:
        return work
    idx = work.groupby(["Time", "Latitude", "Longitude"], sort=False)["RSRP"].idxmax()
    return work.loc[idx].reset_index(drop=True)


def best_server(df: pd.DataFrame) -> pd.DataFrame:
    """All-band best server: strongest RSRP per scanner sweep.

    Rows that share a latitude, longitude and timestamp are reduced to the
    strongest RSRP first. The scanner still writes one row per band with its own
    millisecond timestamp, so the remaining rows are grouped into short sweep
    windows and the strongest band in that window is kept.
    """
    work = strongest_fix(df)
    if work.empty:
        return work
    sort_cols = ["Source", "Time"] if "Source" in work.columns else ["Time"]
    work = work.sort_values(sort_cols).reset_index(drop=True)
    source = (
        work["Source"].to_numpy()
        if "Source" in work.columns
        else np.zeros(len(work), dtype=np.int64)
    )
    time_s = work["Time"].to_numpy(dtype="datetime64[ns]").astype("int64") / 1e9
    work["_sweep"] = _sweep_group_ids(
        source,
        time_s,
        work["Latitude"].to_numpy(dtype=float),
        work["Longitude"].to_numpy(dtype=float),
    )
    idx = work.groupby("_sweep", sort=False)["RSRP"].idxmax()
    return work.loc[idx].drop(columns=["_sweep"]).reset_index(drop=True)


def strongest_location(df: pd.DataFrame) -> pd.DataFrame:
    """Active auto mode: one band at each latitude and longitude.

    Rows that share a latitude, longitude and Cell Id collapse to the strongest
    RSRP, so one cell keeps one band. A secondary carrier with no CGI has no
    Cell Id; those rows still compete, one per EARFCN at that location. The
    strongest RSRP remains. RSRQ and SINR stay on that row.
    """
    work = df.dropna(subset=["RSRP", "Latitude", "Longitude"]).copy()
    if work.empty:
        return work
    work = work.reset_index(drop=True)
    work["RSRP"] = pd.to_numeric(work["RSRP"], errors="coerce")
    work["Latitude"] = pd.to_numeric(work["Latitude"], errors="coerce")
    work["Longitude"] = pd.to_numeric(work["Longitude"], errors="coerce")
    work["Cell Id"] = pd.to_numeric(work["Cell Id"], errors="coerce")
    work = work.dropna(subset=["RSRP", "Latitude", "Longitude"])
    if work.empty:
        return work
    identified = work.loc[work["Cell Id"].notna()]
    anonymous = work.loc[work["Cell Id"].isna()]
    parts = []
    if not identified.empty:
        per_cell = identified.groupby(["Latitude", "Longitude", "Cell Id"], sort=False)["RSRP"].idxmax()
        parts.append(identified.loc[per_cell])
    if not anonymous.empty:
        keys = ["Latitude", "Longitude"]
        if "DL EARFCN" in anonymous.columns:
            keys.append("DL EARFCN")
        per_carrier = anonymous.groupby(keys, sort=False, dropna=False)["RSRP"].idxmax()
        parts.append(anonymous.loc[per_carrier])
    if not parts:
        return work.iloc[0:0].copy()
    work = pd.concat(parts, ignore_index=True)
    per_loc = work.groupby(["Latitude", "Longitude"], sort=False)["RSRP"].idxmax()
    return work.loc[per_loc].reset_index(drop=True)


def select_active_report(df: pd.DataFrame) -> pd.DataFrame:
    """Active rows used on the Excel and web reports.

    Home Operator is stored as Operator. Only Robi primary cells (Type M-PCell)
    are drawn. Other operators and secondary cells stay in the merged CSV.
    """
    if df is None or df.empty:
        return df
    if "Type" not in df.columns:
        sys.exit("Active log has no Type column, so M-PCell cannot be selected")
    work = df
    if "Operator" in work.columns:
        operator = work["Operator"].astype("string").str.strip()
        work = work.loc[operator.eq(ACTIVE_OPERATOR)]
    kind = work["Type"].astype("string").str.strip()
    return work.loc[kind.eq(ACTIVE_CELL_TYPE)].copy()


def best_carrier(df: pd.DataFrame) -> pd.DataFrame:
    """Active Mode: one Robi row per latitude and longitude.

    The UE is in auto mode, so one band is plotted at a location. That band is
    the strongest RSRP, and its RSRQ and SINR are the values that are plotted.
    """
    return strongest_location(df)


def coverage_extent(df: pd.DataFrame) -> tuple[float, float, float, float]:
    """Padded lon/lat box so all-band and per-band maps share the same view."""
    geo = df.dropna(subset=["Longitude", "Latitude"])
    lon_min, lon_max = float(geo["Longitude"].min()), float(geo["Longitude"].max())
    lat_min, lat_max = float(geo["Latitude"].min()), float(geo["Latitude"].max())
    pad_lon = 0.02 * (lon_max - lon_min)
    pad_lat = 0.02 * (lat_max - lat_min)
    return lon_min - pad_lon, lon_max + pad_lon, lat_min - pad_lat, lat_max + pad_lat


def earfcn_to_band(earfcn: float) -> str:
    if pd.isna(earfcn):
        return "Unknown"
    value = int(earfcn)
    for band, (lo, hi, label) in EARFCN_BANDS.items():
        if lo <= value <= hi:
            return f"{label} ({band})"
    return f"EARFCN {value}"


def load_idle_csv(path: Path) -> pd.DataFrame:
    print(f"Loading {path.name} ...")
    df = pd.read_csv(path)
    df = df.rename(columns=SCANNER_RENAME)
    missing = [c for c in COLUMNS if c not in df.columns]
    if missing:
        sys.exit(f"{path.name} is missing columns: {missing}")
    keep = [c for c in df.columns if c in set(COLUMNS) | {
        "Operator", "Technology Band", "eNB-Id", "Sector Id",
        "Bin Name", "Collection", "Campaign",
    }]
    df = df[keep].copy()
    df["Time"] = pd.to_datetime(df["Time"], errors="coerce")
    for col in ("RSRP", "RSRQ", "SINR", "Longitude", "Latitude", "DL EARFCN"):
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["Cell Id"] = pd.to_numeric(df["Cell Id"], errors="coerce").astype("Int64")
    df["DL EARFCN"] = df["DL EARFCN"].astype("Int64")
    df["Source"] = path.name
    df["Band"] = df["DL EARFCN"].map(earfcn_to_band)
    df["Layer"] = df["DL EARFCN"].map(earfcn_to_layer)
    print(f"  {len(df):,} rows")
    return df


def merge_parts() -> pd.DataFrame:
    files = idle_csv_files()
    if not files:
        sys.exit(f"No idle CSV files in {IDLE_DIR}; run extract first")
    frames = [load_idle_csv(p) for p in files]
    merged = pd.concat(frames, ignore_index=True)
    merged = merged.drop_duplicates(subset=["Time", "Cell Id", "DL EARFCN", "RSRP"], keep="first")
    merged = merged.sort_values(["Time", "Source"], kind="mergesort").reset_index(drop=True)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    merged.to_csv(MERGED_CSV, index=False, compression="gzip")
    print(f"Wrote merged file: {MERGED_CSV} ({len(merged):,} rows)")
    return merged


def load_active_csv(path: Path) -> pd.DataFrame:
    """Active Mode UE export. KPIs are already named RSRP / RSRQ / SINR.

    Older exports name the operator Home Operator and carry the cell as CGI.
    Borga_Active_mode_11Nov26 already has Operator and Cell Id. Cell Id is the
    E-UTRAN cell identity. When only a CGI is present, Cell Id is its last number.
    A secondary cell often has no CGI, so its Cell Id stays empty.
    """
    print(f"Loading active {path.name} ...")
    df = pd.read_csv(path)
    df = df.loc[:, ~df.columns.astype(str).str.startswith("Unnamed")].copy()
    if "Home Operator" in df.columns and "Operator" not in df.columns:
        df = df.rename(columns={"Home Operator": "Operator"})
    if "Cell Id" not in df.columns:
        if "CGI" not in df.columns:
            sys.exit(f"{path.name} has no Cell Id and no CGI")
        df["Cell Id"] = df["CGI"].map(_cgi_eci)
    missing = [c for c in COLUMNS if c not in df.columns]
    if missing:
        sys.exit(f"{path.name} is missing columns: {missing}")
    keep = [c for c in df.columns if c in set(COLUMNS) | {
        "Operator", "Data Technology", "Technology Band", "Collection", "Campaign", "Type",
    }]
    df = df[keep].copy()
    if "Operator" in df.columns:
        df["Operator"] = df["Operator"].astype("string").str.strip()
    df["Time"] = _parse_active_time(df["Time"])
    for col in ("RSRP", "RSRQ", "SINR", "Longitude", "Latitude", "DL EARFCN"):
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["Cell Id"] = pd.to_numeric(df["Cell Id"], errors="coerce").astype("Int64")
    df["DL EARFCN"] = df["DL EARFCN"].astype("Int64")
    df["Source"] = path.name
    df["Band"] = df["DL EARFCN"].map(earfcn_to_band)
    df["Layer"] = df["DL EARFCN"].map(earfcn_to_layer)
    print(f"  {len(df):,} rows")
    return df


def merge_active() -> pd.DataFrame:
    if not ACTIVE_ARCHIVE.exists():
        sys.exit(f"Missing active log: {ACTIVE_ARCHIVE.name}")
    print(f"Extracting {ACTIVE_ARCHIVE.name} -> {ACTIVE_DIR}")
    extract_7z(ACTIVE_ARCHIVE, EXTRACT_DIR)
    files = active_csv_files()
    if not files:
        sys.exit(f"Extraction finished but no CSV files were found in {ACTIVE_DIR}")
    print(f"  {len(files)} active CSV files")
    frames = [load_active_csv(path) for path in files]
    merged = pd.concat(frames, ignore_index=True)
    merged = merged.drop_duplicates(subset=["Time", "Operator", "Cell Id", "DL EARFCN", "RSRP"], keep="first")
    sort_cols = ["Time", "Source"]
    if "Operator" in merged.columns:
        sort_cols = ["Time", "Operator", "Source"]
    merged = merged.sort_values(sort_cols, kind="mergesort").reset_index(drop=True)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    merged.to_csv(ACTIVE_MERGED, index=False, compression="gzip")
    print(f"Wrote merged active file: {ACTIVE_MERGED} ({len(merged):,} rows)")
    return merged


## 11. Summary text, small plots, and the report entry

Write analysis_summary.md and the small KPI plots, then build the Excel workbook.
build_report is defined in a later cell. Call main() only after every cell above the last one has been run.


In [ ]:
# ------------------------------------------------------------------------
# 11. Summary text, small plots, and the report entry
# Write analysis_summary.md and the small KPI plots, then build the Excel workbook.
# build_report is defined in a later cell. Call main() only after every cell above the last one has been run.
# ------------------------------------------------------------------------

def format_stats(series: pd.Series) -> str:
    s = series.dropna()
    if s.empty:
        return "n=0"
    return (
        f"n={len(s):,}  min={s.min():.1f}  p5={s.quantile(0.05):.1f}  "
        f"mean={s.mean():.1f}  median={s.median():.1f}  "
        f"p95={s.quantile(0.95):.1f}  max={s.max():.1f}"
    )


def write_summary(df: pd.DataFrame) -> None:
    lines = [
        "# Bogura Drive-Test Analysis Summary",
        "",
        "## What the source files are",
        "",
        "- `Borga_Idle_V2.0.7z` is the IDLE Mode scanner export (folder `Borga_Idle_V2.0/` with `1.csv` … `15.csv`).",
        "- Columns include Time, Longitude, Latitude, Cell Id, DL EARFCN, and scanner KPIs `NB RSRP` / `NB RSRQ` / `NB RS SINR` (renamed to RSRP / RSRQ / SINR).",
        "- Concatenating the 15 CSVs produces the **final log**: `output/Bogura_merged.csv.gz`.",
        "",
        "## Row counts",
        "",
        f"- CSV files: {df['Source'].nunique()}",
        f"- Merged scanner detections: {len(df):,} samples",
        "",
        "## Time and area",
        "",
    ]
    for source, part in df.groupby("Source", sort=False):
        tmin, tmax = part["Time"].min(), part["Time"].max()
        lines.append(
            f"- {source}: {tmin} → {tmax}; "
            f"lon {part['Longitude'].min():.5f}–{part['Longitude'].max():.5f}, "
            f"lat {part['Latitude'].min():.5f}–{part['Latitude'].max():.5f}"
        )
    lines += [
        "",
        "## Radio KPIs (merged)",
        "",
        f"- RSRP: {format_stats(df['RSRP'])}",
        f"- RSRQ: {format_stats(df['RSRQ'])}",
        f"- SINR: {format_stats(df['SINR'])} (blank SINR samples are skipped)",
        "",
        "## RSRP ranges (dBm)",
        "",
    ]
    rsrp_counts = rsrp_range_counts(df["RSRP"])
    for label, n in rsrp_counts.items():
        lines.append(f"- {label}: {int(n):,} ({n / len(df) * 100:.1f}%)")

    lines += ["", "## Bands (from DL EARFCN)", ""]
    band_counts = df["Band"].value_counts()
    for band, n in band_counts.items():
        lines.append(f"- {band}: {int(n):,} ({n / len(df) * 100:.1f}%)")

    lines += [
        "",
        f"- Unique Cell Ids: {df['Cell Id'].nunique():,}",
        "",
        "## Notes",
        "",
        "- Campaign is **Scanner** (NB RSRP / NB RSRQ / NB RS SINR). Coverage maps use the strongest RSRP per scanner sweep (~4 s / 40 m) so every band measured at a location competes for best server.",
        "- Files 1.csv–15.csv cover 11–21 May 2026. Operator is Robi. Layers: L900 (B8), L1800 (B3), L2100 (B1), L2600 (B41).",
        "",
        "Plots are written to `output/plots/`.",
        "",
    ]
    SUMMARY_MD.write_text("\n".join(lines), encoding="utf-8")
    print(f"Wrote {SUMMARY_MD}")


def downsample(df: pd.DataFrame, max_points: int = MAP_MAX_POINTS) -> pd.DataFrame:
    if len(df) <= max_points:
        return df
    step = max(1, len(df) // max_points)
    return df.iloc[::step]


def style_axes(ax, title: str, xlabel: str, ylabel: str) -> None:
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)


def plot_stacked_map_hist(ax, values, classifier, labels, colors, bins, title, xlabel, legend_title, xlim) -> None:
    v = pd.to_numeric(values, errors="coerce")
    cls = classifier(v)
    bottom = np.zeros(len(bins) - 1)
    widths = np.diff(bins)
    for lab, color in zip(labels, colors):
        counts, _ = np.histogram(v[cls == lab].dropna(), bins=bins)
        ax.bar(
            bins[:-1],
            counts,
            width=widths,
            align="edge",
            bottom=bottom,
            color=color,
            edgecolor="none",
            label=lab,
        )
        bottom += counts
    ax.legend(title=legend_title, fontsize=6, loc="upper right")
    style_axes(ax, title, xlabel, "Samples")
    ax.set_xlim(*xlim)


def plot_all(df: pd.DataFrame) -> None:
    PLOTS_DIR.mkdir(parents=True, exist_ok=True)
    geo = best_server(df.dropna(subset=["Longitude", "Latitude", "RSRP"]))
    plot_rsrp_coverage_map(geo, PLOTS_DIR / "01_route_rsrp.png")
    plot_rsrq_coverage_map(geo, PLOTS_DIR / "01_route_rsrq.png")
    plot_sinr_coverage_map(geo, PLOTS_DIR / "01_route_sinr.png")
    rsrp_spots = find_bad_spots(geo, "RSRP")
    rsrq_spots = find_bad_spots(geo, "RSRQ")
    sinr_spots = find_bad_spots(geo, "SINR")
    plot_rsrp_bad_spot_map(geo, rsrp_spots, PLOTS_DIR / "08_bad_spots_rsrp.png")
    plot_rsrq_bad_spot_map(geo, rsrq_spots, PLOTS_DIR / "08_bad_spots_rsrq.png")
    plot_sinr_bad_spot_map(geo, sinr_spots, PLOTS_DIR / "08_bad_spots_sinr.png")

    fig, axes = plt.subplots(1, 3, figsize=(16, 5.2))
    plot_stacked_map_hist(
        axes[0], df["RSRP"], rsrp_map_class, MAP_RSRP_LABELS, MAP_RSRP_COLORS,
        np.arange(-140, -40, 2), "RSRP distribution", "RSRP (dBm)", "RSRP (dBm)", (-140, -50),
    )
    plot_stacked_map_hist(
        axes[1], df["RSRQ"], rsrq_map_class, MAP_RSRQ_LABELS, MAP_RSRQ_COLORS,
        np.arange(-30, 0, 0.5), "RSRQ distribution", "RSRQ (dB)", "RSRQ (dB)", (-24, -3),
    )
    plot_stacked_map_hist(
        axes[2], df["SINR"], sinr_map_class, MAP_SINR_LABELS, MAP_SINR_COLORS,
        np.arange(-20, 32, 1), "SINR distribution", "SINR (dB)", "SINR (dB)", (-15, 30),
    )
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / "02_kpi_histograms.png", dpi=140)
    plt.close(fig)

    fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
    ts = downsample(df.dropna(subset=["Time"]), max_points=40000)
    for ax, col in zip(axes, ("RSRP", "RSRQ", "SINR")):
        ax.scatter(ts["Time"], ts[col], s=2, alpha=0.25, color="#1d3557", linewidths=0)
        style_axes(ax, f"{col} vs time", "Time", col)
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / "03_kpi_vs_time.png", dpi=140)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(9, 5.5))
    counts = rsrp_range_counts(df["RSRP"])
    ax.barh(list(counts.index[::-1]), counts.values[::-1], color=list(reversed(RSRP_COLORS)))
    style_axes(ax, "RSRP ranges (dBm)", "Samples", "")
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / "04_rsrp_quality_bins.png", dpi=140)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(8, 5))
    band_counts = df["Band"].value_counts()
    ax.bar(band_counts.index.astype(str), band_counts.values, color="#457b9d")
    ax.tick_params(axis="x", rotation=20)
    style_axes(ax, "Samples by LTE band (DL EARFCN)", "Band", "Samples")
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / "05_band_counts.png", dpi=140)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(10, 6))
    top_cells = df["Cell Id"].dropna().astype("int64").value_counts().head(15)
    ax.barh(top_cells.index.astype(str)[::-1], top_cells.values[::-1], color="#1d3557")
    style_axes(ax, "Top 15 scanner Cell Ids", "Samples", "Cell Id")
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / "06_top_cells.png", dpi=140)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(10, 5))
    source_counts = df["Source"].value_counts()
    order = sorted(source_counts.index, key=lambda n: int(str(n).replace(".csv", "")) if str(n).replace(".csv", "").isdigit() else str(n))
    source_counts = source_counts.reindex(order)
    ax.bar(source_counts.index.astype(str), source_counts.values, color="#2a6f97")
    ax.tick_params(axis="x", rotation=45)
    style_axes(ax, "Merged idle log: samples by CSV", "Source file", "Samples")
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / "07_csv_counts.png", dpi=140)
    plt.close(fig)

    print(f"Wrote plots to {PLOTS_DIR}")


def _finalize_merged(df: pd.DataFrame) -> pd.DataFrame:
    df["Time"] = pd.to_datetime(df["Time"], errors="coerce")
    df["Cell Id"] = pd.to_numeric(df["Cell Id"], errors="coerce").astype("Int64")
    df["DL EARFCN"] = pd.to_numeric(df["DL EARFCN"], errors="coerce").astype("Int64")
    df["Band"] = df["DL EARFCN"].map(earfcn_to_band)
    df["Layer"] = df["DL EARFCN"].map(earfcn_to_layer)
    return df


def load_merged_csv() -> pd.DataFrame:
    print(f"Loading existing merged file {MERGED_CSV}")
    return _finalize_merged(pd.read_csv(MERGED_CSV))


def load_active_merged() -> pd.DataFrame:
    print(f"Loading existing merged file {ACTIVE_MERGED}")
    return _finalize_merged(pd.read_csv(ACTIVE_MERGED))


def main() -> None:
    if "--from-merged" in sys.argv:
        if not MERGED_CSV.exists():
            sys.exit(f"{MERGED_CSV} not found; run without --from-merged first")
        merged = load_merged_csv()
        if not ACTIVE_MERGED.exists():
            sys.exit(f"{ACTIVE_MERGED} not found; run without --from-merged first")
        active = load_active_merged()
    else:
        extract_archives()
        merged = merge_parts()
        active = merge_active()
    write_summary(merged)
    plot_all(merged)

    report_path = build_report(merged, active)
    result = verify_report(report_path)
    if result["problems"]:
        raise SystemExit("Excel view checks failed: " + "; ".join(result["problems"]))
    print("Excel report verified: no freeze panes, no gridlines.")
    print("Done.")


## 12. Workbook version and sheet colours

Excel v1.63. The version in the filename changes only when the workbook itself changes.
Every sheet has freeze panes off and gridlines off.


In [ ]:
# ------------------------------------------------------------------------
# 12. Workbook version and sheet colours
# Excel v1.63. The version in the filename changes only when the workbook itself changes.
# Every sheet has freeze panes off and gridlines off.
# ------------------------------------------------------------------------

# Raise this on every report write. The versioned filename is the name that
# gets downloaded, so a new Excel or web report must not reuse the previous number.
REPORT_VERSION = "1.63"
SVG_DIR = PLOTS_DIR / "svg"
SVG_FALLBACK_SCALE = 3.0
SVG_BLIP_EXT_URI = "{96DAC541-7B7A-43D3-8B79-37D633B846F1}"
SVG_BLIP_NS = "http://schemas.microsoft.com/office/drawing/2016/SVG/main"
IMAGE_REL_TYPE = "http://schemas.openxmlformats.org/officeDocument/2006/relationships/image"
# PNG fallback -> SVG it was rendered from; these pictures get an SVG layer in the workbook.
SVG_LAYERS: dict[Path, Path] = {}
# Every poor run is a bad spot, so a dense KPI produces over a thousand of them.
# The map marks them all; the table lists the longest ones a team can act on.
BAD_SPOT_TABLE_MAX = 300
REPORT_XLSX = OUTPUT_DIR / "Bogura_DriveTest_Report.xlsx"
VERSIONED_XLSX = OUTPUT_DIR / f"Bogura_DriveTest_Report_v{REPORT_VERSION}.xlsx"

NAVY = "1B4F72"
WHITE = "FFFFFF"
LIGHT = "F4F6F7"
CARD = "EAF2F8"
YELLOW = "F4D03F"
RSRP_COLOR = "C0392B"
RSRQ_COLOR = "1F618D"
SINR_COLOR = "117A65"
VS_SINR_LINE = "48C9B0"
VS_RSRQ_LINE = "5DADE2"
THIN = Border(
    left=Side(style="thin", color="D5D8DC"),
    right=Side(style="thin", color="D5D8DC"),
    top=Side(style="thin", color="D5D8DC"),
    bottom=Side(style="thin", color="D5D8DC"),
)


## 13. Cells, pictures, and the movable circles

Helpers that write cells and pictures.
The orange circles are drawing shapes stored in the xlsx zip. A normal openpyxl save removes them, so later steps edit the zip instead of saving the workbook.


In [ ]:
# ------------------------------------------------------------------------
# 13. Cells, pictures, and the movable circles
# Helpers that write cells and pictures.
# The orange circles are drawing shapes stored in the xlsx zip. A normal openpyxl save removes them, so later steps edit the zip instead of saving the workbook.
# ------------------------------------------------------------------------

def apply_sheet_view(ws: Worksheet) -> None:
    """No freeze panes, no gridlines (on-screen or printed)."""
    ws.freeze_panes = None
    ws.sheet_view.showGridLines = False
    ws.print_options.gridLines = False
    ws.page_setup.orientation = "landscape"
    ws.page_setup.fitToPage = True
    ws.page_setup.fitToWidth = 1
    ws.page_setup.fitToHeight = 0
    ws.sheet_properties.pageSetUpPr.fitToPage = True
    ws.page_setup.paperSize = ws.PAPERSIZE_A4
    ws.sheet_view.zoomScale = 100
    ws.sheet_view.view = "normal"


def fill(color: str) -> PatternFill:
    return PatternFill("solid", fgColor=color)


def font(size=11, bold=False, color=NAVY, name="Calibri") -> Font:
    return Font(name=name, size=size, bold=bold, color=color)


def write_cell(ws, row, col, value, *, size=11, bold=False, color=NAVY, fill_color=None, align="left", num_fmt=None, wrap=False):
    cell = ws.cell(row, col, value)
    cell.font = font(size=size, bold=bold, color=color)
    cell.alignment = Alignment(horizontal=align, vertical="center", wrap_text=wrap)
    if fill_color:
        cell.fill = fill(fill_color)
    if num_fmt:
        cell.number_format = num_fmt
    return cell


def set_widths(ws, widths: dict[str, float]) -> None:
    for col, width in widths.items():
        ws.column_dimensions[col].width = width


def kpi_stats(series: pd.Series) -> dict[str, float]:
    s = pd.to_numeric(series, errors="coerce").dropna()
    if s.empty:
        return {k: np.nan for k in ("Count", "Min", "P5", "Median", "Mean", "P95", "Max", "Std")}
    return {
        "Count": float(len(s)),
        "Min": float(s.min()),
        "P5": float(s.quantile(0.05)),
        "Median": float(s.median()),
        "Mean": float(s.mean()),
        "P95": float(s.quantile(0.95)),
        "Max": float(s.max()),
        "Std": float(s.std()),
    }


def map_range_counts(series: pd.Series, classifier, labels) -> pd.Series:
    return classifier(series).value_counts(dropna=True).reindex(labels, fill_value=0)


def add_image(ws: Worksheet, path: Path, anchor: str, width: int, height: int) -> tuple[int, int]:
    """Place the picture inside a width x height box without stretching it; returns its size."""
    img = XLImage(str(path))
    fit = min(width / img.width, height / img.height)
    img.width = round(img.width * fit)
    img.height = round(img.height * fit)
    ws.add_image(img, anchor)
    return img.width, img.height


def svg_with_fallback(svg_path: Path | None, png_path: Path) -> Path | None:
    """Render the PNG that viewers without SVG support show, and pair it with its SVG."""
    if svg_path is None:
        return None
    cairosvg.svg2png(
        url=str(svg_path), write_to=str(png_path), scale=SVG_FALLBACK_SCALE, background_color="white"
    )
    SVG_LAYERS[png_path.resolve()] = svg_path
    return png_path


def embed_svg_layers(path: Path) -> int:
    """Attach each paired SVG to its picture as an Office svgBlip.

    Excel 365 then draws the vector map; older Excel and other viewers keep the PNG.
    Returns the number of pictures that received an SVG layer.
    """
    by_digest = {
        hashlib.sha1(png.read_bytes()).hexdigest(): svg for png, svg in SVG_LAYERS.items() if png.exists()
    }
    with zipfile.ZipFile(path) as zf:
        names = zf.namelist()
        parts = {name: zf.read(name) for name in names}
    svg_for_media: dict[str, str] = {}
    for name in names:
        if name.startswith("xl/media/") and name.lower().endswith(".png"):
            svg = by_digest.get(hashlib.sha1(parts[name]).hexdigest())
            if svg is not None:
                svg_name = name[:-4] + ".svg"
                parts[svg_name] = svg.read_bytes()
                svg_for_media[name] = svg_name
    embedded = 0
    for rels_name in [n for n in names if re.fullmatch(r"xl/drawings/_rels/drawing\d+\.xml\.rels", n)]:
        drawing_name = "xl/drawings/" + rels_name.rsplit("/", 1)[1][: -len(".rels")]
        rels = parts[rels_name].decode("utf-8")
        drawing = parts[drawing_name].decode("utf-8")
        next_id = max((int(i) for i in re.findall(r'Id="rId(\d+)"', rels)), default=0) + 1
        added = []
        for rel in re.findall(r"<Relationship\b[^>]*/>", rels):
            rid = re.search(r'Id="([^"]+)"', rel).group(1)
            target = re.search(r'Target="([^"]+)"', rel).group(1)
            media = target.lstrip("/") if target.startswith("/") else "xl/media/" + target.rsplit("/", 1)[1]
            svg_name = svg_for_media.get(media)
            if svg_name is None:
                continue
            svg_rid = f"rId{next_id}"
            next_id += 1
            svg_target = "/" + svg_name if target.startswith("/") else "../media/" + svg_name.rsplit("/", 1)[1]
            added.append(f'<Relationship Type="{IMAGE_REL_TYPE}" Target="{svg_target}" Id="{svg_rid}" />')
            blip = re.compile(r'<a:blip\b([^>]*\br:embed="' + re.escape(rid) + r'"[^>]*?)\s*/>')
            drawing, hits = blip.subn(
                lambda m: (
                    f"<a:blip{m.group(1)}><a:extLst><a:ext uri=\"{SVG_BLIP_EXT_URI}\">"
                    f"<asvg:svgBlip xmlns:asvg=\"{SVG_BLIP_NS}\" r:embed=\"{svg_rid}\" />"
                    "</a:ext></a:extLst></a:blip>"
                ),
                drawing,
            )
            embedded += hits
        if added:
            parts[rels_name] = rels.replace("</Relationships>", "".join(added) + "</Relationships>").encode("utf-8")
            parts[drawing_name] = drawing.encode("utf-8")
    types = parts["[Content_Types].xml"].decode("utf-8")
    if svg_for_media and 'Extension="svg"' not in types:
        types = types.replace("<Override ", '<Default Extension="svg" ContentType="image/svg+xml" /><Override ', 1)
        parts["[Content_Types].xml"] = types.encode("utf-8")
    order = names + [n for n in parts if n not in names]
    tmp = path.with_suffix(".tmp.xlsx")
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as zf:
        for name in order:
            zf.writestr(name, parts[name])
    tmp.replace(path)
    return embedded


def _sheet_drawings(parts: dict[str, bytes]) -> dict[str, str]:
    """Map a worksheet name to its drawing part, when the sheet has one."""
    main = "{http://schemas.openxmlformats.org/spreadsheetml/2006/main}"
    rel_ns = "{http://schemas.openxmlformats.org/officeDocument/2006/relationships}"
    wb = ET.fromstring(parts["xl/workbook.xml"])
    book_rels = ET.fromstring(parts["xl/_rels/workbook.xml.rels"])
    rid_target = {rel.attrib["Id"]: rel.attrib["Target"].lstrip("/") for rel in book_rels}
    found = {}
    for sheet in wb.find(f"{main}sheets"):
        sheet_path = rid_target[sheet.attrib[f"{rel_ns}id"]]
        rels_path = sheet_path.replace("worksheets/", "worksheets/_rels/") + ".rels"
        if rels_path not in parts:
            continue
        for rel in ET.fromstring(parts[rels_path]):
            target = rel.attrib["Target"].lstrip("/")
            if "drawings/drawing" in target:
                found[sheet.attrib["name"]] = target
    return found


def embed_circle_shapes(path: Path) -> int:
    """Add one Excel oval per bad-spot circle, anchored on top of its map picture.

    Each oval is its own drawing object, so it can be dragged without moving the
    map or the other circles. Ovals are added only on Bad Spot sheets.
    """
    if not MAP_CIRCLE_PIXELS:
        return 0
    by_digest = {}
    for png, geom in MAP_CIRCLE_PIXELS.items():
        file = Path(png)
        if file.exists() and geom.get("circles"):
            by_digest[hashlib.sha1(file.read_bytes()).hexdigest()] = geom
    with zipfile.ZipFile(path) as zf:
        names = zf.namelist()
        parts = {name: zf.read(name) for name in names}
    media_digest = {
        name: hashlib.sha1(parts[name]).hexdigest()
        for name in names
        if name.startswith("xl/media/") and name.lower().endswith(".png")
    }
    placed = 0
    bad_spot_drawings = {
        drawing for sheet, drawing in _sheet_drawings(parts).items() if sheet.startswith("Bad Spot")
    }
    for rels_name in [n for n in names if re.fullmatch(r"xl/drawings/_rels/drawing\d+\.xml\.rels", n)]:
        drawing_name = "xl/drawings/" + rels_name.rsplit("/", 1)[1][: -len(".rels")]
        if drawing_name not in bad_spot_drawings:
            continue
        rels = parts[rels_name].decode("utf-8")
        drawing = parts[drawing_name].decode("utf-8")
        rid_media = {}
        for rel in re.findall(r"<Relationship\b[^>]*/>", rels):
            rid = re.search(r'Id="([^"]+)"', rel).group(1)
            target = re.search(r'Target="([^"]+)"', rel).group(1)
            media = target.lstrip("/") if target.startswith("/") else "xl/media/" + target.rsplit("/", 1)[1]
            rid_media[rid] = media
        next_shape = max((int(i) for i in re.findall(r'<cNvPr id="(\d+)"', drawing)), default=0) + 1
        extras = []
        for block in re.findall(r"<oneCellAnchor>.*?</oneCellAnchor>", drawing, flags=re.S):
            embed = re.search(r'r:embed="([^"]+)"', block)
            if embed is None:
                continue
            digest = media_digest.get(rid_media.get(embed.group(1), ""))
            geom = by_digest.get(digest or "")
            if geom is None:
                continue
            img_w, img_h = geom["size"]
            origin = re.search(
                r"<from><col>(\d+)</col><colOff>(\d+)</colOff><row>(\d+)</row><rowOff>(\d+)</rowOff></from><ext cx=\"(\d+)\" cy=\"(\d+)\"",
                block,
            )
            if origin is None or img_w <= 0 or img_h <= 0:
                continue
            col, col_off, row, row_off, pic_cx, pic_cy = (int(v) for v in origin.groups())
            for circle in geom["circles"]:
                extras.append(_circle_anchor(circle, next_shape, col, col_off, row, row_off, pic_cx, pic_cy, img_w, img_h))
                next_shape += 1
                placed += 1
        if extras:
            drawing = drawing.replace("</wsDr>", "".join(extras) + "</wsDr>")
            parts[drawing_name] = drawing.encode("utf-8")
    if placed:
        tmp = path.with_suffix(".tmp.xlsx")
        with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as zf:
            for name in names:
                zf.writestr(name, parts[name])
        tmp.replace(path)
    spread_circle_anchors(path)
    return placed


def _circle_anchor(circle, shape_id, col, col_off, row, row_off, pic_cx, pic_cy, img_w, img_h) -> str:
    """One movable oval, positioned as a fraction of the picture it sits on."""
    left = max(0.0, min(float(circle["x"]), img_w - 1))
    top = max(0.0, min(float(circle["y"]), img_h - 1))
    width = max(1.0, min(float(circle["w"]), img_w - left))
    height = max(1.0, min(float(circle["h"]), img_h - top))
    dash = "sysDot" if circle.get("dotted") else "dash"
    cx = max(1, int(width / img_w * pic_cx))
    cy = max(1, int(height / img_h * pic_cy))
    return (
        "<oneCellAnchor>"
        f"<from><col>{col}</col><colOff>{col_off + int(left / img_w * pic_cx)}</colOff>"
        f"<row>{row}</row><rowOff>{row_off + int(top / img_h * pic_cy)}</rowOff></from>"
        f"<ext cx=\"{cx}\" cy=\"{cy}\"/>"
        "<sp><nvSpPr>"
        f"<cNvPr id=\"{shape_id}\" name=\"{circle['name']}\"/><cNvSpPr/></nvSpPr>"
        "<spPr><a:xfrm><a:off x=\"0\" y=\"0\"/>"
        f"<a:ext cx=\"{cx}\" cy=\"{cy}\"/>"
        "</a:xfrm><a:prstGeom prst=\"ellipse\"><a:avLst/></a:prstGeom><a:noFill/>"
        "<a:ln w=\"28575\"><a:solidFill><a:srgbClr val=\"FF6600\"/></a:solidFill>"
        f"<a:prstDash val=\"{dash}\"/></a:ln></spPr>"
        "<txBody><a:bodyPr/><a:lstStyle/><a:p><a:endParaRPr lang=\"en-US\"/></a:p></txBody>"
        "</sp><clientData/></oneCellAnchor>"
    )


def _column_width_px(width: float) -> int:
    """Excel's display width for a stored column width, in pixels at 96 dpi."""
    if width < 1:
        return int(width * 12 + 0.5)
    return int(width * 7 + 0.5) + 5


def _sheet_metric(sheet_xml: bytes):
    """Column widths and row heights for one worksheet, in EMUs."""
    ns = "{http://schemas.openxmlformats.org/spreadsheetml/2006/main}"
    root = ET.fromstring(sheet_xml)
    fmt = root.find(f"{ns}sheetFormatPr")
    default_pt = float(fmt.attrib.get("defaultRowHeight", "15")) if fmt is not None else 15.0
    default_width = 8.43
    if fmt is not None and "defaultColWidth" in fmt.attrib:
        default_width = float(fmt.attrib["defaultColWidth"])
    col_px: dict[int, int] = {}
    cols = root.find(f"{ns}cols")
    if cols is not None:
        for col in cols:
            px = _column_width_px(float(col.attrib.get("width", default_width)))
            for index in range(int(col.attrib["min"]), int(col.attrib["max"]) + 1):
                col_px[index] = px
    row_pt: dict[int, float] = {}
    for row in root.findall(f"{ns}sheetData/{ns}row"):
        if "ht" in row.attrib:
            row_pt[int(row.attrib["r"])] = float(row.attrib["ht"])
    return col_px, default_width, row_pt, default_pt


def _spread_drawing_anchors(drawing: str, sheet_xml: bytes) -> str:
    """Move each circle's anchor into the cell under its bad spot.

    A from-offset larger than the cell is ignored by Excel, which stacks every
    circle on the picture's first cell. The offset is split across the columns
    and rows the picture covers, and each circle keeps its own cell.
    """
    col_px, default_width, row_pt, default_pt = _sheet_metric(sheet_xml)
    px_emu = 9525
    pt_emu = 12700

    def col_emu(index0: int) -> int:
        return col_px.get(index0 + 1, _column_width_px(default_width)) * px_emu

    def row_emu(index0: int) -> int:
        return int(round(row_pt.get(index0 + 1, default_pt) * pt_emu))

    def prefix(count: int, size_of) -> list[int]:
        total = [0]
        for i in range(count):
            total.append(total[-1] + size_of(i))
        return total

    col_at = prefix(50, col_emu)
    row_at = prefix(8000, row_emu)

    def locate(at: list[int], absolute: int) -> tuple[int, int]:
        absolute = max(0, absolute)
        lo, hi = 0, len(at) - 2
        while lo < hi:
            mid = (lo + hi + 1) // 2
            if at[mid] <= absolute:
                lo = mid
            else:
                hi = mid - 1
        offset = absolute - at[lo]
        room = at[lo + 1] - at[lo]
        if offset >= room:
            offset = max(0, room - 1)
        return lo, int(offset)

    def place(match: re.Match) -> str:
        block = match.group(0)
        if 'prst="ellipse"' not in block:
            return block
        found = re.search(
            r"<from><col>(\d+)</col><colOff>(\d+)</colOff><row>(\d+)</row><rowOff>(\d+)</rowOff></from>",
            block,
        )
        if found is None:
            return block
        col, col_off, row, row_off = (int(v) for v in found.groups())
        if col >= len(col_at) - 1 or row >= len(row_at) - 1:
            return block
        new_col, new_col_off = locate(col_at, col_at[col] + col_off)
        new_row, new_row_off = locate(row_at, row_at[row] + row_off)
        updated = (
            f"<from><col>{new_col}</col><colOff>{new_col_off}</colOff>"
            f"<row>{new_row}</row><rowOff>{new_row_off}</rowOff></from>"
        )
        return block.replace(found.group(0), updated, 1)

    return re.sub(r"<oneCellAnchor>.*?</oneCellAnchor>", place, drawing, flags=re.S)


def _worksheet_xml(parts: dict[str, bytes]) -> dict[str, bytes]:
    """Map a worksheet name to its worksheet XML."""
    main = "{http://schemas.openxmlformats.org/spreadsheetml/2006/main}"
    rel_ns = "{http://schemas.openxmlformats.org/officeDocument/2006/relationships}"
    wb = ET.fromstring(parts["xl/workbook.xml"])
    book_rels = ET.fromstring(parts["xl/_rels/workbook.xml.rels"])
    rid_target = {rel.attrib["Id"]: rel.attrib["Target"].lstrip("/") for rel in book_rels}
    found = {}
    for node in wb.find(f"{main}sheets"):
        found[node.attrib["name"]] = parts[rid_target[node.attrib[f"{rel_ns}id"]]]
    return found


def _emu_prefix(sheet_xml: bytes):
    col_px, default_width, row_pt, default_pt = _sheet_metric(sheet_xml)

    def col_emu(index0: int) -> int:
        return col_px.get(index0 + 1, _column_width_px(default_width)) * 9525

    def row_emu(index0: int) -> int:
        return int(round(row_pt.get(index0 + 1, default_pt) * 12700))

    def prefix(count: int, size_of) -> list[int]:
        total = [0]
        for i in range(count):
            total.append(total[-1] + size_of(i))
        return total

    return prefix(60, col_emu), prefix(8000, row_emu)


def _drop_ellipses_on_picture(drawing: str, sheet_xml: bytes, picture_block: str, keep_last: int = 0) -> str:
    """Remove circle shapes that sit on this picture, optionally keeping the newest ones."""
    origin = re.search(
        r"<from><col>(\d+)</col><colOff>(\d+)</colOff><row>(\d+)</row><rowOff>(\d+)</rowOff></from><ext cx=\"(\d+)\" cy=\"(\d+)\"",
        picture_block,
    )
    if origin is None:
        return drawing
    col_at, row_at = _emu_prefix(sheet_xml)
    pcol, pcoff, prow, proff, pcx, pcy = (int(v) for v in origin.groups())
    if pcol >= len(col_at) - 1 or prow >= len(row_at) - 1:
        return drawing
    left, top = col_at[pcol] + pcoff, row_at[prow] + proff
    right, bottom = left + pcx, top + pcy
    blocks = list(re.finditer(r"<oneCellAnchor>.*?</oneCellAnchor>", drawing, flags=re.S))
    on_picture = []
    for match in blocks:
        block = match.group(0)
        if 'prst="ellipse"' not in block:
            continue
        found = re.search(
            r"<from><col>(\d+)</col><colOff>(\d+)</colOff><row>(\d+)</row><rowOff>(\d+)</rowOff></from><ext cx=\"(\d+)\" cy=\"(\d+)\"",
            block,
        )
        if found is None:
            continue
        col, coff, row, roff, cx, cy = (int(v) for v in found.groups())
        if col >= len(col_at) - 1 or row >= len(row_at) - 1:
            continue
        cx0 = col_at[col] + coff + cx / 2.0
        cy0 = row_at[row] + roff + cy / 2.0
        if left - 20000 <= cx0 <= right + 20000 and top - 20000 <= cy0 <= bottom + 20000:
            on_picture.append(match)
    drop = set(id(m) for m in on_picture[: max(0, len(on_picture) - keep_last)])
    if not drop:
        return drawing
    pieces = []
    cursor = 0
    for match in blocks:
        if id(match) in drop:
            pieces.append(drawing[cursor:match.start()])
            cursor = match.end()
    pieces.append(drawing[cursor:])
    return "".join(pieces)


def spread_circle_anchors(path: Path) -> int:
    """Re-anchor every bad-spot circle so Excel draws it on its own bad spot."""
    with zipfile.ZipFile(path) as zf:
        names = zf.namelist()
        parts = {name: zf.read(name) for name in names}
    moved = 0
    drawings = _sheet_drawings(parts)
    main = "{http://schemas.openxmlformats.org/spreadsheetml/2006/main}"
    rel_ns = "{http://schemas.openxmlformats.org/officeDocument/2006/relationships}"
    wb = ET.fromstring(parts["xl/workbook.xml"])
    book_rels = ET.fromstring(parts["xl/_rels/workbook.xml.rels"])
    rid_target = {rel.attrib["Id"]: rel.attrib["Target"].lstrip("/") for rel in book_rels}
    sheet_part = {}
    for node in wb.find(f"{main}sheets"):
        sheet_part[node.attrib["name"]] = rid_target[node.attrib[f"{rel_ns}id"]]
    changed = False
    for sheet, drawing_name in drawings.items():
        if not sheet.startswith("Bad Spot"):
            continue
        original = parts[drawing_name].decode("utf-8")
        updated = _spread_drawing_anchors(original, parts[sheet_part[sheet]])
        if updated != original:
            parts[drawing_name] = updated.encode("utf-8")
            moved += original.count('prst="ellipse"')
            changed = True
    if changed:
        tmp = path.with_suffix(".tmp.xlsx")
        with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as zf:
            for name in names:
                zf.writestr(name, parts[name])
        tmp.replace(path)
    return moved


_CIRCLE_NOTE_OLD = (
    "Each circle is a separate orange shape on the satellite map, named "
    "Circle 1, Circle 2, and so on, in the same order as the table, so a circle can be dragged "
    "without moving the others."
)
_CIRCLE_NOTE_NEW = (
    "Each circle is a separate orange shape on the combined map, named "
    "Circle 1, Circle 2, and so on, in the same order as the table. Each zoom panel has that "
    "same circle as its own shape, not painted into the picture, so you can drag it. The RSRP, "
    "RSRQ and SINR sheets do not repeat these circles."
)
_PIE_NOTE_OLD = "The RSRP, RSRQ and SINR sheets do not repeat these circles."
_PIE_NOTE_NEW = (
    "The RSRP, RSRQ and SINR sheets do not repeat these circles. "
    "Each zoom panel draws a sector pie for every nearby site from System DUMP.xlsb. "
    "The label is the sector name. Where a sector shoots toward that bad spot, a yellow line shows the distance."
)


def refresh_zoom_circles(src: Path | None = None) -> Path:
    """Update an existing report without redrawing the sheets that stay the same.

    Removes the circle shapes from the RSRP, RSRQ and SINR sheets, redraws only
    the zoom pictures without a painted circle, and adds one movable circle on
    each zoom panel.
    """
    import io
    from PIL import Image


    src = src or REPORT_XLSX
    print("Loading drive logs", flush=True)
    idle = load_merged_csv() if MERGED_CSV.exists() else None
    active = load_active_merged() if ACTIVE_MERGED.exists() else None
    if idle is None or active is None:
        raise SystemExit("Merged drive logs are missing; a full report build is required.")
    active = select_active_report(active)
    print(f"Idle rows {len(idle):,}  Active {ACTIVE_OPERATOR} {ACTIVE_CELL_TYPE} rows {len(active):,}", flush=True)
    idle_track = best_server(idle)
    active_track = best_carrier(active)
    MAP_CIRCLE_PIXELS.clear()
    zoom_for_sheet = {}
    for tag, frame in (("", idle_track), ("active", active_track)):
        suffix = f"_{tag}" if tag else ""
        zoom_bit = "Active" if tag else "first all-band map"
        for kpi, classify, labels, colors in (
            ("RSRP", rsrp_map_class, MAP_RSRP_LABELS, MAP_RSRP_COLORS),
            ("RSRQ", rsrq_map_class, MAP_RSRQ_LABELS, MAP_RSRQ_COLORS),
            ("SINR", sinr_map_class, MAP_SINR_LABELS, MAP_SINR_COLORS),
        ):
            print(f"Drawing {kpi} zooms ({tag or 'idle'})", flush=True)
            spots = find_bad_spots(frame, kpi)
            path = PLOTS_DIR / f"excel_bad_spots_{kpi.lower()}_zooms{suffix}.png"
            plot_bad_spot_zoom_grid(
                frame, spots, path, kpi, classify(frame[kpi]), labels, colors,
                f"{kpi} zoomed locations ({zoom_bit})", shapes=True,
            )
            sheet = f"Bad Spot {kpi}" + ("_Active" if tag else "")
            zoom_for_sheet[sheet] = path
            recorded = len(MAP_CIRCLE_PIXELS[str(path.resolve())]["circles"])
            print(f"  {sheet}: {recorded} movable circles", flush=True)

    print("Updating the workbook", flush=True)
    with zipfile.ZipFile(src) as zf:
        names = zf.namelist()
        parts = {name: zf.read(name) for name in names}
    drawings = _sheet_drawings(parts)
    sheet_xml = _worksheet_xml(parts)
    for sheet, drawing_name in drawings.items():
        if sheet.startswith("Bad Spot"):
            continue
        drawing = parts[drawing_name].decode("utf-8")
        stripped = re.sub(
            r'<oneCellAnchor>(?:(?!</oneCellAnchor>).)*prst="ellipse".*?</oneCellAnchor>',
            "",
            drawing,
            flags=re.S,
        )
        parts[drawing_name] = stripped.encode("utf-8")

    for sheet, png in zoom_for_sheet.items():
        drawing_name = drawings[sheet]
        rels_name = drawing_name.replace("drawings/", "drawings/_rels/") + ".rels"
        rels = parts[rels_name].decode("utf-8")
        drawing = parts[drawing_name].decode("utf-8")
        rid_media = {}
        for rel in re.findall(r"<Relationship\b[^>]*/>", rels):
            rid = re.search(r'Id="([^"]+)"', rel).group(1)
            target = re.search(r'Target="([^"]+)"', rel).group(1)
            media = target.lstrip("/") if target.startswith("/") else "xl/media/" + target.rsplit("/", 1)[1]
            rid_media[rid] = media
        pictures = []
        for block in re.findall(r"<oneCellAnchor>.*?</oneCellAnchor>", drawing, flags=re.S):
            if "<pic>" not in block:
                continue
            row = int(re.search(r"<row>(\d+)</row>", block).group(1))
            pictures.append((row, block))
        _row, zoom_block = max(pictures)
        embed = re.search(r'r:embed="([^"]+)"', zoom_block).group(1)
        media = rid_media[embed]
        image = Image.open(png).convert("RGB")
        buf = io.BytesIO()
        image.save(buf, "JPEG", quality=74, optimize=True)
        parts[media] = buf.getvalue()
        geom = MAP_CIRCLE_PIXELS[str(Path(png).resolve())]
        img_w, img_h = geom["size"]
        origin = re.search(
            r"<from><col>(\d+)</col><colOff>(\d+)</colOff><row>(\d+)</row><rowOff>(\d+)</rowOff></from><ext cx=\"(\d+)\" cy=\"(\d+)\"",
            zoom_block,
        )
        col, col_off, row, row_off, pic_cx, pic_cy = (int(v) for v in origin.groups())
        drawing = _drop_ellipses_on_picture(drawing, sheet_xml[sheet], zoom_block, keep_last=0)
        next_shape = max(int(i) for i in re.findall(r'<cNvPr id="(\d+)"', drawing)) + 1
        extras = []
        for circle in geom["circles"]:
            extras.append(_circle_anchor(circle, next_shape, col, col_off, row, row_off, pic_cx, pic_cy, img_w, img_h))
            next_shape += 1
        parts[drawing_name] = drawing.replace("</wsDr>", "".join(extras) + "</wsDr>").encode("utf-8")
        print(f"  {sheet}: replaced zoom picture, added {len(extras)} circles", flush=True)

    note_hits = 0
    for name, body in list(parts.items()):
        if not name.endswith(".xml"):
            continue
        text = body.decode("utf-8")
        if _CIRCLE_NOTE_OLD not in text:
            continue
        parts[name] = text.replace(_CIRCLE_NOTE_OLD, _CIRCLE_NOTE_NEW).encode("utf-8")
        note_hits += text.count(_CIRCLE_NOTE_OLD)
    pie_hits = 0
    for name, body in list(parts.items()):
        if not name.endswith(".xml"):
            continue
        text = body.decode("utf-8")
        changed = False
        if _PIE_NOTE_OLD in text and "sector pie" not in text:
            text = text.replace(_PIE_NOTE_OLD, _PIE_NOTE_NEW)
            pie_hits += 1
            changed = True
        if "Physical_Site_Database_V2.0.xlsx" in text or "Physical_Site_Database_V1.0.xlsb" in text:
            text = text.replace("Physical_Site_Database_V2.0.xlsx", "System DUMP.xlsb")
            text = text.replace("Physical_Site_Database_V1.0.xlsb", "System DUMP.xlsb")
            changed = True
        if changed:
            parts[name] = text.encode("utf-8")
    print(f"Updated circle note in {note_hits} place(s) and sector-pie note in {pie_hits}", flush=True)

    order = [n for n in names if n in parts]
    tmp = src.with_suffix(".tmp.xlsx")
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as zf:
        for name in order:
            zf.writestr(name, parts[name])
    tmp.replace(VERSIONED_XLSX)
    if VERSIONED_XLSX.resolve() != REPORT_XLSX.resolve():
        shutil.copy2(VERSIONED_XLSX, REPORT_XLSX)
    spread_circle_anchors(VERSIONED_XLSX)
    if VERSIONED_XLSX.resolve() != REPORT_XLSX.resolve():
        shutil.copy2(VERSIONED_XLSX, REPORT_XLSX)
    print(f"Wrote {VERSIONED_XLSX}", flush=True)
    return VERSIONED_XLSX


def compress_workbook_images(path: Path, quality: int = 74) -> int:
    """Store the satellite maps as JPEG so the workbook stays a normal download.

    Picture size and the circle shapes are unchanged; only the photo bytes shrink.
    """
    from PIL import Image
    import io

    with zipfile.ZipFile(path) as zf:
        names = zf.namelist()
        parts = {name: zf.read(name) for name in names}
    renamed = {}
    for name in list(parts):
        if not (name.startswith("xl/media/") and name.lower().endswith(".png")):
            continue
        if len(parts[name]) < 400_000:
            continue
        image = Image.open(io.BytesIO(parts[name])).convert("RGB")
        buf = io.BytesIO()
        image.save(buf, "JPEG", quality=quality, optimize=True)
        jpg = name[:-4] + ".jpg"
        parts[jpg] = buf.getvalue()
        del parts[name]
        renamed[name] = jpg
    if not renamed:
        return 0
    for name, body in list(parts.items()):
        if not name.endswith(".rels"):
            continue
        text = body.decode("utf-8")
        for old, new in renamed.items():
            text = text.replace(old, new)
            text = text.replace(old.rsplit("/", 1)[1], new.rsplit("/", 1)[1])
        parts[name] = text.encode("utf-8")
    types = parts["[Content_Types].xml"].decode("utf-8")
    if 'Extension="jpg"' not in types:
        types = types.replace(
            "<Default ",
            '<Default Extension="jpg" ContentType="image/jpeg" /><Default ',
            1,
        )
        parts["[Content_Types].xml"] = types.encode("utf-8")
    order = [n for n in names if n in parts] + [n for n in parts if n not in names]
    tmp = path.with_suffix(".tmp.xlsx")
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as zf:
        for name in order:
            zf.writestr(name, parts[name])
    tmp.replace(path)
    return len(renamed)


## 14. Banners, tables, and chart data

Section bars, the suggestion tables, and the numbers behind the cover charts.


In [ ]:
# ------------------------------------------------------------------------
# 14. Banners, tables, and chart data
# Section bars, the suggestion tables, and the numbers behind the cover charts.
# ------------------------------------------------------------------------

def banner(ws: Worksheet, title: str, subtitle: str, last_col: int = 12) -> None:
    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=last_col)
    write_cell(ws, 1, 1, title, size=20, bold=True, color=WHITE, fill_color=NAVY, align="left")
    ws.row_dimensions[1].height = 28
    ws.merge_cells(start_row=2, start_column=1, end_row=2, end_column=last_col)
    write_cell(ws, 2, 1, subtitle, size=12, color=WHITE, fill_color=NAVY)
    ws.row_dimensions[2].height = 20
    for col in range(1, last_col + 1):
        ws.cell(1, col).fill = fill(NAVY)
        ws.cell(2, col).fill = fill(NAVY)
        ws.cell(1, col).font = font(20, bold=True, color=WHITE)
        ws.cell(2, col).font = font(12, color=WHITE)


def section_bar(ws: Worksheet, row: int, text: str, start_col: int, end_col: int, fill_color=YELLOW) -> None:
    """Yellow section header like IDLE Mode / Active Mode."""
    ws.merge_cells(start_row=row, start_column=start_col, end_row=row, end_column=end_col)
    write_cell(ws, row, start_col, text, size=13, bold=True, color=NAVY, fill_color=fill_color, align="center")
    ws.row_dimensions[row].height = 22
    for col in range(start_col, end_col + 1):
        ws.cell(row, col).fill = fill(fill_color)
        ws.cell(row, col).font = font(13, bold=True, color=NAVY)
        ws.cell(row, col).alignment = Alignment(horizontal="center", vertical="center")


def save_active_placeholder(path: Path) -> Path:
    """Blue placeholder shown until Active Mode logs are provided."""
    PLOTS_DIR.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(7.4, 8.2))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    ax.add_patch(plt.Rectangle((0.12, 0.38), 0.76, 0.24, facecolor="#2E86C1", edgecolor="none", linewidth=0))
    ax.text(
        0.5,
        0.50,
        "Later I will give you data,\nthen plot here",
        ha="center",
        va="center",
        color="white",
        fontsize=14,
        fontweight="bold",
    )
    fig.savefig(path, dpi=140, facecolor="white", bbox_inches="tight")
    plt.close(fig)
    return path


def write_table(ws, start_row: int, start_col: int, headers: list[str], rows: list[list], header_fill=NAVY, num_formats: dict[int, str] | None = None):
    for i, h in enumerate(headers):
        cell = write_cell(ws, start_row, start_col + i, h, size=11, bold=True, color=WHITE, fill_color=header_fill, align="center")
        cell.border = THIN
    for r_i, row in enumerate(rows, start=start_row + 1):
        bg = LIGHT if (r_i - start_row) % 2 == 0 else WHITE
        for c_i, value in enumerate(row):
            fmt = (num_formats or {}).get(c_i)
            cell = write_cell(ws, r_i, start_col + c_i, value, fill_color=bg, align="center" if c_i else "left", num_fmt=fmt)
            cell.border = THIN
    return start_row + len(rows)


def _title_outside(title_obj) -> None:
    if title_obj is None:
        return
    try:
        title_obj.overlay = False
    except Exception:
        pass


def reserve_plot_area(chart, left=0.14, top=0.16, width=0.78, height=0.60) -> None:
    """Keep plot area inside the frame so axis titles and tick levels stay visible."""
    chart.layout = Layout(
        manualLayout=ManualLayout(
            xMode="edge",
            yMode="edge",
            x=left,
            y=top,
            w=width,
            h=height,
        )
    )


def apply_xy_levels(
    chart,
    x_title: str,
    y_title: str,
    y_min=None,
    y_max=None,
    show_all_x_ticks=True,
    x_tick_skip=None,
    rotate_x=False,
    y_major_unit=None,
    y_ax_pos="l",
    layout=True,
    y_num_fmt=None,
) -> None:
    """Put category levels on the bottom (X) and value levels on the left (Y).

    openpyxl defaults both axes to axPos='l', which Excel then draws without
    horizontal tick labels or a usable vertical scale — the bug in the v1.3 charts.
    """
    chart.x_axis.title = x_title
    chart.y_axis.title = y_title
    _title_outside(getattr(chart, "title", None))
    _title_outside(chart.x_axis.title)
    _title_outside(chart.y_axis.title)
    chart.x_axis.delete = False
    chart.y_axis.delete = False
    chart.x_axis.axPos = "b"
    chart.y_axis.axPos = y_ax_pos
    chart.x_axis.tickLblPos = "nextTo"
    chart.y_axis.tickLblPos = "nextTo"
    chart.x_axis.majorTickMark = "out"
    chart.y_axis.majorTickMark = "out"
    chart.x_axis.minorTickMark = "none"
    chart.y_axis.minorTickMark = "none"
    if y_num_fmt:
        chart.y_axis.numFmt = y_num_fmt
    if x_tick_skip is not None:
        try:
            chart.x_axis.tickLblSkip = int(x_tick_skip)
            chart.x_axis.tickMarkSkip = int(x_tick_skip)
        except Exception:
            pass
    elif show_all_x_ticks:
        try:
            chart.x_axis.tickLblSkip = 1
            chart.x_axis.tickMarkSkip = 1
        except Exception:
            pass
    if y_min is not None:
        chart.y_axis.scaling.min = y_min
    if y_max is not None:
        chart.y_axis.scaling.max = y_max
    if y_major_unit is not None:
        try:
            chart.y_axis.majorUnit = y_major_unit
        except Exception:
            pass
    if rotate_x:
        # Rotation only — do not add an empty text run, which blanks tick labels.
        chart.x_axis.txPr = RichText(
            bodyPr=RichTextProperties(rot="-5400000", wrap="square", anchor="ctr"),
            p=[Paragraph(pPr=ParagraphProperties(defRPr=CharacterProperties(sz=700)))],
        )
    if layout:
        reserve_plot_area(chart)


def style_chart(chart, color: str, title: str, y_title: str, x_title: str, width=14, height=8, show_legend=False) -> None:
    chart.title = title
    chart.y_axis.title = y_title
    chart.x_axis.title = x_title
    chart.style = 10
    chart.width = width
    chart.height = height
    if show_legend:
        chart.legend.position = "b"
    else:
        chart.legend = None
    if chart.series:
        chart.series[0].graphicalProperties.solidFill = color
        chart.series[0].graphicalProperties.line.solidFill = color
    apply_xy_levels(chart, x_title, y_title, show_all_x_ticks=False, rotate_x=False)


def add_col_chart(ws_data, ws_dest, anchor, title, color, cat_col, data_col, min_row, max_row, y_title, x_title, width=14, height=8, show_legend=False):
    chart = BarChart()
    chart.type = "col"
    cats = Reference(ws_data, min_col=cat_col, min_row=min_row, max_row=max_row)
    data = Reference(ws_data, min_col=data_col, min_row=min_row - 1, max_row=max_row)
    chart.add_data(data, titles_from_data=True)
    chart.set_categories(cats)
    style_chart(chart, color, title, y_title, x_title, width=width, height=height, show_legend=show_legend)
    chart.shape = 4
    ws_dest.add_chart(chart, anchor)
    return chart


def add_stacked_col_chart(ws_data, ws_dest, anchor, title, cat_col, data_min, data_max, min_row, max_row, y_title, x_title, colors, width=14, height=8):
    chart = BarChart()
    chart.type = "col"
    chart.grouping = "stacked"
    chart.overlap = 100
    cats = Reference(ws_data, min_col=cat_col, min_row=min_row, max_row=max_row)
    data = Reference(ws_data, min_col=data_min, max_col=data_max, min_row=min_row - 1, max_row=max_row)
    chart.add_data(data, titles_from_data=True)
    chart.set_categories(cats)
    chart.title = title
    chart.y_axis.title = y_title
    chart.x_axis.title = x_title
    chart.style = 10
    chart.width = width
    chart.height = height
    chart.legend.position = "b"
    apply_xy_levels(chart, x_title, y_title, show_all_x_ticks=False, rotate_x=False)
    for i, color in enumerate(colors):
        if i < len(chart.series):
            hexcol = color.lstrip("#")
            chart.series[i].graphicalProperties.solidFill = hexcol
            chart.series[i].graphicalProperties.line.solidFill = hexcol
    ws_dest.add_chart(chart, anchor)
    return chart


def add_line_chart(ws_data, ws_dest, anchor, title, color, cat_col, data_col, min_row, max_row, y_title, x_title, width=14, height=8, legend=False):
    chart = LineChart()
    cats = Reference(ws_data, min_col=cat_col, min_row=min_row, max_row=max_row)
    data = Reference(ws_data, min_col=data_col, min_row=min_row - 1, max_row=max_row)
    chart.add_data(data, titles_from_data=True)
    chart.set_categories(cats)
    style_chart(chart, color, title, y_title, x_title, width=width, height=height)
    if legend:
        chart.legend.position = "b"
    chart.series[0].graphicalProperties.line.solidFill = color
    chart.series[0].graphicalProperties.line.width = 18000
    ws_dest.add_chart(chart, anchor)
    return chart


def write_stacked_hist(ws, start_col, values, fine_bins, class_series, class_labels):
    values = pd.to_numeric(values, errors="coerce")
    mask = values.notna()
    values = values[mask]
    classes = class_series.reindex(values.index).astype(str)
    ws.cell(1, start_col, "bin")
    for j, lab in enumerate(class_labels):
        ws.cell(1, start_col + 1 + j, str(lab))
    lefts = fine_bins[:-1]
    for i, left in enumerate(lefts, start=2):
        ws.cell(i, start_col, float(left))
    for j, lab in enumerate(class_labels):
        subset = values[classes == str(lab)]
        counts, _ = np.histogram(subset.to_numpy(), bins=fine_bins)
        for i, n in enumerate(counts, start=2):
            ws.cell(i, start_col + 1 + j, int(n))
    return {
        "cat_col": start_col,
        "data_min": start_col + 1,
        "data_max": start_col + len(class_labels),
        "n": len(lefts),
    }


def write_chart_data(ws: Worksheet, df: pd.DataFrame) -> dict:
    """Hidden sheet holding all series used by Excel charts."""
    rsrp_bins = np.arange(-140, -48, 2)
    rsrq_bins = np.arange(-24, -2.5, 0.5)
    sinr_bins = np.arange(-15, 31, 1)

    rsrp_cls = rsrp_map_class(df["RSRP"])
    rsrq_cls = rsrq_map_class(df["RSRQ"])
    sinr_cls = sinr_map_class(df["SINR"])

    blocks = {
        "RSRP": write_stacked_hist(ws, 26, df["RSRP"], rsrp_bins, rsrp_cls, MAP_RSRP_LABELS),
        "RSRQ": write_stacked_hist(ws, 35, df["RSRQ"], rsrq_bins, rsrq_cls, MAP_RSRQ_LABELS),
        "SINR": write_stacked_hist(ws, 41, df["SINR"], sinr_bins, sinr_cls, MAP_SINR_LABELS),
    }

    percentiles = list(range(0, 101, 2))
    ws.cell(1, 7, "Percentile")
    ws.cell(1, 8, "RSRP_CDF")
    ws.cell(1, 9, "RSRQ_CDF")
    ws.cell(1, 10, "SINR_CDF")
    rsrp = df["RSRP"].dropna()
    rsrq = df["RSRQ"].dropna()
    sinr = df["SINR"].dropna()
    rsrp_p = np.nanpercentile(rsrp, percentiles)
    rsrq_p = np.nanpercentile(rsrq, percentiles)
    sinr_p = np.nanpercentile(sinr, percentiles)
    for i, p in enumerate(percentiles, start=2):
        ws.cell(i, 7, p)
        ws.cell(i, 8, float(rsrp_p[i - 2]))
        ws.cell(i, 9, float(rsrq_p[i - 2]))
        ws.cell(i, 10, float(sinr_p[i - 2]))
    blocks["cdf_n"] = len(percentiles)

    ws.cell(1, 12, "RSRP_quality")
    ws.cell(1, 13, "RSRP_quality_n")
    rsrp_q = rsrp_range_counts(df["RSRP"])
    for i, (label, n) in enumerate(rsrp_q.items(), start=2):
        ws.cell(i, 12, str(label))
        ws.cell(i, 13, int(n))
    blocks["rsrp_q_n"] = len(rsrp_q)
    ws.cell(1, 15, "RSRQ_quality")
    ws.cell(1, 16, "RSRQ_quality_n")
    for i, (label, n) in enumerate(map_range_counts(df["RSRQ"], rsrq_map_class, MAP_RSRQ_LABELS).items(), start=2):
        ws.cell(i, 15, str(label))
        ws.cell(i, 16, int(n))
    ws.cell(1, 18, "SINR_quality")
    ws.cell(1, 19, "SINR_quality_n")
    for i, (label, n) in enumerate(map_range_counts(df["SINR"], sinr_map_class, MAP_SINR_LABELS).items(), start=2):
        ws.cell(i, 18, str(label))
        ws.cell(i, 19, int(n))

    ts = (
        df.dropna(subset=["Time"])
        .set_index("Time")[["RSRP", "RSRQ", "SINR"]]
        .resample("10min")
        .mean()
        .dropna(how="all")
    )
    if len(ts) > 900:
        ts = ts.iloc[:: max(1, len(ts) // 900)]
    ws.cell(1, 21, "Time")
    ws.cell(1, 22, "RSRP")
    ws.cell(1, 23, "RSRQ")
    ws.cell(1, 24, "SINR")
    for i, (t, row) in enumerate(ts.iterrows(), start=2):
        cell = ws.cell(i, 21, pd.Timestamp(t).to_pydatetime())
        cell.number_format = "yyyy-mm-dd hh:mm"
        ws.cell(i, 22, None if pd.isna(row["RSRP"]) else float(row["RSRP"]))
        ws.cell(i, 23, None if pd.isna(row["RSRQ"]) else float(row["RSRQ"]))
        ws.cell(i, 24, None if pd.isna(row["SINR"]) else float(row["SINR"]))
    blocks["ts_n"] = len(ts)

    rsrp_vs_bins = list(range(-80, -126, -1))
    pair = df.copy()
    pair["rsrp_bin"] = pd.to_numeric(pair["RSRP"], errors="coerce").round().astype("Int64")
    ws.cell(1, 50, "RSRP_dBm")
    ws.cell(1, 51, "SINR")
    ws.cell(1, 52, "RSRQ")
    all_mean = pair.groupby("rsrp_bin")[["SINR", "RSRQ"]].mean()
    for i, b in enumerate(rsrp_vs_bins, start=2):
        ws.cell(i, 50, int(b))
        if b in all_mean.index:
            ws.cell(i, 51, None if pd.isna(all_mean.loc[b, "SINR"]) else float(all_mean.loc[b, "SINR"]))
            ws.cell(i, 52, None if pd.isna(all_mean.loc[b, "RSRQ"]) else float(all_mean.loc[b, "RSRQ"]))
    blocks["vs_n"] = len(rsrp_vs_bins)
    return blocks


def write_vs_block(ws: Worksheet, df: pd.DataFrame, cat_col: int, sinr_col: int, rsrq_col: int) -> int:
    """Mean SINR and RSRQ for each 1 dB RSRP bin, used by the RSRP vs KPIs charts."""
    rsrp_vs_bins = list(range(-80, -126, -1))
    pair = df.copy()
    pair["rsrp_bin"] = pd.to_numeric(pair["RSRP"], errors="coerce").round().astype("Int64")
    ws.cell(1, cat_col, "RSRP_dBm")
    ws.cell(1, sinr_col, "SINR")
    ws.cell(1, rsrq_col, "RSRQ")
    all_mean = pair.groupby("rsrp_bin")[["SINR", "RSRQ"]].mean()
    for i, b in enumerate(rsrp_vs_bins, start=2):
        ws.cell(i, cat_col, int(b))
        if b in all_mean.index:
            ws.cell(i, sinr_col, None if pd.isna(all_mean.loc[b, "SINR"]) else float(all_mean.loc[b, "SINR"]))
            ws.cell(i, rsrq_col, None if pd.isna(all_mean.loc[b, "RSRQ"]) else float(all_mean.loc[b, "RSRQ"]))
    return len(rsrp_vs_bins)


def _campaign_label(series: pd.Series) -> str:
    """Collapse SYS suffixes so AUTO VOICE_SYS01 and AUTO VOICE_SYS02 read as AUTO VOICE."""
    values = series.dropna().astype(str).str.strip()
    values = values[values.ne("") & values.str.lower().ne("nan")]
    if values.empty:
        return ""
    upper = values.str.upper()
    if upper.str.contains("AUTO VOICE").all():
        return "AUTO VOICE"
    if upper.str.contains("AUTO DATA").all():
        return "AUTO DATA"
    return str(values.mode().iloc[0])


## 15. Cover sheet and KPI sheets

Drive details, the DT route, and one RSRP, RSRQ, and SINR sheet for Idle and for Active.


In [ ]:
# ------------------------------------------------------------------------
# 15. Cover sheet and KPI sheets
# Drive details, the DT route, and one RSRP, RSRQ, and SINR sheet for Idle and for Active.
# ------------------------------------------------------------------------

def build_cover(
    ws: Worksheet,
    df: pd.DataFrame,
    active: pd.DataFrame | None = None,
    active_raw_n: int = 0,
    active_operator_n: int = 0,
) -> None:
    last_col = 20
    banner(
        ws,
        "  Bogura Drive-Test Report",
        f"  Generated {datetime.now().strftime('%Y-%m-%d %H:%M')}  |  IDLE + Active Mode  |  Gridlines off  |  Freeze panes off",
        last_col=last_col,
    )
    set_widths(ws, {get_column_letter(i): 14 for i in range(1, last_col + 1)})
    ws.column_dimensions["A"].width = 26
    ws.column_dimensions["B"].width = 18
    ws.column_dimensions["E"].width = 20
    section_bar(ws, 3, "IDLE Mode", 1, last_col)

    write_cell(ws, 5, 1, "Dataset", size=14, bold=True)
    info_rows = [
        ["Mode", "IDLE (scanner)"],
        ["Generated", datetime.now().strftime("%Y-%m-%d %H:%M")],
        ["Archive", "Borga_Idle_V2.0.7z"],
        ["CSV files", int(df["Source"].nunique())],
        ["Merged samples", int(len(df))],
        ["Start time", df["Time"].min()],
        ["End time", df["Time"].max()],
        ["Unique Cell Ids", int(df["Cell Id"].nunique())],
        ["Longitude range", f"{df['Longitude'].min():.5f} – {df['Longitude'].max():.5f}"],
        ["Latitude range", f"{df['Latitude'].min():.5f} – {df['Latitude'].max():.5f}"],
        ["Site database", SITE_DB_PATH.name if SITE_DB_PATH.exists() else "not provided"],
    ]
    write_table(ws, 6, 1, ["Item", "Value"], info_rows, num_formats={1: "#,##0"})
    ws.cell(12, 2).number_format = "yyyy-mm-dd hh:mm:ss"
    ws.cell(13, 2).number_format = "yyyy-mm-dd hh:mm:ss"

    write_cell(ws, 5, 4, "Radio KPIs (all samples)", size=14, bold=True)
    stats_headers = ["KPI", "Count", "Min", "P5", "Median", "Mean", "P95", "Max"]
    stats_rows = []
    for name in ("RSRP", "RSRQ", "SINR"):
        st = kpi_stats(df[name])
        stats_rows.append(
            [name, st["Count"], st["Min"], st["P5"], st["Median"], st["Mean"], st["P95"], st["Max"]]
        )
    write_table(
        ws,
        6,
        4,
        stats_headers,
        stats_rows,
        num_formats={1: "#,##0", 2: "0.00", 3: "0.00", 4: "0.00", 5: "0.00", 6: "0.00", 7: "0.00"},
    )

    write_cell(ws, 18, 1, "How to read this workbook", size=14, bold=True)
    notes = [
        "Cover \u2014 Idle dataset and plots on top. Active Mode dataset, histograms and the DT route are below.",
        "RSRP / RSRQ / SINR \u2014 Idle scanner on the left (combined all-band map with bad spots, then L900 / L1800 / L2100 / L2600). Active Mode Robi UE on the right, same layers. No site pies.",
        "Bad Spot RSRP / RSRQ / SINR are Idle. Bad Spot RSRP_Active / RSRQ_Active / SINR_Active apply the same poor-run rules to the Active Robi track. Each bad spot is its own large map, in order. The combined view has no circles.",
        "Poor is RSRP below -120 dBm (red only; -115 through -120 is light red and is not poor), RSRQ at or below -20 dB (inside red -30 to -17), SINR at or below 0 dB (orange/red); every poor run is circled, however short.",
        "RSRP vs KPIs is Idle. RSRP vs KPIs_Active is the same chart set for the Active Robi UE.",
        "Sample Log \u2014 evenly spaced subset of the Idle samples (full logs: output/Bogura_merged.csv.gz and output/Bogura_active_merged.csv.gz).",
    ]
    for i, text in enumerate(notes):
        ws.merge_cells(start_row=19 + i, start_column=1, end_row=19 + i, end_column=11)
        write_cell(ws, 19 + i, 1, text, wrap=True)
        ws.row_dimensions[19 + i].height = 18

    write_cell(ws, 25, 1, "Band mix (DL EARFCN)", size=14, bold=True)
    band = df["Band"].value_counts()
    band_rows = [[str(k), int(v), v / len(df)] for k, v in band.items()]
    write_table(ws, 26, 1, ["Band", "Samples", "Share"], band_rows, num_formats={1: "#,##0", 2: "0.0%"})

    write_cell(ws, 25, 5, "RSRP ranges (dBm)", size=14, bold=True)
    rsrp_q = rsrp_range_counts(df["RSRP"])
    rsrp_rows = [[str(label), int(n), n / len(df)] for label, n in rsrp_q.items()]
    write_table(
        ws,
        26,
        5,
        ["Range", "Samples", "Share"],
        rsrp_rows,
        num_formats={1: "#,##0", 2: "0.0%"},
    )
    write_cell(
        ws,
        36,
        5,
        "Ranges match the coverage-map legend: -95 to 0 (dark green), -110 to -95 (green), -115 to -110 (yellow), -120 to -115 (light red, from -115), -150 to -120 (red).",
        size=9,
        wrap=True,
    )
    ws.merge_cells(start_row=36, start_column=5, end_row=36, end_column=8)
    section_bar(ws, 42, "Active Mode", 1, last_col)
    if active is None or active.empty:
        write_cell(ws, 44, 1, "Active Mode log was not provided.", size=12, color="7F8C8D")
        return
    write_cell(ws, 44, 1, "Dataset", size=14, bold=True)
    plotted_n = int(len(active))
    operator_n = int(active_operator_n) if active_operator_n else plotted_n
    campaign = _campaign_label(active["Campaign"]) if "Campaign" in active.columns else ""
    voice = "VOICE" in campaign.upper()
    info_rows = [
        ["Mode", "Active (UE voice)" if voice else "Active (UE)"],
        ["Archive", ACTIVE_ARCHIVE.name],
        ["Campaign", campaign or "—"],
        ["Home Operator", ACTIVE_OPERATOR],
        ["Type", ACTIVE_CELL_TYPE],
        [f"{ACTIVE_OPERATOR} {ACTIVE_CELL_TYPE} rows", operator_n],
        ["Plotted samples", plotted_n],
        ["CSV files", int(active["Source"].nunique())],
        ["Start time", active["Time"].min()],
        ["End time", active["Time"].max()],
        ["Unique Cell Ids", int(active["Cell Id"].nunique())],
    ]
    write_table(ws, 45, 1, ["Item", "Value"], info_rows, num_formats={1: "#,##0"})
    ws.cell(54, 2).number_format = "yyyy-mm-dd hh:mm:ss"
    ws.cell(55, 2).number_format = "yyyy-mm-dd hh:mm:ss"
    write_cell(ws, 44, 4, f"Radio KPIs ({ACTIVE_OPERATOR}, plotted rows)", size=14, bold=True)
    stats_headers = ["KPI", "Count", "Min", "P5", "Median", "Mean", "P95", "Max"]
    stats_rows = []
    for name in ("RSRP", "RSRQ", "SINR"):
        st = kpi_stats(active[name])
        stats_rows.append(
            [name, st["Count"], st["Min"], st["P5"], st["Median"], st["Mean"], st["P95"], st["Max"]]
        )
    write_table(
        ws, 45, 4, stats_headers, stats_rows,
        num_formats={1: "#,##0", 2: "0.00", 3: "0.00", 4: "0.00", 5: "0.00", 6: "0.00", 7: "0.00"},
    )
    ws.merge_cells(start_row=50, start_column=4, end_row=51, end_column=11)
    write_cell(
        ws, 50, 4,
        f"Maps and bad spots use Home Operator {ACTIVE_OPERATOR} and Type {ACTIVE_CELL_TYPE}. "
        f"Where several of those rows share a latitude and longitude, the strongest RSRP is kept. "
        "That row is one band, and its RSRQ and SINR are the values drawn. "
        "Cell Id is the Cell Id column, or the last number of the CGI when that column is empty. "
        f"The log is {ACTIVE_ARCHIVE.name}. Start and end are the timestamps in that log.",
        size=9, wrap=True,
    )


def build_kpi_sheet(
    ws,
    name: str,
    unit: str,
    map_path: Path | None,
    active_map: Path | None,
    band_maps: dict[str, Path],
    active_band_maps: dict[str, Path] | None = None,
):
    last_col = 18
    em = "\u2014"
    banner(ws, f"  {name} Report", f"  Unit: {unit}  |  IDLE and Active coverage maps  |  Gridlines off", last_col=last_col)
    set_widths(ws, {get_column_letter(i): 14 for i in range(1, last_col + 1)})
    section_bar(ws, 4, f"{name} coverage map {em} IDLE Mode  |  Combined view  |  bad spots", 1, 8)
    section_bar(ws, 4, f"{name} coverage map {em} Active Mode  |  {ACTIVE_OPERATOR}", 10, 18)
    if map_path and map_path.exists():
        add_image(ws, map_path, "A6", width=620, height=760)
    if active_map and active_map.exists():
        add_image(ws, active_map, "J6", width=620, height=760)

    start_row = 6 + 760 // 20 + 3
    block = 700 // 20 + 5
    active_band_maps = active_band_maps or {}
    for i, layer in enumerate(REPORT_LAYERS):
        row = start_row + i * block
        section_bar(ws, row, f"{name} coverage map {em} IDLE Mode_{layer}", 1, 8)
        section_bar(ws, row, f"{name} coverage map {em} Active Mode_{layer}", 10, 18)
        band_path = band_maps.get(layer)
        if band_path and band_path.exists():
            add_image(ws, band_path, f"A{row + 2}", width=560, height=700)
        active_path = active_band_maps.get(layer)
        if active_path and active_path.exists():
            add_image(ws, active_path, f"J{row + 2}", width=560, height=700)


def _kind_label(kind: object, kpi: object = None) -> str:
    text = str(kind or "")
    if text == "discrete":
        return "Discrete area \u2265 1 km\u00b2"
    return consecutive_label(str(kpi)) if kpi in BAD_SPOT_RULES else "Consecutive \u2265 200 m"


BAD_SPOT_KPI_NOTES = {
    "RSRP": (
        "Poor is RSRP below -120 dBm. Only the red bin is a bad spot. Light red, from -115 through -120, is not a bad spot. "
        "Every poor run is covered by a circle. Nearby runs join one circle, up to 10 km across, "
        "and a longer poor road is split into several circles. "
        "The combined view at the top is the all-band best-server map on satellite imagery, the same basemap as the "
        "RSRP sheet. The legend is on the left with the sample count of each colour, and there are no circles. "
        "Best server is the strongest band per scanner sweep, so a weak band is never mapped as the coverage of a road."
    ),
    "RSRQ": (
        "Poor is RSRQ at or below -20 dB. That sits inside the red bin (-30 to -17); red above -20 is not a bad spot. "
        "Every poor run is covered by a circle. "
        "Nearby runs join one circle, up to 10 km across, and separate pockets stay separate circles. "
        "The combined view at the top is the all-band best-server map on satellite imagery, the same basemap as the "
        "RSRP sheet. The legend is on the left with the sample count of each colour, and there are no circles."
    ),
    "SINR": (
        "Poor is SINR at or below 0 dB (orange and red), and every poor run is covered by a circle. "
        "Nearby runs join one circle, up to 10 km across. Where the poor road is spread out, "
        "several circles sit inside the same 10 km neighbourhood instead of one circle covering the good road between them. "
        "The combined view at the top is the all-band best-server map on satellite imagery, with no circles."
    ),
}


def place_solution_summary(ws, row: int, plans: list[dict]) -> int:
    """Count of tilt, 4th sector, and new site, just above the bad-spot table."""
    counts = {"tune": 0, "fourth": 0, "new": 0}
    for plan in plans:
        kind = str(plan.get("kind") or "")
        if kind in counts:
            counts[kind] += 1
    write_cell(ws, row, 1, "Summary of proposed solutions", size=14, bold=True, color=NAVY)
    write_table(
        ws,
        row + 2,
        1,
        ["Solution", "Count"],
        [
            ["Tilt and azimuth", counts["tune"]],
            ["4th sector, L900", counts["fourth"]],
            ["New 3-sector site", counts["new"]],
            ["All bad spots", len(plans)],
        ],
        num_formats={1: "#,##0"},
    )
    return row + 8


def _bad_spot_table_rows(spots: pd.DataFrame, *, include_kpi: bool = True, plans: list[dict] | None = None) -> list[list]:
    rows = []
    for n, (_, s) in enumerate(spots.iterrows(), start=1):
        top_cell = "" if pd.isna(s.get("top_cell", pd.NA)) else int(s["top_cell"])
        length_m = s.get("length_m", np.nan)
        area_km2 = s.get("area_km2", np.nan)
        row = [
            f"Circle {n}",
            _kind_label(s.get("kind"), s.get("kpi")),
        ]
        if include_kpi:
            row.append(str(s["kpi"]))
        row.extend(
            [
                float(s["lat"]),
                float(s["lon"]),
                int(s["n"]),
                int(s["n_poor"]),
                float(s["map_poor_share"]) if "map_poor_share" in s.index and pd.notna(s.get("map_poor_share")) else float(s["poor_pct"]),
                float(s["mean"]),
                float(s["mean_rsrp"]),
                float(s["mean_rsrq"]),
                float(s["mean_sinr"]),
                None if pd.isna(length_m) else float(length_m),
                None if pd.isna(area_km2) else float(area_km2),
                top_cell,
            ]
        )
        if plans is not None:
            plan = plans[n - 1]
            row.extend([plan.get("site", ""), plan.get("population", ""), plan.get("final", "")])
        rows.append(row)
    return rows


def _detection_summary(kpi: str, spots: pd.DataFrame) -> str:
    """State how many spots the rules found and how many of them are drawn."""
    if spots is None or spots.empty:
        return (
            f"The rules find 0 poor {kpi} circle(s) and 0 discrete area(s). "
            f"No {kpi} location meets either rule."
        )
    total_consec = int(spots.attrs.get("total_consecutive", int((spots["kind"] == "consecutive").sum())))
    total_discrete = int(spots.attrs.get("total_discrete", int((spots["kind"] == "discrete").sum())))
    shown_consec = int((spots["kind"] == "consecutive").sum())
    minimum = min_consecutive_m(kpi)
    if minimum <= 0:
        text = (
            f"The rules find {total_consec:,} poor {kpi} circle(s), each drawn up to 10 km and together "
            f"covering every poor run, and {total_discrete:,} discrete area(s) of at least 1 km\u00b2."
        )
    else:
        text = (
            f"The rules find {total_consec:,} consecutive {kpi} run(s) of at least {minimum:,.0f} m "
            f"and {total_discrete:,} discrete area(s) of at least 1 km\u00b2."
        )
    if total_consec > shown_consec:
        text += f" The {shown_consec:,} longest runs are mapped and listed below."
    longest_short = float(spots.attrs.get("longest_short_m", 0.0))
    if minimum > 0 and longest_short >= 50:
        text += (
            f" The longest poor run that misses the rule is {math.floor(longest_short):,} m, "
            "so shorter poor patches on the map are left unmarked."
        )
    return text


def _zoom_display_height(path: Path | None, spots: pd.DataFrame, width: int = 860) -> int:
    n = 0 if spots is None or spots.empty else int(min(len(spots), 16))
    rows = max(1, (n + 3) // 4)
    return int(min(280 * rows, 1120))


def place_bad_spot_advice(ws, header_row: int, plan: dict) -> None:
    """Suggestion table on the right of one bad-spot map.

    Row heights stay at the default, so the movable circle on that map stays put.
    """
    title = header_row + 2
    find = header_row + 4
    head = header_row + 10
    ws.merge_cells(start_row=title, start_column=9, end_row=title, end_column=16)
    write_cell(
        ws, title, 9, "Suggestions and solutions",
        size=11, bold=True, color=NAVY, fill_color=LIGHT, align="center",
    )
    for col in range(9, 17):
        ws.cell(title, col).fill = fill(LIGHT)
        ws.cell(title, col).font = font(11, bold=True, color=NAVY)
        ws.cell(title, col).alignment = Alignment(horizontal="center", vertical="center")
    ws.merge_cells(start_row=find, start_column=9, end_row=find + 4, end_column=9)
    ws.merge_cells(start_row=find, start_column=10, end_row=find + 4, end_column=16)
    write_cell(ws, find, 9, "Findings", size=11, bold=True, color=NAVY, align="center", wrap=True)
    write_cell(ws, find, 10, plan.get("finding", ""), size=11, color=NAVY, align="left", wrap=True)
    ws.cell(find, 9).alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.cell(find, 10).alignment = Alignment(horizontal="left", vertical="top", wrap_text=True)
    ws.merge_cells(start_row=head, start_column=10, end_row=head, end_column=11)
    ws.merge_cells(start_row=head, start_column=12, end_row=head, end_column=16)
    for col, text in ((9, "Step"), (10, "Action"), (12, "Proposal")):
        write_cell(ws, head, col, text, size=11, bold=True, color=NAVY, fill_color=LIGHT, align="center")
        ws.cell(head, col).alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        ws.cell(head, col).border = THIN
    for col in range(9, 17):
        ws.cell(head, col).fill = fill(LIGHT)
        ws.cell(head, col).border = THIN
        ws.cell(head, col).font = font(11, bold=True, color=NAVY)
    for index, step in enumerate(plan.get("steps") or []):
        row = head + 1 + index * 4
        ws.merge_cells(start_row=row, start_column=9, end_row=row + 3, end_column=9)
        ws.merge_cells(start_row=row, start_column=10, end_row=row + 3, end_column=11)
        ws.merge_cells(start_row=row, start_column=12, end_row=row + 3, end_column=16)
        write_cell(ws, row, 9, step.get("step", ""), size=12, bold=True, color=NAVY, align="center", wrap=True)
        write_cell(ws, row, 10, step.get("action", ""), size=11, bold=True, color=NAVY, align="center", wrap=True)
        write_cell(ws, row, 12, step.get("proposal", ""), size=11, color=NAVY, align="left", wrap=True)
        for col in (9, 10, 12):
            ws.cell(row, col).alignment = Alignment(
                horizontal="center" if col < 12 else "left", vertical="top", wrap_text=True,
            )
            ws.cell(row, col).border = THIN


## 16. Bad-spot sheets

One map, one suggestion table, and one movable circle for every RSRP bad spot. Idle has 26. Active has 99.


In [ ]:
# ------------------------------------------------------------------------
# 16. Bad-spot sheets
# One map, one suggestion table, and one movable circle for every RSRP bad spot. Idle has 26. Active has 99.
# ------------------------------------------------------------------------

def build_kpi_bad_spot_sheet(
    ws,
    kpi: str,
    spots: pd.DataFrame,
    combined_map: Path,
    serial_maps: list[Path] | None,
    *,
    mode: str = "IDLE Mode",
    samples: pd.DataFrame | None = None,
) -> None:
    """Combined route with no circles, then one large map per bad spot down the left side."""
    last_col = 17
    map_span = 8
    em = "\u2014"
    banner(
        ws,
        f"  Bad Spot Analysis  {em}  {kpi}",
        f"  {mode}  |  Combined view has no circles  |  Bad spots in order  |  No numbering",
        last_col=last_col,
    )
    set_widths(ws, {get_column_letter(i): 13 for i in range(1, last_col + 1)})
    ws.column_dimensions["A"].width = 16
    ws.column_dimensions["C"].width = 24
    ws.column_dimensions["O"].width = 18
    ws.column_dimensions["P"].width = 16
    ws.column_dimensions["Q"].width = 64
    section_bar(ws, 3, mode, 1, last_col)
    ws.merge_cells(start_row=4, start_column=1, end_row=7, end_column=last_col)
    active_rule = ""
    if mode.startswith("Active"):
        active_rule = (
            f"Active Mode uses Home Operator {ACTIVE_OPERATOR} and Type {ACTIVE_CELL_TYPE}, "
            "so one band is plotted at a location: the strongest RSRP. "
            "The RSRQ and SINR maps use that same row. "
        )
    note = (
        active_rule
        + BAD_SPOT_KPI_NOTES[kpi]
        + " Maps and rules read the same 50 m road bins: a bin is coloured by the median of the "
        "samples measured in it and counts as poor only when that median is "
        + ("below -120 dBm" if kpi == "RSRP" else "at or below the threshold")
        + ", and poor bins are drawn on top of good ones. A circle can therefore only sit on "
        "colour you can see. The combined view has no circles. Each bad spot below is one large map, "
        "Bad Spot 1 then Bad Spot 2, and its orange circle is a separate shape named Circle 1, "
        "Circle 2, and so on, in the same order as the table, so you can drag it. The RSRP, RSRQ and "
        "SINR sheets do not repeat these circles. Each bad-spot map draws a transparent cyan outline "
        "for every sector of a site that falls in that view, from System DUMP.xlsb. No sector of "
        "that site is left out, and the outline is not filled. The label on each arm is the Sector "
        "name. A yellow dashed line starts at the end of the serving sector, not at the middle of the site, "
        "and runs to each red sample inside the orange circle. That cell is the Cell Id on the sample, "
        "on L900, L1800, L2100 or L2600, and it has to be in System DUMP.xlsb. Every serving cell is drawn. "
        "The length beside the red is the full red stretch, and a short gap of good coverage stays in that length. "
        "The columns to the right of each map are a suggestion table. Azimuth and tilt are used "
        "when a sector is within about 2.5 km, and downtilt is limited to 2°. A larger downtilt "
        "becomes a 4th L900 sector on a site within 1 km. A 4th sector also replaces a new site "
        "when a host inside 3 km can aim at the bad spot. A new 3-sector site is kept only past that. "
        "A blank with no plot inside the circle is counted "
        "with that bad spot. The list at the bottom has every bad spot, with Site, Population "
        "and Final solution on the right. "
        "Several circles can sit inside one 10 km neighbourhood when the poor "
        "road is not one stretch. A discrete patch of at least 1 km\u00b2 is a dotted circle. "
        + _detection_summary(kpi, spots)
        + " Numbers are not drawn on the maps."
    )
    if mode.startswith("Active"):
        note = note.replace(
            "Best server is the strongest band per scanner sweep, so a weak band is never mapped as the coverage of a road.",
            "That map keeps the strongest RSRP at each latitude and longitude, so only one Cell Id and one band are drawn there.",
        )
    write_cell(
        ws,
        4,
        1,
        note,
        size=11,
        wrap=True,
    )
    for row in range(4, 8):
        ws.row_dimensions[row].height = 36

    section_bar(ws, 9, f"Combined view {em} all-band {mode}  |  {kpi}  |  no circles", 1, last_col)
    map_h = 0
    if combined_map.exists():
        _, map_h = add_image(ws, combined_map, "A11", width=760, height=900)

    cursor = 11 + max(map_h, 660) // 20 + 3
    listed = spots if spots is None or len(spots) <= BAD_SPOT_TABLE_MAX else spots.head(BAD_SPOT_TABLE_MAX)
    plans = None
    if samples is not None and listed is not None and not listed.empty:
        plans = [bad_spot_advice(samples, kpi, listed.iloc[i]) for i in range(len(listed))]
    maps = [p for p in (serial_maps or []) if p is not None and p.exists()]
    for i, path in enumerate(maps, start=1):
        section_bar(ws, cursor, f"Bad Spot {i}", 1, map_span)
        if plans is not None and i <= len(plans):
            place_bad_spot_advice(ws, cursor, plans[i - 1])
        _, spot_h = add_image(ws, path, f"A{cursor + 2}", width=780, height=720)
        cursor = cursor + 2 + max(spot_h, 360) // 20 + 3
    table_row = cursor if maps else cursor + 1
    if plans:
        table_row = place_solution_summary(ws, table_row, plans)

    table_title = f"Identified {kpi} bad spots"
    if spots is not None and len(spots) > BAD_SPOT_TABLE_MAX:
        table_title = f"{table_title}  \u2014  {BAD_SPOT_TABLE_MAX:,} longest of {len(spots):,}"
    write_cell(ws, table_row, 1, table_title, size=14, bold=True)
    headers = [
        "Circle",
        "Type",
        "Latitude",
        "Longitude",
        "Samples",
        "Poor samples",
        "Poor share",
        "Mean KPI",
        "Mean RSRP",
        "Mean RSRQ",
        "Mean SINR",
        "Length (m)",
        "Area (km\u00b2)",
        "Top Cell Id",
        "Site",
        "Population",
        "Final solution",
    ]
    rows = _bad_spot_table_rows(listed, include_kpi=False, plans=plans)
    write_table(
        ws,
        table_row + 1,
        1,
        headers,
        rows,
        num_formats={
            2: "0.00000",
            3: "0.00000",
            4: "#,##0",
            5: "#,##0",
            6: "0.0%",
            7: "0.00",
            8: "0.00",
            9: "0.00",
            10: "0.00",
            11: "#,##0",
            12: "0.00",
        },
    )
    if plans is not None:
        for offset in range(len(rows)):
            cell = ws.cell(table_row + 2 + offset, 17)
            cell.alignment = Alignment(horizontal="left", vertical="center", wrap_text=True)
            ws.row_dimensions[table_row + 2 + offset].height = 48
    note_row = table_row + 2 + len(rows)
    ws.merge_cells(start_row=note_row, start_column=1, end_row=note_row + 1, end_column=13)
    if kpi == "SINR":
        combined_line = "The combined view is the all-band coverage map on satellite imagery and has no circles. "
    else:
        combined_line = (
            "The combined view is the all-band coverage map on satellite imagery, the same basemap as the RSRP sheet. "
            "The legend is on the left with the sample count of each colour, and there are no circles. "
        )
    write_cell(
        ws,
        note_row,
        1,
        combined_line
        + "Each bad spot above is one large map. Its orange circle is a separate shape with the same "
        "name as the Circle column, so it can be dragged on its own. The space to the right of each "
        "map is that suggestion table. Site, Population and Final solution are filled for every bad spot "
        "in the table below. A blank with no plot inside the circle is counted with that bad spot. "
        "Poor share is the share of raw samples inside the spot box that are at or "
        "below the threshold. Length is the poor road inside the circle, and a single 50 m bin is "
        "reported as 50 m. Map numbers are omitted.",
        size=9,
        wrap=True,
    )
    ws.row_dimensions[note_row].height = 22
    ws.row_dimensions[note_row + 1].height = 22


## 17. Versus charts and the sample log

RSRP against SINR and RSRQ, and a sample of the merged rows.


In [ ]:
# ------------------------------------------------------------------------
# 17. Versus charts and the sample log
# RSRP against SINR and RSRQ, and a sample of the merged rows.
# ------------------------------------------------------------------------

def style_smooth_line(series, color: str, width=25000) -> None:
    hexcol = color.lstrip("#")
    series.graphicalProperties.line.solidFill = hexcol
    series.graphicalProperties.line.width = width
    series.smooth = True
    series.marker = Marker(symbol="none")


def add_vs_line_chart(ws_data, ws_dest, anchor, title, cat_col, data_min, data_max, min_row, max_row, y_title, colors, width=16, height=8):
    chart = LineChart()
    chart.style = 10
    chart.title = title
    chart.y_axis.title = y_title
    chart.x_axis.title = "RSRP (dBm)"
    chart.width = width
    chart.height = height
    chart.legend = None
    cats = Reference(ws_data, min_col=cat_col, min_row=min_row, max_row=max_row)
    data = Reference(ws_data, min_col=data_min, max_col=data_max, min_row=min_row - 1, max_row=max_row)
    chart.add_data(data, titles_from_data=True)
    chart.set_categories(cats)
    for i, color in enumerate(colors):
        if i >= len(chart.series):
            break
        style_smooth_line(chart.series[i], color)
    if "RSRQ" in y_title:
        y_min, y_max, y_major, y_fmt = -20, 0, 5, "0.0"
    else:
        y_min, y_max, y_major, y_fmt = 0, 30, 5, "0"
    apply_xy_levels(
        chart, "RSRP (dBm)", y_title,
        y_min=y_min, y_max=y_max, show_all_x_ticks=False, x_tick_skip=5, rotate_x=False,
        y_major_unit=y_major, y_num_fmt=y_fmt,
    )
    reserve_plot_area(chart, left=0.12, top=0.14, width=0.82, height=0.70)
    ws_dest.add_chart(chart, anchor)
    return chart


def add_dual_axis_vs_chart(ws_data, ws_dest, anchor, n_rows, width=16, height=9, cat_col=50, sinr_col=51, rsrq_col=52):
    cats = Reference(ws_data, min_col=cat_col, min_row=2, max_row=1 + n_rows)
    sinr = LineChart()
    sinr.style = 10
    sinr.title = "RSRP vs SINR vs RSRQ"
    sinr.y_axis.title = "SINR (dB)"
    sinr.x_axis.title = "RSRP (dBm)"
    sinr.width = width
    sinr.height = height
    sinr.add_data(Reference(ws_data, min_col=sinr_col, min_row=1, max_row=1 + n_rows), titles_from_data=True)
    sinr.set_categories(cats)
    style_smooth_line(sinr.series[0], VS_SINR_LINE)
    sinr.legend.position = "b"

    rsrq = LineChart()
    rsrq.y_axis.axId = 200
    rsrq.y_axis.title = "RSRQ (dB)"
    rsrq.add_data(Reference(ws_data, min_col=rsrq_col, min_row=1, max_row=1 + n_rows), titles_from_data=True)
    style_smooth_line(rsrq.series[0], VS_RSRQ_LINE)
    sinr.y_axis.crosses = "min"
    rsrq.y_axis.crosses = "max"
    apply_xy_levels(
        sinr, "RSRP (dBm)", "SINR (dB)",
        y_min=0, y_max=30, show_all_x_ticks=False, x_tick_skip=5, rotate_x=False,
        y_major_unit=5, layout=False, y_num_fmt="0",
    )
    reserve_plot_area(sinr, left=0.12, top=0.14, width=0.74, height=0.58)
    rsrq.y_axis.delete = False
    rsrq.y_axis.axPos = "r"
    rsrq.y_axis.tickLblPos = "nextTo"
    rsrq.y_axis.majorTickMark = "out"
    rsrq.y_axis.minorTickMark = "none"
    rsrq.y_axis.title = "RSRQ (dB)"
    _title_outside(rsrq.y_axis.title)
    rsrq.y_axis.scaling.min = -20
    rsrq.y_axis.scaling.max = 0
    rsrq.y_axis.numFmt = "0.0"
    try:
        rsrq.y_axis.majorUnit = 10
    except Exception:
        pass
    sinr += rsrq
    ws_dest.add_chart(sinr, anchor)


def save_rsrp_scatter(df: pd.DataFrame, ycol: str, path: Path, ylabel: str, title: str, ylim=None) -> Path:
    sample = downsample(df.dropna(subset=["RSRP", ycol]), 18000)
    fig, ax = plt.subplots(figsize=(9.2, 6.2))
    ax.scatter(sample["RSRP"], sample[ycol], s=6, alpha=0.22, c="#1F618D", linewidths=0)
    x = sample["RSRP"].to_numpy()
    y = sample[ycol].to_numpy()
    if len(x) > 20:
        coeff = np.polyfit(x, y, 1)
        xs = np.linspace(np.nanmin(x), np.nanmax(x), 80)
        ax.plot(xs, np.polyval(coeff, xs), color="#17202A", lw=1.8, label="Trend")
        ax.legend(fontsize=8)
    ax.set_xlabel("RSRP (dBm)")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_xlim(-50, -130)
    if ylim:
        ax.set_ylim(*ylim)
    ax.xaxis.set_major_locator(MultipleLocator(5))
    ax.yaxis.set_major_locator(MultipleLocator(5 if "SINR" in ylabel else 2))
    ax.tick_params(axis="both", labelsize=8)
    ax.grid(True, alpha=0.35)
    fig.tight_layout()
    fig.savefig(path, dpi=140, facecolor="white")
    plt.close(fig)
    return path


def save_binned_vs_preview(df: pd.DataFrame, path: Path) -> Path:
    pair = df.dropna(subset=["RSRP", "SINR", "RSRQ"]).copy()
    pair["rsrp_bin"] = pair["RSRP"].round().astype(int)
    g = pair.groupby("rsrp_bin")[["SINR", "RSRQ"]].mean()
    g = g.loc[(g.index <= -80) & (g.index >= -125)].sort_index(ascending=False)
    fig, axes = plt.subplots(1, 3, figsize=(18.6, 5.2))

    ax = axes[0]
    ax.plot(g.index, g["SINR"], color=f"#{VS_SINR_LINE}", lw=2.4)
    ax.set_title("RSRP vs SINR")
    ax.set_xlabel("RSRP (dBm)")
    ax.set_ylabel("SINR (dB)")
    ax.set_xlim(-80, -125)
    ax.set_ylim(0, 30)
    ax.set_xticks(range(-80, -126, -5))
    ax.yaxis.set_major_locator(MultipleLocator(5))
    ax.grid(True, axis="y", color="#D5D8DC", lw=0.8)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax = axes[1]
    ax.plot(g.index, g["RSRQ"], color=f"#{VS_RSRQ_LINE}", lw=2.4)
    ax.set_title("RSRP vs RSRQ")
    ax.set_xlabel("RSRP (dBm)")
    ax.set_ylabel("RSRQ (dB)")
    ax.set_xlim(-80, -125)
    ax.set_ylim(-20, 0)
    ax.set_xticks(range(-80, -126, -5))
    ax.yaxis.set_major_locator(MultipleLocator(5))
    ax.grid(True, axis="y", color="#D5D8DC", lw=0.8)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax = axes[2]
    ax.plot(g.index, g["SINR"], color=f"#{VS_SINR_LINE}", lw=2.4, label="SINR")
    ax.set_ylim(0, 30)
    ax.set_ylabel("SINR (dB)")
    ax.yaxis.set_major_locator(MultipleLocator(5))
    ax2 = ax.twinx()
    ax2.plot(g.index, g["RSRQ"], color=f"#{VS_RSRQ_LINE}", lw=2.4, label="RSRQ")
    ax2.set_ylim(-20, 0)
    ax2.set_ylabel("RSRQ (dB)")
    ax2.yaxis.set_major_locator(MultipleLocator(10))
    ax.set_title("RSRP vs SINR vs RSRQ")
    ax.set_xlabel("RSRP (dBm)")
    ax.set_xlim(-80, -125)
    ax.set_xticks(range(-80, -126, -5))
    ax.grid(True, axis="y", color="#D5D8DC", lw=0.8)
    ax.spines["top"].set_visible(False)
    handles = ax.get_lines() + ax2.get_lines()
    labels = [h.get_label() for h in handles]
    ax.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=2, frameon=False)
    fig.tight_layout()
    fig.savefig(path, dpi=140, facecolor="white")
    plt.close(fig)
    return path


def build_rsrp_vs_sheet(
    ws, data_ws, n_rows: int, scatter_sinr: Path, scatter_rsrq: Path,
    *,
    title: str = "  RSRP vs KPIs IDLE Mode",
    subtitle: str = "  Idle V2.0 scanner  |  Smooth mean lines and scatter with trend  |  Gridlines off",
    cat_col: int = 50,
    sinr_col: int = 51,
    rsrq_col: int = 52,
) -> None:
    banner(ws, title, subtitle, last_col=12)
    set_widths(ws, {get_column_letter(i): 14 for i in range(1, 13)})
    add_vs_line_chart(
        data_ws, ws, "A4", "RSRP vs SINR",
        cat_col, sinr_col, sinr_col, 2, 1 + n_rows, "SINR (dB)",
        [VS_SINR_LINE], width=15, height=8,
    )
    add_vs_line_chart(
        data_ws, ws, "I4", "RSRP vs RSRQ",
        cat_col, rsrq_col, rsrq_col, 2, 1 + n_rows, "RSRQ (dB)",
        [VS_RSRQ_LINE], width=15, height=8,
    )
    add_dual_axis_vs_chart(
        data_ws, ws, "A22", n_rows, width=22, height=8,
        cat_col=cat_col, sinr_col=sinr_col, rsrq_col=rsrq_col,
    )
    write_cell(ws, 48, 1, "RSRP vs SINR scatter (trend line)", size=14, bold=True)
    write_cell(ws, 48, 8, "RSRP vs RSRQ scatter (trend line)", size=14, bold=True)
    if scatter_sinr.exists():
        add_image(ws, scatter_sinr, "A50", width=520, height=350)
    if scatter_rsrq.exists():
        add_image(ws, scatter_rsrq, "H50", width=520, height=350)


def build_sample_log(ws: Worksheet, df: pd.DataFrame, n: int = 8000) -> None:
    banner(ws, "  Sample Drive Log IDLE Mode", "  Evenly spaced subset for inspection  |  Full merged file is CSV  |  No freeze / no gridlines", last_col=10)
    sample = downsample(df, max_points=n).copy()
    cols = ["Time", "RSRP", "RSRQ", "SINR", "Longitude", "Latitude", "Cell Id", "DL EARFCN", "Source", "Band"]
    sample = sample[cols]
    for c, w in zip(cols, (22, 12, 12, 12, 14, 14, 14, 14, 10, 22)):
        ws.column_dimensions[get_column_letter(cols.index(c) + 1)].width = w
    headers = cols
    for i, h in enumerate(headers, start=1):
        write_cell(ws, 4, i, h, bold=True, color=WHITE, fill_color=NAVY, align="center")
    for r_i, row in enumerate(sample.itertuples(index=False), start=5):
        bg = LIGHT if r_i % 2 == 0 else WHITE
        values = list(row)
        for c_i, value in enumerate(values, start=1):
            if isinstance(value, pd.Timestamp):
                value = value.to_pydatetime()
            if hasattr(value, "item"):
                try:
                    value = value.item()
                except Exception:
                    pass
            if pd.isna(value):
                value = None
            cell = write_cell(ws, r_i, c_i, value, fill_color=bg, align="center")
            cell.border = THIN
            if c_i == 1 and value is not None:
                cell.number_format = "yyyy-mm-dd hh:mm:ss.000"
            if c_i in (2, 3, 4, 5, 6):
                cell.number_format = "0.000"


## 18. Check the workbook, and the cover charts

verify_report checks freeze panes, gridlines, the sheet list, and the 125 movable circles.
The cover charts and the active route sit on the Cover sheet.


In [ ]:
# ------------------------------------------------------------------------
# 18. Check the workbook, and the cover charts
# verify_report checks freeze panes, gridlines, the sheet list, and the 125 movable circles.
# The cover charts and the active route sit on the Cover sheet.
# ------------------------------------------------------------------------

def _chart_title_text(title) -> str | None:
    if title is None:
        return None
    try:
        return str(title.tx.rich.p[0].r[0].t)
    except Exception:
        return str(title)


def verify_report(path: Path) -> dict:
    wb = load_workbook(path)
    info = {"sheets": {}, "chart_titles": [], "axis_check": [], "problems": []}
    for name in wb.sheetnames:
        ws = wb[name]
        freeze = ws.freeze_panes
        grid = ws.sheet_view.showGridLines
        print_grid = ws.print_options.gridLines
        n_charts = len(ws._charts)
        n_images = len(ws._images)
        info["sheets"][name] = {
            "freeze_panes": freeze,
            "showGridLines": grid,
            "print_gridLines": print_grid,
            "charts": n_charts,
            "images": n_images,
        }
        if freeze not in (None, "A1"):
            info["problems"].append(f"{name}: freeze_panes={freeze}")
        if grid is not False:
            info["problems"].append(f"{name}: showGridLines={grid}")
        if print_grid:
            info["problems"].append(f"{name}: print gridlines on")
        for ch in ws._charts:
            title = _chart_title_text(getattr(ch, "title", None))
            x_title = _chart_title_text(getattr(ch.x_axis, "title", None))
            y_title = _chart_title_text(getattr(ch.y_axis, "title", None))
            x_pos = getattr(ch.x_axis, "axPos", None)
            y_pos = getattr(ch.y_axis, "axPos", None)
            info["chart_titles"].append((name, title, type(ch).__name__))
            info["axis_check"].append((name, title, x_title, y_title, x_pos, y_pos))
            if x_pos != "b":
                info["problems"].append(f"{name}/{title}: x axPos={x_pos} (expected b)")
            if y_pos not in ("l", "r"):
                info["problems"].append(f"{name}/{title}: y axPos={y_pos} (expected l/r)")
            if not x_title:
                info["problems"].append(f"{name}/{title}: missing horizontal axis title")
            if not y_title:
                info["problems"].append(f"{name}/{title}: missing vertical axis title")
    for name in (
        "Bad Spot RSRP",
        "Bad Spot RSRP_Active",
    ):
        if name not in info["sheets"]:
            info["problems"].append(f"missing {name} sheet")
        elif info["sheets"][name]["images"] < 1:
            info["problems"].append(f"{name}: expected combined map image")
    if "RSRP vs KPIs_Active" not in info["sheets"]:
        info["problems"].append("missing RSRP vs KPIs_Active sheet")
    elif info["sheets"]["RSRP vs KPIs_Active"]["charts"] < 3:
        info["problems"].append("RSRP vs KPIs_Active: expected three charts")
    info["problems"].extend(logic_drift())
    count_re = re.compile(r"find ([\d,]+) .*? and ([\d,]+) discrete")
    for sheet, (runs, areas) in LOGIC_LOCK["counts"].items():
        text = str(wb[sheet].cell(4, 1).value or "") if sheet in wb.sheetnames else ""
        found = count_re.search(text)
        got = None if found is None else (int(found.group(1).replace(",", "")), int(found.group(2).replace(",", "")))
        if got != (runs, areas):
            info["problems"].append(f"{sheet}: bad-spot count {got}, locked {(runs, areas)}")
    wb.close()
    with zipfile.ZipFile(path) as zf:
        ellipses = sum(
            zf.read(n).decode("utf-8").count('prst="ellipse"')
            for n in zf.namelist()
            if re.fullmatch(r"xl/drawings/drawing\d+\.xml", n)
        )
        info["circle_shapes"] = ellipses
        # One movable circle on each serial bad-spot map. The combined view has none.
        expected_shapes = 0
        for sheet, (runs, areas) in LOGIC_LOCK["counts"].items():
            n = runs + areas
            shown = min(BAD_SPOT_MAX_ZOOMS, n)
            expected_shapes += shown
            images = info["sheets"].get(sheet, {}).get("images", 0)
            if sheet in info["sheets"] and images != 1 + shown:
                info["problems"].append(f"{sheet}: {images} images, expected {1 + shown}")
        if ellipses != expected_shapes:
            info["problems"].append(f"{ellipses} circle shapes, expected {expected_shapes}")
        for item in zf.namelist():
            if not item.startswith("xl/charts/chart"):
                continue
            xml = zf.read(item).decode("utf-8")
            for cat in re.findall(r"<catAx>.*?</catAx>", xml):
                pos = re.search(r'<axPos val="([^"]+)"', cat)
                if not pos or pos.group(1) != "b":
                    info["problems"].append(f"{item}: category axis axPos is {pos.group(1) if pos else 'missing'} (need b)")
                if 'tickLblPos val="none"' in cat:
                    info["problems"].append(f"{item}: category tick labels hidden")
            for val in re.findall(r"<valAx>.*?</valAx>", xml):
                pos = re.search(r'<axPos val="([^"]+)"', val)
                if not pos or pos.group(1) not in ("l", "r"):
                    info["problems"].append(f"{item}: value axis axPos is {pos.group(1) if pos else 'missing'}")
                if 'tickLblPos val="none"' in val:
                    info["problems"].append(f"{item}: value tick labels hidden")
            if "<a:t></a:t>" in xml or "<a:t/>" in xml:
                info["problems"].append(f"{item}: empty text run would blank axis labels")
    return info


def route_length_km(df: pd.DataFrame) -> float:
    """Kilometres along the drive. A step longer than 500 m is a new pass, not road."""
    work = df.dropna(subset=["Time", "Latitude", "Longitude"]).copy()
    if "Time" in work.columns:
        work["Time"] = pd.to_datetime(work["Time"], errors="coerce")
        work = work.dropna(subset=["Time"])
    if len(work) < 2:
        return 0.0
    sort_cols = ["Source", "Time"] if "Source" in work.columns else ["Time"]
    work = work.sort_values(sort_cols)
    groups = work.groupby("Source", sort=False) if "Source" in work.columns else [(None, work)]
    total = 0.0
    for _, part in groups:
        if len(part) < 2:
            continue
        lat = part["Latitude"].to_numpy(dtype=float)
        lon = part["Longitude"].to_numpy(dtype=float)
        step = np.asarray(_equirect_km(lat[:-1], lon[:-1], lat[1:], lon[1:]), dtype=float)
        total += float(step[step < 0.5].sum())
    return total


def write_active_histograms(ws: Worksheet, df: pd.DataFrame) -> dict:
    """Legend-coloured histogram series for the plotted Active rows."""
    rsrp_bins = np.arange(-140, -48, 2)
    rsrq_bins = np.arange(-24, -2.5, 0.5)
    sinr_bins = np.arange(-15, 31, 1)
    return {
        "RSRP": write_stacked_hist(ws, 60, df["RSRP"], rsrp_bins, rsrp_map_class(df["RSRP"]), MAP_RSRP_LABELS),
        "RSRQ": write_stacked_hist(ws, 68, df["RSRQ"], rsrq_bins, rsrq_map_class(df["RSRQ"]), MAP_RSRQ_LABELS),
        "SINR": write_stacked_hist(ws, 76, df["SINR"], sinr_bins, sinr_map_class(df["SINR"]), MAP_SINR_LABELS),
    }


def add_active_cover_extras(cover, data_ws, blocks, route_path: Path, active: pd.DataFrame) -> None:
    """Active histograms and the DT route. Idle charts and the other sheets stay put."""
    write_cell(cover, 44, 12, "Active histograms (plotted Robi rows)", size=14, bold=True)
    cover.merge_cells(start_row=44, start_column=12, end_row=44, end_column=20)
    add_stacked_col_chart(
        data_ws, cover, "L46", "RSRP histogram — Active",
        blocks["RSRP"]["cat_col"], blocks["RSRP"]["data_min"], blocks["RSRP"]["data_max"],
        2, 1 + blocks["RSRP"]["n"],
        "Samples", "RSRP (dBm)", MAP_RSRP_COLORS, width=9, height=6,
    )
    add_stacked_col_chart(
        data_ws, cover, "R46", "RSRQ histogram — Active",
        blocks["RSRQ"]["cat_col"], blocks["RSRQ"]["data_min"], blocks["RSRQ"]["data_max"],
        2, 1 + blocks["RSRQ"]["n"],
        "Samples", "RSRQ (dB)", MAP_RSRQ_COLORS, width=9, height=6,
    )
    add_stacked_col_chart(
        data_ws, cover, "L58", "SINR histogram — Active",
        blocks["SINR"]["cat_col"], blocks["SINR"]["data_min"], blocks["SINR"]["data_max"],
        2, 1 + blocks["SINR"]["n"],
        "Samples", "SINR (dB)", MAP_SINR_COLORS, width=9, height=6,
    )

    section_bar(cover, 72, "DT Route", 1, 20)
    write_cell(cover, 74, 1, "Drive details", size=14, bold=True)
    write_cell(cover, 74, 12, "DT route plot", size=14, bold=True)
    if route_path.exists():
        add_image(cover, route_path, "L76", width=520, height=640)
    ordered = active.dropna(subset=["Time", "Latitude", "Longitude"]).copy()
    ordered["Time"] = pd.to_datetime(ordered["Time"], errors="coerce")
    ordered = ordered.dropna(subset=["Time"]).sort_values("Time")
    start = ordered.iloc[0]
    end = ordered.iloc[-1]
    span = end["Time"] - start["Time"]
    hours = span.total_seconds() / 3600.0
    layers = pd.to_numeric(active["DL EARFCN"], errors="coerce").map(earfcn_to_layer)
    layer_bits = ", ".join(f"{name} {int(n):,}" for name, n in layers.value_counts().items() if name)
    detail_rows = [
        ["Mode", "Active (UE)"],
        ["Campaign", _campaign_label(active["Campaign"]) if "Campaign" in active.columns else "—"],
        ["Operator", ACTIVE_OPERATOR],
        ["Plotted samples", int(len(active))],
        ["Route length (km)", round(route_length_km(active), 1)],
        ["Duration (hours)", round(hours, 1)],
        ["Start time", start["Time"]],
        ["End time", end["Time"]],
        ["Start latitude", round(float(start["Latitude"]), 5)],
        ["Start longitude", round(float(start["Longitude"]), 5)],
        ["End latitude", round(float(end["Latitude"]), 5)],
        ["End longitude", round(float(end["Longitude"]), 5)],
        ["Unique Cell Ids", int(active["Cell Id"].nunique())],
        ["CSV files", int(active["Source"].nunique()) if "Source" in active.columns else "—"],
    ]
    write_table(cover, 75, 1, ["Item", "Value"], detail_rows)
    cover.cell(80, 2).number_format = "#,##0.0"
    cover.cell(81, 2).number_format = "#,##0.0"
    cover.cell(82, 2).number_format = "yyyy-mm-dd hh:mm:ss"
    cover.cell(83, 2).number_format = "yyyy-mm-dd hh:mm:ss"
    cover.merge_cells(start_row=91, start_column=1, end_row=93, end_column=11)
    write_cell(
        cover, 91, 1,
        "The route is the plotted Robi M-PCell track: one band at each latitude and longitude, "
        "the strongest RSRP. RSRQ and SINR on the histograms are the values from that same row. "
        f"Source is {ACTIVE_ARCHIVE.name}. "
        f"Plotted band mix: {layer_bits}.",
        size=9, wrap=True,
    )
    write_cell(
        cover, 19, 1,
        "Cover — Idle dataset and plots on top. Active Mode dataset, histograms and the DT route are below.",
        wrap=True,
    )


def add_cover_charts(cover, data_ws, blocks) -> None:
    write_cell(cover, 5, 12, "RSRP, RSRQ and SINR plots (native Excel charts with legends)", size=14, bold=True)
    cover.merge_cells(start_row=5, start_column=12, end_row=5, end_column=20)
    add_stacked_col_chart(
        data_ws, cover, "L6", "RSRP histogram",
        blocks["RSRP"]["cat_col"], blocks["RSRP"]["data_min"], blocks["RSRP"]["data_max"],
        2, 1 + blocks["RSRP"]["n"],
        "Samples", "RSRP (dBm)", MAP_RSRP_COLORS, width=11, height=7,
    )
    add_stacked_col_chart(
        data_ws, cover, "R6", "RSRQ histogram",
        blocks["RSRQ"]["cat_col"], blocks["RSRQ"]["data_min"], blocks["RSRQ"]["data_max"],
        2, 1 + blocks["RSRQ"]["n"],
        "Samples", "RSRQ (dB)", MAP_RSRQ_COLORS, width=11, height=7,
    )
    add_stacked_col_chart(
        data_ws, cover, "L23", "SINR histogram",
        blocks["SINR"]["cat_col"], blocks["SINR"]["data_min"], blocks["SINR"]["data_max"],
        2, 1 + blocks["SINR"]["n"],
        "Samples", "SINR (dB)", MAP_SINR_COLORS, width=11, height=7,
    )
    add_col_chart(
        data_ws, cover, "R23", "RSRP ranges (dBm)", RSRP_COLOR,
        12, 13, 2, 1 + blocks["rsrp_q_n"],
        "Samples", "RSRP range", width=11, height=7, show_legend=True,
    )


def save_report_preview(df: pd.DataFrame, path: Path) -> Path:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5.2))
    plot_stacked_map_hist(
        axes[0], df["RSRP"], rsrp_map_class, MAP_RSRP_LABELS, MAP_RSRP_COLORS,
        np.arange(-140, -48, 2), "RSRP histogram", "RSRP (dBm)", "RSRP (dBm)", (-140, -50),
    )
    plot_stacked_map_hist(
        axes[1], df["RSRQ"], rsrq_map_class, MAP_RSRQ_LABELS, MAP_RSRQ_COLORS,
        np.arange(-24, -2.5, 0.5), "RSRQ histogram", "RSRQ (dB)", "RSRQ (dB)", (-24, -3),
    )
    plot_stacked_map_hist(
        axes[2], df["SINR"], sinr_map_class, MAP_SINR_LABELS, MAP_SINR_COLORS,
        np.arange(-15, 31, 1), "SINR histogram", "SINR (dB)", "SINR (dB)", (-15, 30),
    )
    for ax in axes:
        ax.grid(False)
        ax.set_facecolor("white")
    fig.suptitle("Bogura Drive-Test Report — RSRP / RSRQ / SINR", fontsize=13, fontweight="bold")
    fig.tight_layout()
    fig.savefig(path, dpi=140, facecolor="white")
    plt.close(fig)
    return path


def _with_layers(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "Band" not in out.columns:
        out["Band"] = out["DL EARFCN"].map(earfcn_to_band)
    if "Layer" not in out.columns:
        out["Layer"] = pd.to_numeric(out["DL EARFCN"], errors="coerce").map(earfcn_to_layer)
    return out


## 19. Build the workbook

Draw every map, then write output/Bogura_DriveTest_Report_v1.63.xlsx.
This step creates the sheets. The solution text and the redrawn pies are the two steps after it.


In [ ]:
# ------------------------------------------------------------------------
# 19. Build the workbook
# Draw every map, then write output/Bogura_DriveTest_Report_v1.63.xlsx.
# This step creates the sheets. The solution text and the redrawn pies are the two steps after it.
# ------------------------------------------------------------------------

def _render_mode(df: pd.DataFrame, tag: str, extent, mode_label: str | None) -> dict:
    """Coverage maps, bad-spot maps and spot tables for one mode.

    tag is "" for Idle (existing file names) and "active" for the Active Mode copies.
    Idle rows that share a latitude, longitude and timestamp collapse to the
    strongest RSRP, then the strongest band in each scanner sweep is kept.
    Active auto mode keeps one Cell Id and one band per latitude and longitude,
    the strongest RSRP, and RSRQ and SINR stay on that row.
    """
    suffix = f"_{tag}" if tag else ""
    if tag == "active":
        base = strongest_location(df)
        map_df = base
    else:
        base = strongest_fix(df)
        map_df = best_server(base)
    plot_rsrp_coverage_map(map_df, PLOTS_DIR / f"excel_route_rsrp{suffix}.png", extent=extent, mode_label=mode_label)
    plot_rsrq_coverage_map(map_df, PLOTS_DIR / f"excel_route_rsrq{suffix}.png", extent=extent, mode_label=mode_label)
    plot_sinr_coverage_map(map_df, PLOTS_DIR / f"excel_route_sinr{suffix}.png", extent=extent, mode_label=mode_label)
    rsrp_band: dict[str, Path] = {}
    rsrq_band: dict[str, Path] = {}
    sinr_band: dict[str, Path] = {}
    for layer in REPORT_LAYERS:
        part = filter_by_layer(base, layer)
        if tag != "active":
            part = best_server(part)
        rsrp_band[layer] = plot_rsrp_coverage_map(
            part, PLOTS_DIR / f"excel_route_rsrp_{layer}{suffix}.png",
            layer=layer, extent=extent, mode_label=mode_label,
        )
        rsrq_band[layer] = plot_rsrq_coverage_map(
            part, PLOTS_DIR / f"excel_route_rsrq_{layer}{suffix}.png",
            layer=layer, extent=extent, mode_label=mode_label,
        )
        sinr_band[layer] = plot_sinr_coverage_map(
            part, PLOTS_DIR / f"excel_route_sinr_{layer}{suffix}.png",
            layer=layer, extent=extent, mode_label=mode_label,
        )
    rsrp_spots = find_bad_spots(map_df, "RSRP")
    rsrq_spots = find_bad_spots(map_df, "RSRQ")
    sinr_spots = find_bad_spots(map_df, "SINR")
    title_bit = f" — {mode_label}" if mode_label else ""
    bad_maps = {}
    zoom_maps = {}
    for kpi, plot_map, classify, labels, colors in (
        ("RSRP", plot_rsrp_bad_spot_map, rsrp_map_class, MAP_RSRP_LABELS, MAP_RSRP_COLORS),
        ("RSRQ", plot_rsrq_bad_spot_map, rsrq_map_class, MAP_RSRQ_LABELS, MAP_RSRQ_COLORS),
        ("SINR", plot_sinr_bad_spot_map, sinr_map_class, MAP_SINR_LABELS, MAP_SINR_COLORS),
    ):
        key = kpi.lower()
        spots = {"RSRP": rsrp_spots, "RSRQ": rsrq_spots, "SINR": sinr_spots}[kpi]
        total = int(spots.attrs.get("total_consecutive", len(spots))) if spots is not None else 0
        discrete = int(spots.attrs.get("total_discrete", 0)) if spots is not None else 0
        print(f"{kpi} {tag or 'idle'}: {total} poor runs, {discrete} discrete", flush=True)
        extra = {}
        if kpi != "SINR":
            mode_name = "Active" if tag == "active" else "Idle"
            extra["legend_title"] = f"{mode_name} by {kpi}"
        bad_maps[kpi] = plot_map(
            map_df, spots, PLOTS_DIR / f"excel_bad_spots_{key}{suffix}.png",
            extent=extent, title=f"Combined view — {kpi}{title_bit}", shapes=False, outlines=False,
            **extra,
        )
        zoom_maps[kpi] = plot_bad_spot_serial_maps(
            map_df, spots, PLOTS_DIR, f"excel_bad_spots_{key}_spot{suffix}",
            kpi, classify(map_df[kpi]), labels, colors,
            basemap_style="satellite",
            route_casing=True,
        )
    rsrp_bad, rsrq_bad, sinr_bad = bad_maps["RSRP"], bad_maps["RSRQ"], bad_maps["SINR"]
    rsrp_zooms, rsrq_zooms, sinr_zooms = zoom_maps["RSRP"], zoom_maps["RSRQ"], zoom_maps["SINR"]
    return {
        "map_df": map_df,
        "rsrp_band": rsrp_band, "rsrq_band": rsrq_band, "sinr_band": sinr_band,
        "rsrp_spots": rsrp_spots, "rsrq_spots": rsrq_spots, "sinr_spots": sinr_spots,
        "rsrp_bad": rsrp_bad, "rsrq_bad": rsrq_bad, "sinr_bad": sinr_bad,
        "rsrp_zooms": rsrp_zooms, "rsrq_zooms": rsrq_zooms, "sinr_zooms": sinr_zooms,
    }


def build_report(df: pd.DataFrame, active: pd.DataFrame | None = None, out_path: Path = REPORT_XLSX) -> Path:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    PLOTS_DIR.mkdir(parents=True, exist_ok=True)
    df = _with_layers(df)
    active_raw_n = 0
    active_robi = None
    if active is not None and not active.empty:
        active = _with_layers(active)
        active_raw_n = len(active)
        active_robi = select_active_report(active)
        print(
            f"Active Mode {ACTIVE_OPERATOR} {ACTIVE_CELL_TYPE}: "
            f"{len(active_robi):,} rows of {active_raw_n:,}"
        )

    idle_fixed = strongest_fix(df)
    print(f"Idle strongest row per lat/lon/time: {len(idle_fixed):,} of {len(df):,}", flush=True)
    idle_track = best_server(idle_fixed)
    extent = coverage_extent(idle_track)
    print(f"Idle extent {tuple(round(v, 5) for v in extent)}", flush=True)
    idle = _render_mode(df, "", extent, None)
    active_view = None
    active_plotted = None
    if active_robi is not None and not active_robi.empty:
        active_plotted = strongest_location(active_robi)
        print(
            f"Active plotted: {len(active_plotted):,} rows, one band per lat/lon",
            flush=True,
        )
        active_extent = coverage_extent(active_plotted)
        print(f"Active extent {tuple(round(v, 5) for v in active_extent)}", flush=True)
        active_view = _render_mode(active_robi, "active", active_extent, "Active")

    scatter_sinr = save_rsrp_scatter(idle_fixed, "SINR", PLOTS_DIR / "excel_scatter_rsrp_sinr.png", "SINR (dB)", "RSRP vs SINR", ylim=(-10, 32))
    scatter_rsrq = save_rsrp_scatter(idle_fixed, "RSRQ", PLOTS_DIR / "excel_scatter_rsrp_rsrq.png", "RSRQ (dB)", "RSRP vs RSRQ", ylim=(-24, 0))
    shutil.copy2(idle["rsrp_bad"], PLOTS_DIR / "08_bad_spots_rsrp.png")
    shutil.copy2(idle["rsrq_bad"], PLOTS_DIR / "08_bad_spots_rsrq.png")
    shutil.copy2(idle["sinr_bad"], PLOTS_DIR / "08_bad_spots_sinr.png")
    save_binned_vs_preview(idle_fixed, PLOTS_DIR / "excel_rsrp_vs_sinr_lines.png")
    save_report_preview(df, PLOTS_DIR / "excel_rsrp_rsrq_sinr_histograms.png")
    if active_view is not None:
        save_rsrp_scatter(
            active_plotted, "SINR", PLOTS_DIR / "excel_scatter_rsrp_sinr_active.png",
            "SINR (dB)", "RSRP vs SINR — Active", ylim=(-10, 32),
        )
        save_rsrp_scatter(
            active_plotted, "RSRQ", PLOTS_DIR / "excel_scatter_rsrp_rsrq_active.png",
            "RSRQ (dB)", "RSRP vs RSRQ — Active", ylim=(-24, 0),
        )

    wb = Workbook()
    cover = wb.active
    cover.title = "Cover"
    rsrp_ws = wb.create_sheet("RSRP")
    rsrq_ws = wb.create_sheet("RSRQ")
    sinr_ws = wb.create_sheet("SINR")
    bad_rsrp_ws = wb.create_sheet("Bad Spot RSRP")
    bad_rsrp_a = wb.create_sheet("Bad Spot RSRP_Active")
    time_ws = wb.create_sheet("RSRP vs KPIs")
    time_active_ws = wb.create_sheet("RSRP vs KPIs_Active")
    sample_ws = wb.create_sheet("Sample Log")
    data_ws = wb.create_sheet("_ChartData")
    data_ws.sheet_state = "hidden"

    blocks = write_chart_data(data_ws, df)
    blocks["vs_n"] = write_vs_block(data_ws, idle_fixed, 50, 51, 52)
    active_vs_n = blocks["vs_n"]
    if active_view is not None:
        active_vs_n = write_vs_block(data_ws, active_plotted, 54, 55, 56)
    build_cover(
        cover, df, active_plotted, active_raw_n,
        0 if active_robi is None else len(active_robi),
    )
    add_cover_charts(cover, data_ws, blocks)
    if active_plotted is not None and not active_plotted.empty:
        route_path = plot_road_map(
            active_plotted, PLOTS_DIR / "excel_cover_active_route.png",
        )
        add_active_cover_extras(
            cover, data_ws, write_active_histograms(data_ws, active_plotted), route_path, active_plotted,
        )
    empty_bands: dict[str, Path] = {}
    build_kpi_sheet(
        rsrp_ws, "RSRP", "dBm", idle["rsrp_bad"],
        None if active_view is None else active_view["rsrp_bad"],
        idle["rsrp_band"], empty_bands if active_view is None else active_view["rsrp_band"],
    )
    build_kpi_sheet(
        rsrq_ws, "RSRQ", "dB", idle["rsrq_bad"],
        None if active_view is None else active_view["rsrq_bad"],
        idle["rsrq_band"], empty_bands if active_view is None else active_view["rsrq_band"],
    )
    build_kpi_sheet(
        sinr_ws, "SINR", "dB", idle["sinr_bad"],
        None if active_view is None else active_view["sinr_bad"],
        idle["sinr_band"], empty_bands if active_view is None else active_view["sinr_band"],
    )
    build_kpi_bad_spot_sheet(
        bad_rsrp_ws, "RSRP", idle["rsrp_spots"], idle["rsrp_bad"], idle["rsrp_zooms"],
        samples=idle["map_df"],
    )
    if active_view is not None:
        build_kpi_bad_spot_sheet(
            bad_rsrp_a, "RSRP", active_view["rsrp_spots"], active_view["rsrp_bad"], active_view["rsrp_zooms"],
            mode="Active Mode", samples=active_view["map_df"],
        )
        build_rsrp_vs_sheet(
            time_active_ws, data_ws, active_vs_n,
            PLOTS_DIR / "excel_scatter_rsrp_sinr_active.png",
            PLOTS_DIR / "excel_scatter_rsrp_rsrq_active.png",
            title="  RSRP vs KPIs Active Mode",
            subtitle=f"  {ACTIVE_OPERATOR} UE  |  RSRQ and SINR from the strongest RSRP row  |  Gridlines off",
            cat_col=54, sinr_col=55, rsrq_col=56,
        )
    build_rsrp_vs_sheet(
        time_ws, data_ws, blocks["vs_n"], scatter_sinr, scatter_rsrq,
        subtitle="  Idle V2.0 scanner  |  RSRQ and SINR from the strongest RSRP row  |  Gridlines off",
    )
    build_sample_log(sample_ws, df)

    for ws in wb.worksheets:
        apply_sheet_view(ws)

    wb.properties.title = "Bogura Drive-Test Report IDLE and Active Mode"
    wb.properties.subject = "IDLE and Active Mode"
    wb.properties.description = (
        "Bogura drive-test report. Idle scanner on the left and on Bad Spot RSRP/RSRQ/SINR. "
        "Active Mode Robi UE on the right and on Bad Spot *_Active and RSRP vs KPIs_Active. "
        "Poor runs use the same thresholds in both modes."
    )
    out_path.parent.mkdir(parents=True, exist_ok=True)
    wb.save(out_path)
    n_svg = embed_svg_layers(out_path)
    n_circles = embed_circle_shapes(out_path)
    n_jpg = compress_workbook_images(out_path)
    print(f"Embedded {n_svg} SVG layers and {n_circles} separate bad-spot circles; compressed {n_jpg} maps")
    shutil.copy2(out_path, VERSIONED_XLSX)
    print(f"Wrote Excel report: {out_path}")
    print(f"Wrote versioned copy: {VERSIONED_XLSX}")
    return out_path


_AIM_SENTENCE_OLD = (
    "The label on each arm is the Sector name. A yellow line shows the distance only from a sector whose azimuth "
    "points at that bad spot. A sector that faces another way has no line."
)
_AIM_SENTENCE_NEW = (
    "The label on each arm is the Sector name. A yellow dashed line starts at the end of the serving sector, "
    "not at the middle of the site, and runs to each red sample inside the orange circle. That cell is the Cell Id "
    "on the sample, on L900, L1800, L2100 or L2600, and it has to be in System DUMP.xlsb. Every serving cell is drawn. "
    "The length beside the red is the full red stretch, and a short gap of good coverage stays in that length."
)
_AIM_SENTENCE_CELL = (
    "The label on each arm is the Sector name. A yellow line runs from the serving cell to each red sample inside "
    "the orange circle. That cell is the Cell Id on the sample, on L900, L1800, L2100 or L2600, and it has to be in "
    "System DUMP.xlsb. The length of each red run is written beside it in metres."
)
_PIE_SENTENCE_OLD = (
    "Each bad-spot map draws a sector pie for every nearby site from System DUMP.xlsb. "
    "The label is the sector name. Where a sector shoots toward that bad spot, a yellow line shows the distance."
)
_PIE_SENTENCE_NEW = (
    "Each bad-spot map draws a transparent cyan outline for every sector of a site that falls in that view, "
    "from System DUMP.xlsb. No sector of that site is left out, and the outline is not filled. "
    "Where a sector shoots toward that bad spot, a yellow line shows the distance."
)


_BAD_SPOT_STRIDE = 41


## 20. Replace map pictures without dropping the circles

refresh_bad_spot_pies redraws all 125 bad-spot maps and puts the JPEGs back into the workbook.
refresh_sheet_maps does the same for the coverage maps and the DT route. Run these one at a time, never in the same save as the solution text.


In [ ]:
# ------------------------------------------------------------------------
# 20. Replace map pictures without dropping the circles
# refresh_bad_spot_pies redraws all 125 bad-spot maps and puts the JPEGs back into the workbook.
# refresh_sheet_maps does the same for the coverage maps and the DT route. Run these one at a time, never in the same save as the solution text.
# ------------------------------------------------------------------------

def _sheet_part_path(parts: dict[str, bytes], sheet_name: str) -> str:
    main = "{http://schemas.openxmlformats.org/spreadsheetml/2006/main}"
    rel_ns = "{http://schemas.openxmlformats.org/officeDocument/2006/relationships}"
    wb = ET.fromstring(parts["xl/workbook.xml"])
    book_rels = ET.fromstring(parts["xl/_rels/workbook.xml.rels"])
    rid_target = {rel.attrib["Id"]: rel.attrib["Target"].lstrip("/") for rel in book_rels}
    for node in wb.find(f"{main}sheets"):
        if node.attrib["name"] == sheet_name:
            return rid_target[node.attrib[f"{rel_ns}id"]]
    raise SystemExit(f"{sheet_name} was not found")


def _append_missing_bad_spot_maps(
    parts: dict[str, bytes],
    sheet_name: str,
    drawing: str,
    rels: str,
    paths: list[Path],
    plans: list[dict],
    first_number: int,
) -> tuple[str, str]:
    """Add one map, circle and suggestion table for each bad spot past the ones already drawn.

    The maps already in the sheet stay on their rows. The summary and the identified
    table move down to make room.
    """
    if len(paths) != len(plans):
        raise SystemExit(f"{sheet_name}: {len(paths)} new maps and {len(plans)} solutions")
    if not paths:
        return drawing, rels
    sheet_path = _sheet_part_path(parts, sheet_name)
    xml = parts[sheet_path].decode("utf-8")
    found = re.search(
        r'<row r="(\d+)"[^>]*>(?:(?!</row>).)*Summary of proposed solutions',
        xml,
        flags=re.S,
    )
    if found is None:
        raise SystemExit(f"{sheet_name}: summary row was not found, so the extra maps have nowhere to go")
    start = int(found.group(1))
    xml = _shift_rows_down(xml, start, len(paths) * _BAD_SPOT_STRIDE)
    rows: dict[int, str] = {}
    merges: list[str] = []
    anchors: list[str] = []
    next_shape = max((int(i) for i in re.findall(r'<cNvPr id="(\d+)"', drawing)), default=0) + 1
    next_rid = max((int(i) for i in re.findall(r'Id="rId(\d+)"', rels)), default=0) + 1
    media_numbers = [int(i) for i in re.findall(r"xl/media/image(\d+)\.jpg", "\n".join(parts))]
    next_media = max(media_numbers, default=0) + 1
    for offset, (png, plan) in enumerate(zip(paths, plans)):
        header = start + offset * _BAD_SPOT_STRIDE
        number = first_number + offset
        bar = [_inline_cell(f"A{header}", 5, f"Bad Spot {number}")]
        for col in range(2, 9):
            bar.append(f'<c r="{get_column_letter(col)}{header}" s="5" t="n" />')
        rows[header] = f'<row r="{header}" ht="22" customHeight="1">{"".join(bar)}</row>'
        merges.append(f"A{header}:H{header}")
        advice_rows, advice_merges = _advice_sheet_rows(header, plan)
        rows.update(advice_rows)
        merges.extend(advice_merges)
        media = f"xl/media/image{next_media}.jpg"
        next_media += 1
        parts[media] = _jpeg_bytes(png)
        rid = f"rId{next_rid}"
        next_rid += 1
        rels = rels.replace(
            "</Relationships>",
            f'<Relationship Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/image" '
            f'Target="/{media}" Id="{rid}" /></Relationships>',
            1,
        )
        drawing_row = header + 1
        anchors.append(
            f'<oneCellAnchor><from><col>0</col><colOff>0</colOff><row>{drawing_row}</row><rowOff>0</rowOff></from>'
            f'<ext cx="7286625" cy="6858000" /><pic><nvPicPr><cNvPr id="{next_shape}" name="Image {next_shape}" '
            f'descr="Picture" /><cNvPicPr /></nvPicPr><blipFill><a:blip cstate="print" r:embed="{rid}" />'
            f'<a:stretch><a:fillRect /></a:stretch></blipFill><spPr><a:prstGeom prst="rect" /></spPr></pic>'
            f'<clientData /></oneCellAnchor>'
        )
        geom = MAP_CIRCLE_PIXELS.get(str(Path(png).resolve()))
        if geom and geom.get("circles"):
            img_w, img_h = geom["size"]
            for circle in geom["circles"]:
                anchors.append(_circle_anchor(circle, next_shape + 1, 0, 0, drawing_row, 0, 7286625, 6858000, img_w, img_h))
                next_shape += 1
        next_shape += 1
    xml = _insert_sheet_rows(xml, rows, merges)
    parts[sheet_path] = xml.encode("utf-8")
    drawing = drawing.replace("</wsDr>", "".join(anchors) + "</wsDr>")
    print(f"  {sheet_name}: added {len(paths)} maps", flush=True)
    return drawing, rels


def refresh_bad_spot_pies(src: Path | None = None) -> Path:
    """Redraw the serial bad-spot maps and swap them into the current workbook.

    Combined views, KPI sheets, and the circle count stay as they are. Each serial
    map keeps one movable circle, placed again from the new picture. Spots that
    were not drawn before get a new map below the last one.
    """
    import io
    from PIL import Image


    src = src or REPORT_XLSX
    if not src.exists():
        raise SystemExit(f"Missing workbook: {src}")
    print("Loading drive logs", flush=True)
    idle = load_merged_csv() if MERGED_CSV.exists() else None
    active = load_active_merged() if ACTIVE_MERGED.exists() else None
    if idle is None or active is None:
        raise SystemExit("Merged drive logs are missing; a full report build is required.")
    active = select_active_report(active)
    print(f"Idle rows {len(idle):,}  Active {ACTIVE_OPERATOR} {ACTIVE_CELL_TYPE} rows {len(active):,}", flush=True)
    idle_track = best_server(idle)
    active_track = best_carrier(active)
    MAP_CIRCLE_PIXELS.clear()
    zoom_for_sheet: dict[str, list[Path]] = {}
    plans_for_sheet: dict[str, list[dict]] = {}
    for tag, frame in (("", idle_track), ("active", active_track)):
        suffix = f"_{tag}" if tag else ""
        for kpi, classify, labels, colors in (
            ("RSRP", rsrp_map_class, MAP_RSRP_LABELS, MAP_RSRP_COLORS),
        ):
            print(f"Drawing {kpi} bad spots ({tag or 'idle'})", flush=True)
            spots = find_bad_spots(frame, kpi)
            paths = plot_bad_spot_serial_maps(
                frame, spots, PLOTS_DIR, f"excel_bad_spots_{kpi.lower()}_spot{suffix}",
                kpi, classify(frame[kpi]), labels, colors,
                basemap_style="satellite",
                route_casing=True,
            )
            sheet = f"Bad Spot {kpi}" + ("_Active" if tag else "")
            zoom_for_sheet[sheet] = paths
            shown = spots.head(len(paths))
            plans_for_sheet[sheet] = [
                bad_spot_advice(frame, kpi, shown.iloc[i]) for i in range(len(shown))
            ]
            print(f"  {sheet}: {len(paths)} maps", flush=True)

    print("Updating the workbook", flush=True)
    with zipfile.ZipFile(src) as zf:
        names = zf.namelist()
        parts = {name: zf.read(name) for name in names}
    drawings = _sheet_drawings(parts)
    sheet_xml = _worksheet_xml(parts)
    for sheet, paths in zoom_for_sheet.items():
        drawing_name = drawings[sheet]
        rels_name = drawing_name.replace("drawings/", "drawings/_rels/") + ".rels"
        rels = parts[rels_name].decode("utf-8")
        drawing = parts[drawing_name].decode("utf-8")
        rid_media = {}
        for rel in re.findall(r"<Relationship\b[^>]*/>", rels):
            rid = re.search(r'Id="([^"]+)"', rel).group(1)
            target = re.search(r'Target="([^"]+)"', rel).group(1)
            media = target.lstrip("/") if target.startswith("/") else "xl/media/" + target.rsplit("/", 1)[1]
            rid_media[rid] = media
        pictures = []
        for block in re.findall(r"<oneCellAnchor>.*?</oneCellAnchor>", drawing, flags=re.S):
            if "<pic>" not in block:
                continue
            row = int(re.search(r"<row>(\d+)</row>", block).group(1))
            pictures.append((row, block))
        pictures.sort()
        serial = pictures[1:]
        if len(paths) < len(serial):
            raise SystemExit(f"{sheet}: {len(serial)} pictures, {len(paths)} new maps")
        drawing = re.sub(
            r'<oneCellAnchor>(?:(?!</oneCellAnchor>).)*prst="ellipse".*?</oneCellAnchor>',
            "",
            drawing,
            flags=re.S,
        )
        next_shape = max((int(i) for i in re.findall(r'<cNvPr id="(\d+)"', drawing)), default=0) + 1
        extras = []
        for (_row, block), png in zip(serial, paths):
            embed = re.search(r'r:embed="([^"]+)"', block).group(1)
            media = rid_media[embed]
            image = Image.open(png).convert("RGB")
            buf = io.BytesIO()
            image.save(buf, "JPEG", quality=74, optimize=True)
            parts[media] = buf.getvalue()
            geom = MAP_CIRCLE_PIXELS[str(Path(png).resolve())]
            img_w, img_h = geom["size"]
            origin = re.search(
                r"<from><col>(\d+)</col><colOff>(\d+)</colOff><row>(\d+)</row><rowOff>(\d+)</rowOff></from><ext cx=\"(\d+)\" cy=\"(\d+)\"",
                block,
            )
            col, col_off, row, row_off, pic_cx, pic_cy = (int(v) for v in origin.groups())
            for circle in geom["circles"]:
                extras.append(_circle_anchor(circle, next_shape, col, col_off, row, row_off, pic_cx, pic_cy, img_w, img_h))
                next_shape += 1
        drawing = drawing.replace("</wsDr>", "".join(extras) + "</wsDr>")
        extra_paths = paths[len(serial):]
        if extra_paths:
            drawing, rels = _append_missing_bad_spot_maps(
                parts,
                sheet,
                drawing,
                rels,
                extra_paths,
                plans_for_sheet[sheet][len(serial):],
                len(serial) + 1,
            )
            parts[rels_name] = rels.encode("utf-8")
        parts[drawing_name] = drawing.encode("utf-8")
        print(f"  {sheet}: replaced {len(serial)} maps, added {len(extra_paths)}, {len(extras)} circles on the first maps", flush=True)

    note_hits = 0
    for name, body in list(parts.items()):
        if not name.endswith(".xml"):
            continue
        text = body.decode("utf-8")
        replaced = text
        if _PIE_SENTENCE_OLD in replaced:
            note_hits += replaced.count(_PIE_SENTENCE_OLD)
            replaced = replaced.replace(_PIE_SENTENCE_OLD, _PIE_SENTENCE_NEW)
        if _AIM_SENTENCE_OLD in replaced:
            note_hits += replaced.count(_AIM_SENTENCE_OLD)
            replaced = replaced.replace(_AIM_SENTENCE_OLD, _AIM_SENTENCE_NEW)
        if _AIM_SENTENCE_CELL in replaced:
            note_hits += replaced.count(_AIM_SENTENCE_CELL)
            replaced = replaced.replace(_AIM_SENTENCE_CELL, _AIM_SENTENCE_NEW)
        if replaced == text:
            continue
        parts[name] = replaced.encode("utf-8")
    print(f"Updated the pie note in {note_hits} place(s)", flush=True)

    tmp = src.with_suffix(".tmp.xlsx")
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as zf:
        for name in names:
            zf.writestr(name, parts[name])
        for name, body in parts.items():
            if name not in names:
                zf.writestr(name, body)
    tmp.replace(VERSIONED_XLSX)
    spread_circle_anchors(VERSIONED_XLSX)
    if VERSIONED_XLSX.resolve() != REPORT_XLSX.resolve():
        shutil.copy2(VERSIONED_XLSX, REPORT_XLSX)
    print(f"Wrote {VERSIONED_XLSX}", flush=True)
    return VERSIONED_XLSX


def _jpeg_bytes(path: Path) -> bytes:
    import io
    from PIL import Image

    image = Image.open(path).convert("RGB")
    buf = io.BytesIO()
    image.save(buf, "JPEG", quality=74, optimize=True)
    return buf.getvalue()


def _sheet_pictures(parts: dict[str, bytes], sheet_name: str) -> list[tuple[int, int, str]]:
    """Pictures on a sheet as (row, col, media path), top to bottom and left to right."""
    drawings = _sheet_drawings(parts)
    drawing_name = drawings[sheet_name]
    rels_name = drawing_name.replace("drawings/", "drawings/_rels/") + ".rels"
    rels = parts[rels_name].decode("utf-8")
    rid_media = {}
    for rel in re.findall(r"<Relationship\b[^>]*/>", rels):
        rid = re.search(r'Id="([^"]+)"', rel).group(1)
        target = re.search(r'Target="([^"]+)"', rel).group(1)
        if target.startswith("/"):
            media = target.lstrip("/")
        elif target.startswith("../"):
            media = "xl/" + target[3:]
        else:
            media = "xl/media/" + target.rsplit("/", 1)[-1]
        rid_media[rid] = media
    drawing = parts[drawing_name].decode("utf-8")
    pics = []
    for block in re.findall(r"<oneCellAnchor>.*?</oneCellAnchor>", drawing, flags=re.S):
        if "<pic>" not in block:
            continue
        row = int(re.search(r"<row>(\d+)</row>", block).group(1))
        col = int(re.search(r"<col>(\d+)</col>", block).group(1))
        embed = re.search(r'r:embed="([^"]+)"', block).group(1)
        pics.append((row, col, rid_media[embed]))
    pics.sort()
    return pics


def refresh_sheet_maps(src: Path | None = None) -> Path:
    """Redraw coverage maps, combined views and the DT route with a scale bar.

    Serial bad-spot maps are left for refresh-pies. Pictures are swapped in the
    zip so the movable circles stay.
    """

    src = src or REPORT_XLSX
    if not src.exists():
        raise SystemExit(f"Missing workbook: {src}")
    print("Loading drive logs", flush=True)
    idle_raw = load_merged_csv() if MERGED_CSV.exists() else None
    active_raw = load_active_merged() if ACTIVE_MERGED.exists() else None
    if idle_raw is None or active_raw is None:
        raise SystemExit("Merged drive logs are missing.")
    idle_raw = _with_layers(idle_raw)
    active_raw = select_active_report(_with_layers(active_raw))
    idle_fixed = strongest_fix(idle_raw)
    idle_track = best_server(idle_fixed)
    active_track = strongest_location(active_raw)
    idle_extent = coverage_extent(idle_track)
    active_extent = coverage_extent(active_track)
    sheet_images: dict[str, list[Path]] = {}
    combined: dict[tuple[str, str], Path] = {}
    for kpi, plot_cov, plot_bad, labels in (
        ("RSRP", plot_rsrp_coverage_map, plot_rsrp_bad_spot_map, MAP_RSRP_LABELS),
        ("RSRQ", plot_rsrq_coverage_map, plot_rsrq_bad_spot_map, MAP_RSRQ_LABELS),
        ("SINR", plot_sinr_coverage_map, plot_sinr_bad_spot_map, MAP_SINR_LABELS),
    ):
        del labels
        ordered: list[Path] = []
        for tag, base, track, extent, mode_label in (
            ("", idle_fixed, idle_track, idle_extent, None),
            ("active", active_raw, active_track, active_extent, "Active"),
        ):
            suffix = f"_{tag}" if tag else ""
            print(f"Drawing {kpi} combined ({tag or 'idle'})", flush=True)
            spots = find_bad_spots(track, kpi)
            mode_name = "Active" if tag == "active" else "Idle"
            title = f"Combined view — {kpi}" + (f" — {mode_label}" if mode_label else "")
            bad_kwargs = {"extent": extent, "title": title, "shapes": False, "outlines": False}
            if kpi != "SINR":
                bad_kwargs["legend_title"] = f"{mode_name} by {kpi}"
            path = plot_bad(
                track, spots, PLOTS_DIR / f"excel_bad_spots_{kpi.lower()}{suffix}.png",
                **bad_kwargs,
            )
            combined[(kpi, tag)] = path
            ordered.append(path)
            band_paths = []
            for layer in REPORT_LAYERS:
                print(f"Drawing {kpi} {layer} ({tag or 'idle'})", flush=True)
                part = filter_by_layer(base, layer)
                if tag != "active":
                    part = best_server(part)
                band_paths.append(plot_cov(
                    part, PLOTS_DIR / f"excel_route_{kpi.lower()}_{layer}{suffix}.png",
                    layer=layer, extent=extent, mode_label=mode_label,
                ))
            if tag == "":
                idle_bands = band_paths
            else:
                for left, right in zip(idle_bands, band_paths):
                    ordered.append(left)
                    ordered.append(right)
        sheet_images[kpi] = ordered
        print(f"  {kpi}: {len(ordered)} maps", flush=True)

    print("Drawing the DT route", flush=True)
    route = plot_road_map(active_track, PLOTS_DIR / "excel_cover_active_route.png")

    print("Updating the workbook", flush=True)
    with zipfile.ZipFile(src) as zf:
        names = zf.namelist()
        parts = {name: zf.read(name) for name in names}
    for sheet, paths in sheet_images.items():
        pics = _sheet_pictures(parts, sheet)
        if len(pics) != len(paths):
            raise SystemExit(f"{sheet}: {len(pics)} pictures, {len(paths)} new maps")
        for (_row, _col, media), path in zip(pics, paths):
            parts[media] = _jpeg_bytes(path)
        print(f"  {sheet}: replaced {len(paths)} maps", flush=True)
    for sheet, key in (("Bad Spot RSRP", ("RSRP", "")), ("Bad Spot RSRP_Active", ("RSRP", "active"))):
        pics = _sheet_pictures(parts, sheet)
        if not pics:
            raise SystemExit(f"{sheet}: no pictures")
        _row, _col, media = pics[0]
        parts[media] = _jpeg_bytes(combined[key])
        print(f"  {sheet}: replaced the combined view", flush=True)
    cover_pics = _sheet_pictures(parts, "Cover")
    route_pics = [pic for pic in cover_pics if pic[0] >= 70]
    if len(route_pics) != 1:
        raise SystemExit(f"Cover: expected one DT route picture, found {len(cover_pics)}")
    parts[route_pics[0][2]] = _jpeg_bytes(route)
    print("  Cover: replaced the DT route", flush=True)

    tmp = src.with_suffix(".tmp.xlsx")
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as zf:
        for name in names:
            zf.writestr(name, parts[name])
    tmp.replace(VERSIONED_XLSX)
    if VERSIONED_XLSX.resolve() != REPORT_XLSX.resolve():
        shutil.copy2(VERSIONED_XLSX, REPORT_XLSX)
    print(f"Wrote {VERSIONED_XLSX}", flush=True)
    return VERSIONED_XLSX


_ADVICE_NOTE_OLD = (
    "The columns to the right of each map are a suggestion table. Azimuth and tilt are used when a sector is within about 2.5 km, and downtilt is limited to 2°. A larger downtilt becomes a 4th L900 sector on a site within 1 km. A 4th sector also replaces a new site when a host inside 4 km can aim at the bad spot. A new 3-sector site is kept only when the nearest site is farther than that. A blank with no plot inside the circle is counted with that bad spot.",
    "The space to the right of each map is that suggestion table. Site, Population and Final solution are filled for every bad spot in the table below. A blank with no plot inside the circle is counted with that bad spot.",
)
_ADVICE_NOTE_NEW = (
    "The columns to the right of each map are a suggestion table. Azimuth and tilt are used when a sector is within about 2.5 km, and downtilt is limited to 2°. A larger downtilt becomes a 4th L900 sector on a site within 1 km. A 4th sector also replaces a new site when a host inside 3 km can aim at the bad spot. A new 3-sector site is kept only when the nearest site is farther than that. A blank with no plot inside the circle is counted with that bad spot.",
    "The space to the right of each map is that suggestion table. Site, Population and Final solution are filled for every bad spot in the table below. A blank with no plot inside the circle is counted with that bad spot.",
)
_DROP_BAD_SPOT_SHEETS = (
    "Bad Spot RSRQ",
    "Bad Spot SINR",
    "Bad Spot RSRQ_Active",
    "Bad Spot SINR_Active",
)


## 21. Write the solution, the cover list, and EP Details

advise writes the findings beside each map, the Site / Population / Final solution columns, and the Cover list.
Bad Spot Summary across Mode sits to the right of the DT route. EP Details is the height, azimuth, E_tilt, and M_tilt tracker. A new sector leaves the existing plan blank.


In [ ]:
# ------------------------------------------------------------------------
# 21. Write the solution, the cover list, and EP Details
# advise writes the findings beside each map, the Site / Population / Final solution columns, and the Cover list.
# Bad Spot Summary across Mode sits to the right of the DT route. EP Details is the height, azimuth, E_tilt, and M_tilt tracker. A new sector leaves the existing plan blank.
# ------------------------------------------------------------------------

def _xml_text(value: str) -> str:
    return (
        value.replace("&", "&amp;")
        .replace("<", "&lt;")
        .replace(">", "&gt;")
    )


def _advice_sheet_rows(header: int, plan: dict) -> tuple[dict[int, str], list[str]]:
    """Inline cells for the suggestion table beside one bad-spot map."""
    title = header + 2
    find = header + 4
    head = header + 10

    def cell(ref: str, style: int, text: str | None = None) -> str:
        if text is None:
            return f'<c r="{ref}" s="{style}"/>'
        return (
            f'<c r="{ref}" s="{style}" t="inlineStr"><is>'
            f'<t xml:space="preserve">{_xml_text(text)}</t></is></c>'
        )

    def styled_row(row: int, cells: list[str]) -> str:
        return f'<row r="{row}">{"".join(cells)}</row>'

    rows = {
        title: styled_row(title, [cell(f"I{title}", 32, "Suggestions and solutions")] + [
            cell(f"{get_column_letter(col)}{title}", 32) for col in range(10, 17)
        ]),
        find: styled_row(find, [
            cell(f"I{find}", 36, "Findings"),
            cell(f"J{find}", 37, plan.get("finding", "")),
        ]),
        head: styled_row(head, [
            cell(f"I{head}", 35, "Step"),
            cell(f"J{head}", 35, "Action"),
            cell(f"K{head}", 35),
            cell(f"L{head}", 35, "Proposal"),
            cell(f"M{head}", 35),
            cell(f"N{head}", 35),
            cell(f"O{head}", 35),
            cell(f"P{head}", 35),
        ]),
    }
    merges = [
        f"I{title}:P{title}",
        f"I{find}:I{find + 4}",
        f"J{find}:P{find + 4}",
        f"J{head}:K{head}",
        f"L{head}:P{head}",
    ]
    for index, step in enumerate(plan.get("steps") or []):
        row = head + 1 + index * 4
        rows[row] = styled_row(row, [
            cell(f"I{row}", 36, str(step.get("step", ""))),
            cell(f"J{row}", 36, str(step.get("action", ""))),
            cell(f"L{row}", 37, str(step.get("proposal", ""))),
        ])
        merges.extend((
            f"I{row}:I{row + 3}",
            f"J{row}:K{row + 3}",
            f"L{row}:P{row + 3}",
        ))
    return rows, merges


def _strip_advice_rows(xml: str, headers: list[tuple[int, int]]) -> str:
    """Remove a previous suggestion block so the new table can take its rows."""
    drop = {row for header, _number in headers for row in range(header + 2, header + 32)}

    def keep_row(match: re.Match) -> str:
        number = int(re.search(r'\br="(\d+)"', match.group(0)).group(1))
        return "" if number in drop else match.group(0)

    xml = re.sub(r"<row\b.*?</row>", keep_row, xml, flags=re.S)

    def keep_merge(match: re.Match) -> str:
        rows = [int(value) for value in re.findall(r"\d+", match.group(0))]
        return "" if any(row in drop for row in rows) else match.group(0)

    xml = re.sub(r"<mergeCell\b[^>]*/>", keep_merge, xml)
    count = len(re.findall(r"<mergeCell\b", xml))
    xml = re.sub(r'<mergeCells count="\d+"', f'<mergeCells count="{count}"', xml, count=1)
    return xml


def _insert_sheet_rows(xml: str, rows: dict[int, str], merges: list[str]) -> str:
    match = re.search(r"(<sheetData>)(.*?)(</sheetData>)", xml, flags=re.S)
    if match is None:
        raise SystemExit("Worksheet has no sheetData")
    existing = re.findall(r"<row\b.*?</row>", match.group(2), flags=re.S)
    by_row: dict[int, str] = {}
    for row in existing:
        number = int(re.search(r'\br="(\d+)"', row).group(1))
        by_row[number] = row
    for number, row in rows.items():
        if number in by_row:
            raise SystemExit(f"Row {number} already exists; the suggestions table would overwrite it")
        by_row[number] = row
    body = "".join(by_row[number] for number in sorted(by_row))
    xml = xml[: match.start(2)] + body + xml[match.end(2) :]
    merge = re.search(r'<mergeCells count="(\d+)">', xml)
    if merge is None:
        raise SystemExit("Worksheet has no mergeCells")
    count = int(merge.group(1)) + len(merges)
    extra = "".join(f'<mergeCell ref="{ref}"/>' for ref in merges)
    xml = xml.replace(merge.group(0), f'<mergeCells count="{count}">', 1)
    xml = xml.replace("</mergeCells>", extra + "</mergeCells>", 1)
    return xml


def _add_advice_styles(styles: str) -> str:
    if "<cellXfs count=\"38\">" in styles:
        return styles
    if "<cellXfs count=\"35\">" not in styles:
        if "<fonts count=\"8\">" not in styles or "<cellXfs count=\"32\">" not in styles:
            raise SystemExit("Workbook styles are not the expected set; refusing to append advice styles")
        styles = styles.replace("<fonts count=\"8\">", "<fonts count=\"9\">", 1)
        styles = styles.replace(
            "</fonts>",
            '<font><name val="Calibri"/><b val="1"/><color rgb="001B4F72"/><sz val="11"/></font></fonts>',
            1,
        )
        styles = styles.replace("<cellXfs count=\"32\">", "<cellXfs count=\"35\">", 1)
        styles = styles.replace(
            "</cellXfs>",
            '<xf numFmtId="0" fontId="8" fillId="5" borderId="0" applyAlignment="1" xfId="0">'
            '<alignment horizontal="center" vertical="center"/></xf>'
            '<xf numFmtId="0" fontId="8" fillId="0" borderId="0" applyAlignment="1" xfId="0">'
            '<alignment horizontal="left" vertical="top" wrapText="1"/></xf>'
            '<xf numFmtId="0" fontId="6" fillId="0" borderId="0" applyAlignment="1" xfId="0">'
            '<alignment horizontal="left" vertical="top" wrapText="1"/></xf>'
            "</cellXfs>",
            1,
        )
    styles = styles.replace("<cellXfs count=\"35\">", "<cellXfs count=\"38\">", 1)
    added = (
        '<xf numFmtId="0" fontId="8" fillId="5" borderId="1" applyAlignment="1" xfId="0">'
        '<alignment horizontal="center" vertical="center" wrapText="1"/></xf>'
        '<xf numFmtId="0" fontId="8" fillId="0" borderId="1" applyAlignment="1" xfId="0">'
        '<alignment horizontal="center" vertical="top" wrapText="1"/></xf>'
        '<xf numFmtId="0" fontId="6" fillId="0" borderId="1" applyAlignment="1" xfId="0">'
        '<alignment horizontal="left" vertical="top" wrapText="1"/></xf>'
    )
    return styles.replace("</cellXfs>", added + "</cellXfs>", 1)


def _inline_cell(ref: str, style: int, text: str) -> str:
    return (
        f'<c r="{ref}" s="{style}" t="inlineStr"><is>'
        f'<t xml:space="preserve">{_xml_text(text)}</t></is></c>'
    )


def _add_solution_columns(xml: str, plans: list[dict]) -> str:
    """Add Site, Population and Final solution to the right of the bad-spot table."""
    header = re.search(
        r'<row r="(\d+)"[^>]*>(?:(?!</row>).)*<t>Top Cell Id</t>(?:(?!</row>).)*</row>',
        xml,
        flags=re.S,
    )
    if header is None:
        raise SystemExit("Bad-spot table header was not found")
    header_row = header.group(1)
    if "Final solution" not in header.group(0):
        extra = (
            _inline_cell(f"O{header_row}", 7, "Site")
            + _inline_cell(f"P{header_row}", 7, "Population")
            + _inline_cell(f"Q{header_row}", 7, "Final solution")
        )
        xml = xml.replace(header.group(0), header.group(0).replace("</row>", extra + "</row>"), 1)
    found = []
    for match in re.finditer(r'<row r="(\d+)"([^>]*)>(.*?)</row>', xml, flags=re.S):
        row_n = match.group(1)
        if re.search(rf'<c r="A{row_n}"[^>]*>\s*<is><t>Circle \d+</t>', match.group(0)):
            found.append(match)
    if len(found) != len(plans):
        raise SystemExit(f"Bad-spot table has {len(found)} rows and {len(plans)} solutions")
    pieces = []
    last = 0
    for match, plan in zip(found, plans):
        row_n = match.group(1)
        attrs = match.group(2)
        body = match.group(3)
        pieces.append(xml[last:match.start()])
        updates = (
            (f"O{row_n}", 20, str(plan.get("site", ""))),
            (f"P{row_n}", 20, str(plan.get("population", ""))),
            (f"Q{row_n}", 37, str(plan.get("final", ""))),
        )
        if f'r="Q{row_n}"' not in body:
            if "ht=" not in attrs:
                attrs += ' ht="52" customHeight="1"'
            body += "".join(_inline_cell(ref, style, text) for ref, style, text in updates)
        else:
            for ref, style, text in updates:
                body = re.sub(
                    rf'<c r="{ref}"[^>]*>.*?</c>',
                    _inline_cell(ref, style, text),
                    body,
                    count=1,
                    flags=re.S,
                )
        pieces.append(f'<row r="{row_n}"{attrs}>{body}</row>')
        last = match.end()
    pieces.append(xml[last:])
    xml = "".join(pieces)
    xml = re.sub(
        r'<col width="13" customWidth="1" min="15" max="15" />',
        '<col width="18" customWidth="1" min="15" max="15" />',
        xml,
        count=1,
    )
    xml = re.sub(
        r'<col width="13" customWidth="1" min="16" max="16" />',
        '<col width="16" customWidth="1" min="16" max="16" />',
        xml,
        count=1,
    )
    if 'min="17"' not in xml:
        xml = xml.replace("</cols>", '<col width="64" customWidth="1" min="17" max="17" /></cols>', 1)
    xml = re.sub(r'(<dimension ref="A1:)[A-Z]+(\d+")', r"\1Q\2", xml, count=1)
    marker = "Site, Population and Final solution on the right are filled for every bad spot in this table."
    if marker not in xml and "Map numbers are omitted." in xml:
        xml = xml.replace("Map numbers are omitted.", "Map numbers are omitted. " + marker, 1)
    return xml


def _shift_rows_down(xml: str, start: int, count: int) -> str:
    """Move every row at or below start down by count. Drawings are left as they are."""
    def shift_r(match: re.Match) -> str:
        row = int(match.group(2))
        if row >= start:
            row += count
        return f'r="{match.group(1)}{row}"'

    def shift_ref(match: re.Match) -> str:
        r1 = int(match.group(2))
        r2 = int(match.group(4))
        if r1 >= start:
            r1 += count
        if r2 >= start:
            r2 += count
        return f'ref="{match.group(1)}{r1}:{match.group(3)}{r2}"'

    xml = re.sub(r'\br="([A-Z]*)(\d+)"', shift_r, xml)
    return re.sub(r'ref="([A-Z]+)(\d+):([A-Z]+)(\d+)"', shift_ref, xml)


def _replace_summary_counts(xml: str, counts: dict[str, int], total: int) -> str:
    """Refresh the four count cells when the summary table is already on the sheet."""
    values = {
        "Tilt and azimuth": f"{counts['tune']:,}",
        "4th sector, L900": f"{counts['fourth']:,}",
        "New 3-sector site": f"{counts['new']:,}",
        "All bad spots": f"{total:,}",
    }

    def repl_row(match: re.Match) -> str:
        row = match.group(0)
        for label, value in values.items():
            if f">{label}<" not in row:
                continue
            return re.sub(
                r'(<c r="B\d+"[^>]*>.*?<t[^>]*>)([^<]*)(</t>)',
                lambda found: found.group(1) + value + found.group(3),
                row,
                count=1,
                flags=re.S,
            )
        return row

    return re.sub(r"<row r=\"\d+\"[^>]*>(?:(?!</row>).)*</row>", repl_row, xml)


def _drop_on_sheet_summary(xml: str) -> str:
    """Take the count table off the bad-spot sheet. The Cover sheet holds it."""
    found = re.search(
        r'<row r="(\d+)"[^>]*>(?:(?!</row>).)*Summary of proposed solutions',
        xml,
        flags=re.S,
    )
    if found is None:
        return xml
    start = int(found.group(1))
    ident = re.search(
        r'<row r="(\d+)"[^>]*>(?:(?!</row>).)*Identified ',
        xml,
        flags=re.S,
    )
    end = int(ident.group(1)) - 1 if ident is not None else start + 7
    span = end - start + 1

    def keep_row(match: re.Match) -> str:
        row = int(match.group(1))
        return "" if start <= row <= end else match.group(0)

    def keep_merge(match: re.Match) -> str:
        rows = [int(number) for number in re.findall(r"\d+", match.group(1))]
        return "" if rows and any(start <= row <= end for row in rows) else match.group(0)

    xml = re.sub(r'<row r="(\d+)"[^>]*>.*?</row>', keep_row, xml, flags=re.S)
    xml = re.sub(r'<mergeCell ref="([^"]+)" />', keep_merge, xml)
    xml = _shift_rows_down(xml, end + 1, -span)
    count = len(re.findall(r"<mergeCell ", xml))
    return re.sub(r'<mergeCells count="\d+">', f'<mergeCells count="{count}">', xml, count=1)


def _insert_solution_summary(xml: str, plans: list[dict]) -> str:
    """Put the solution counts between the last map and the identified-spots table."""
    counts = {"tune": 0, "fourth": 0, "new": 0}
    for plan in plans:
        kind = str(plan.get("kind") or "")
        if kind in counts:
            counts[kind] += 1
    if "Summary of proposed solutions" in xml:
        return _replace_summary_counts(xml, counts, len(plans))
    found = re.search(
        r'<row r="(\d+)"[^>]*>(?:(?!</row>).)*Identified [^<]*bad spots(?:(?!</row>).)*</row>',
        xml,
        flags=re.S,
    )
    if found is None:
        raise SystemExit("Identified bad-spot title was not found")
    start = int(found.group(1))
    xml = _shift_rows_down(xml, start, 8)
    lines = [
        (start, "Summary of proposed solutions", None, 6),
        (start + 2, "Solution", "Count", 7),
        (start + 3, "Tilt and azimuth", f"{counts['tune']:,}", 20),
        (start + 4, "4th sector, L900", f"{counts['fourth']:,}", 20),
        (start + 5, "New 3-sector site", f"{counts['new']:,}", 20),
        (start + 6, "All bad spots", f"{len(plans):,}", 7),
    ]
    rows = {}
    for row, left, right, style in lines:
        cells = [_inline_cell(f"A{row}", style, left)]
        if right is not None:
            cells.append(_inline_cell(f"B{row}", style, right))
        rows[row] = f'<row r="{row}" ht="20" customHeight="1">{"".join(cells)}</row>'
    return _insert_sheet_rows(xml, rows, [])


def _drop_workbook_sheets(parts: dict[str, bytes], sheet_names: tuple[str, ...]) -> None:
    """Remove named sheets, their drawings, and pictures nothing else uses."""
    wb_xml = parts["xl/workbook.xml"].decode("utf-8")
    rels_xml = parts["xl/_rels/workbook.xml.rels"].decode("utf-8")
    worksheets = []
    for name in sheet_names:
        match = re.search(
            rf'<sheet name="{re.escape(name)}" sheetId="\d+" state="[^"]+" r:id="([^"]+)" />',
            wb_xml,
        )
        if match is None:
            continue
        rid = match.group(1)
        wb_xml = wb_xml.replace(match.group(0), "")
        rel = re.search(rf'<Relationship\b[^>]*Id="{rid}"[^>]*/>', rels_xml)
        if rel is None:
            raise SystemExit(f"Workbook relationship {rid} for {name} was not found")
        target = re.search(r'Target="([^"]+)"', rel.group(0)).group(1).lstrip("/")
        rels_xml = rels_xml.replace(rel.group(0), "")
        worksheets.append(target)
    parts["xl/workbook.xml"] = wb_xml.encode("utf-8")
    parts["xl/_rels/workbook.xml.rels"] = rels_xml.encode("utf-8")
    if not worksheets:
        return
    drop = set(worksheets)
    for sheet in worksheets:
        rels_name = sheet.replace("worksheets/", "worksheets/_rels/") + ".rels"
        rels = parts.get(rels_name, b"").decode("utf-8")
        for target in re.findall(r'Target="([^"]+)"', rels):
            path = target.lstrip("/")
            if path.startswith("xl/"):
                drop.add(path)
                drawing_rels = path.replace("drawings/", "drawings/_rels/") + ".rels"
                drawing = parts.get(drawing_rels, b"").decode("utf-8")
                for media in re.findall(r'Target="([^"]+)"', drawing):
                    drop.add(media.lstrip("/"))
                drop.add(drawing_rels)
        drop.add(rels_name)
    kept_text = []
    for name, body in parts.items():
        if name in drop or not name.endswith(".rels"):
            continue
        kept_text.append(body.decode("utf-8"))
    still_used = "\n".join(kept_text)
    for path in list(drop):
        if path.startswith("xl/media/") and path in still_used:
            drop.remove(path)
    content = parts["[Content_Types].xml"].decode("utf-8")
    for path in drop:
        content = re.sub(rf'<Override PartName="/{re.escape(path)}"[^>]*/>', "", content)
        parts.pop(path, None)
    parts["[Content_Types].xml"] = content.encode("utf-8")


_COVER_SUMMARY_ROW = 110
_EP_SHEET = "xl/worksheets/sheet15.xml"
_EP_RID = "rId17"
_SOLUTION_NAME = {
    "tune": "Tilt and azimuth",
    "fourth": "4th sector, L900",
    "new": "New 3-sector site",
}


def _drop_cover_summary(xml: str) -> str:
    """Remove a previous BAD Spot Summary so a second advise does not stack it."""
    if "BAD Spot Summary" not in xml:
        return xml

    def keep_row(match: re.Match) -> str:
        return "" if int(match.group(1)) >= _COVER_SUMMARY_ROW else match.group(0)

    def keep_merge(match: re.Match) -> str:
        rows = [int(number) for number in re.findall(r"\d+", match.group(1))]
        return "" if rows and max(rows) >= _COVER_SUMMARY_ROW else match.group(0)

    xml = re.sub(r'<row r="(\d+)"[^>]*>.*?</row>', keep_row, xml, flags=re.S)
    xml = re.sub(r'<mergeCell ref="([^"]+)" />', keep_merge, xml)
    count = len(re.findall(r"<mergeCell ", xml))
    return re.sub(r'<mergeCells count="\d+">', f'<mergeCells count="{count}">', xml, count=1)


def _graft_cover_summary(xml: str, idle_plans: list[dict], active_plans: list[dict]) -> str:
    """Idle and Active bad-spot counts and the final solution for every spot."""
    xml = _drop_cover_summary(xml)
    rows: list[str] = []
    merges: list[str] = []
    row = _COVER_SUMMARY_ROW
    bar = [_inline_cell(f"A{row}", 5, "BAD Spot Summary")]
    for col in range(2, 21):
        bar.append(f'<c r="{get_column_letter(col)}{row}" s="5" t="n" />')
    rows.append(f'<row r="{row}" ht="22" customHeight="1">{"".join(bar)}</row>')
    merges.append(f"A{row}:T{row}")
    row += 2

    def counts_of(plans: list[dict]) -> dict[str, int]:
        counts = {"tune": 0, "fourth": 0, "new": 0}
        for plan in plans:
            kind = str(plan.get("kind") or "")
            if kind in counts:
                counts[kind] += 1
        return counts

    idle_counts = counts_of(idle_plans)
    active_counts = counts_of(active_plans)
    rows.append(
        f'<row r="{row}" ht="18" customHeight="1">'
        f'{_inline_cell(f"A{row}", 7, "Solution")}'
        f'{_inline_cell(f"B{row}", 7, "Idle")}'
        f'{_inline_cell(f"C{row}", 7, "Active")}</row>'
    )
    row += 1
    for label, key in (
        ("Tilt and azimuth", "tune"),
        ("4th sector, L900", "fourth"),
        ("New 3-sector site", "new"),
        ("All bad spots", None),
    ):
        idle_value = str(len(idle_plans) if key is None else idle_counts[key])
        active_value = str(len(active_plans) if key is None else active_counts[key])
        style = 7 if key is None else 20
        rows.append(
            f'<row r="{row}" ht="18" customHeight="1">'
            f'{_inline_cell(f"A{row}", style, label)}'
            f'{_inline_cell(f"B{row}", style, idle_value)}'
            f'{_inline_cell(f"C{row}", style, active_value)}</row>'
        )
        row += 1
    row += 1
    rows.append(
        f'<row r="{row}" ht="18" customHeight="1">'
        f'{_inline_cell(f"A{row}", 7, "Mode")}'
        f'{_inline_cell(f"B{row}", 7, "Bad spot")}'
        f'{_inline_cell(f"C{row}", 7, "Site")}'
        f'{_inline_cell(f"D{row}", 7, "Sector")}'
        f'{_inline_cell(f"E{row}", 7, "Serving cell")}'
        f'{_inline_cell(f"F{row}", 7, "Population")}'
        f'{_inline_cell(f"G{row}", 7, "Solution")}'
        f'{_inline_cell(f"H{row}", 7, "Final solution")}</row>'
    )
    merges.append(f"H{row}:T{row}")
    row += 1
    for mode, plans in (("Idle", idle_plans), ("Active", active_plans)):
        for index, plan in enumerate(plans, 1):
            sector, cell = _plan_sector_cell(plan)
            rows.append(
                f'<row r="{row}" ht="36" customHeight="1">'
                f'{_inline_cell(f"A{row}", 20, mode)}'
                f'{_inline_cell(f"B{row}", 20, f"Bad spot {index}")}'
                f'{_inline_cell(f"C{row}", 20, str(plan.get("site") or ""))}'
                f'{_inline_cell(f"D{row}", 37, sector)}'
                f'{_inline_cell(f"E{row}", 37, cell)}'
                f'{_inline_cell(f"F{row}", 20, str(plan.get("population") or ""))}'
                f'{_inline_cell(f"G{row}", 20, _SOLUTION_NAME.get(str(plan.get("kind") or ""), ""))}'
                f'{_inline_cell(f"H{row}", 37, str(plan.get("final") or ""))}</row>'
            )
            merges.append(f"H{row}:T{row}")
            row += 1
    last = row - 1
    xml = xml.replace("</sheetData>", "".join(rows) + "</sheetData>", 1)
    xml = re.sub(
        r'(<mergeCells count=")(\d+)(">)',
        lambda match: match.group(1) + str(int(match.group(2)) + len(merges)) + match.group(3),
        xml,
        count=1,
    )
    xml = xml.replace("</mergeCells>", "".join(f'<mergeCell ref="{ref}"/>' for ref in merges) + "</mergeCells>", 1)
    xml = re.sub(r'<dimension ref="A1:[A-Z]+\d+"', f'<dimension ref="A1:T{last}"', xml, count=1)
    xml = _place_cover_summary_beside_route(xml, idle_plans, active_plans)
    return xml


def _place_cover_summary_beside_route(xml: str, idle_plans: list[dict], active_plans: list[dict]) -> str:
    """Put the Idle and Active counts to the right of the DT route."""
    counts = {"tune": 0, "fourth": 0, "new": 0}
    idle_counts = dict(counts)
    active_counts = dict(counts)
    for plan in idle_plans:
        kind = str(plan.get("kind") or "")
        if kind in idle_counts:
            idle_counts[kind] += 1
    for plan in active_plans:
        kind = str(plan.get("kind") or "")
        if kind in active_counts:
            active_counts[kind] += 1
    xml = xml.replace(
        '<col width="14" customWidth="1" min="16" max="16" />',
        '<col width="22" customWidth="1" min="16" max="16" />',
        1,
    )

    def write(row: int, cells: dict[str, str]) -> None:
        nonlocal xml
        match = re.search(rf'(<row r="{row}"[^>]*>)(.*?)(</row>)', xml, flags=re.S)
        if match is None:
            raise SystemExit(f"Cover row {row} was not found for the summary")
        body = match.group(2)
        for col in cells:
            body = re.sub(
                rf'<c r="{col}{row}"[^>]*(?:/>|>.*?</c>)',
                "",
                body,
                count=1,
                flags=re.S,
            )
        body += "".join(cells[col] for col in ("P", "Q", "R", "S", "T") if col in cells)
        xml = xml[: match.start()] + match.group(1) + body + match.group(3) + xml[match.end() :]

    title = {col: f'<c r="{col}74" s="5" t="n" />' for col in "QRST"}
    title["P"] = _inline_cell("P74", 5, "Bad Spot Summary across Mode")
    write(74, title)
    write(76, {
        "P": _inline_cell("P76", 7, "Solution"),
        "Q": _inline_cell("Q76", 7, "Idle"),
        "R": _inline_cell("R76", 7, "Active"),
    })
    data_rows = (
        (77, "Tilt and azimuth", idle_counts["tune"], active_counts["tune"], 20),
        (78, "4th sector, L900", idle_counts["fourth"], active_counts["fourth"], 20),
        (79, "New 3-sector site", idle_counts["new"], active_counts["new"], 20),
        (80, "All bad spots", len(idle_plans), len(active_plans), 7),
    )
    for row, label, idle_n, active_n, style in data_rows:
        write(row, {
            "P": _inline_cell(f"P{row}", style, label),
            "Q": _inline_cell(f"Q{row}", style, str(idle_n)),
            "R": _inline_cell(f"R{row}", style, str(active_n)),
        })
    if 'ref="P74:T74"' not in xml:
        xml = re.sub(
            r'(<mergeCells count=")(\d+)(">)',
            lambda match: match.group(1) + str(int(match.group(2)) + 1) + match.group(3),
            xml,
            count=1,
        )
        xml = xml.replace("</mergeCells>", '<mergeCell ref="P74:T74"/></mergeCells>', 1)
    return xml


def _plan_sector_cell(plan: dict) -> tuple[str, str]:
    """Sector names and serving-cell names proposed for one bad spot."""
    sectors: list[str] = []
    cells: list[str] = []
    for action in plan.get("actions") or []:
        sector = str(action.get("sector") or "").strip()
        cell = str(action.get("cell") or "").strip()
        if sector and sector not in sectors:
            sectors.append(sector)
        if cell and cell not in cells:
            cells.append(cell)
    return ", ".join(sectors), ", ".join(cells)


def _ep_text(value: object) -> str:
    if value is None or value == "":
        return ""
    if isinstance(value, float):
        if abs(value - round(value)) < 0.05:
            return str(int(round(value)))
        return f"{value:.1f}"
    return str(value)


def _ep_rows(mode: str, plans: list[dict]) -> list[dict]:
    rows = []
    for index, plan in enumerate(plans, 1):
        for action in plan.get("actions") or []:
            item = dict(action)
            item["mode"] = mode
            item["spot"] = f"Bad spot {index}"
            rows.append(item)
    return rows


def _ep_sheet_xml(actions: list[dict]) -> bytes:
    """Sector and cell engineering-parameter tracker. Existing plan is blank for a new sector."""
    rest = "".join(f'<c r="{col}{{0}}" s="{{1}}" t="n" />' for col in "BCDEFGHIJKLMN")
    rows = [
        '<row r="1" ht="24" customHeight="1">'
        + _inline_cell("A1", 6, "EP Details")
        + rest.format(1, 6)
        + "</row>",
        '<row r="2" ht="32" customHeight="1">'
        + _inline_cell(
            "A2",
            16,
            "Engineering Parameter action tracker. One row is one sector or cell change for one bad spot. "
            "Existing Plan is blank when the sector is new. Height is metres. Azimuth, ET and MT are degrees.",
        )
        + rest.format(2, 16)
        + "</row>",
    ]
    top = {
        "A": "Site",
        "B": "Sector",
        "C": "Serving cell name",
        "D": "Existing Plan",
        "H": "Proposed Plan",
        "L": "Mode",
        "M": "Bad spot",
        "N": "Action",
    }
    sub = {
        "D": "Height",
        "E": "Azimuth",
        "F": "ET",
        "G": "MT",
        "H": "Height",
        "I": "Azimuth",
        "J": "ET",
        "K": "MT",
    }
    row3 = "".join(_inline_cell(f"{col}3", 7, top.get(col, "")) for col in "ABCDEFGHIJKLMN")
    row4 = "".join(
        _inline_cell(f"{col}4", 7, sub[col]) if col in sub else f'<c r="{col}4" s="7" t="n" />'
        for col in "ABCDEFGHIJKLMN"
    )
    rows.append(f'<row r="3" ht="20" customHeight="1">{row3}</row>')
    rows.append(f'<row r="4" ht="20" customHeight="1">{row4}</row>')
    fields = (
        "site", "sector", "cell",
        "height", "azimuth", "et", "mt",
        "p_height", "p_azimuth", "p_et", "p_mt",
        "mode", "spot", "action",
    )
    for offset, action in enumerate(actions):
        number = 5 + offset
        cells = []
        for col, field in zip("ABCDEFGHIJKLMN", fields):
            style = 20 if col in "ABCLMN" else 36
            if col == "C":
                style = 37
            cells.append(_inline_cell(f"{col}{number}", style, _ep_text(action.get(field))))
        rows.append(f'<row r="{number}" ht="22" customHeight="1">{"".join(cells)}</row>')
    last = 4 + max(len(actions), 1)
    merges = (
        "A1:N1", "A2:N2",
        "A3:A4", "B3:B4", "C3:C4",
        "D3:G3", "H3:K3",
        "L3:L4", "M3:M4", "N3:N4",
    )
    cols = (
        '<col width="28" customWidth="1" min="1" max="1" />'
        '<col width="16" customWidth="1" min="2" max="2" />'
        '<col width="28" customWidth="1" min="3" max="3" />'
        + "".join(
            f'<col width="12" customWidth="1" min="{index}" max="{index}" />'
            for index in range(4, 12)
        )
        + '<col width="12" customWidth="1" min="12" max="12" />'
        '<col width="16" customWidth="1" min="13" max="13" />'
        '<col width="22" customWidth="1" min="14" max="14" />'
    )
    xml = (
        '<worksheet xmlns="http://schemas.openxmlformats.org/spreadsheetml/2006/main">'
        '<sheetPr><outlinePr summaryBelow="1" summaryRight="1" /><pageSetUpPr fitToPage="1" /></sheetPr>'
        f'<dimension ref="A1:N{last}" />'
        '<sheetViews><sheetView showGridLines="0" view="normal" zoomScale="100" workbookViewId="0">'
        '<selection activeCell="A1" sqref="A1" /></sheetView></sheetViews>'
        '<sheetFormatPr baseColWidth="8" defaultRowHeight="15" />'
        f"<cols>{cols}</cols>"
        f'<sheetData>{"".join(rows)}</sheetData>'
        f'<mergeCells count="{len(merges)}">'
        + "".join(f'<mergeCell ref="{ref}"/>' for ref in merges)
        + "</mergeCells>"
        '<printOptions gridLines="0" />'
        '<pageMargins left="0.5" right="0.5" top="0.6" bottom="0.6" header="0.3" footer="0.3" />'
        '<pageSetup orientation="landscape" paperSize="9" fitToHeight="0" fitToWidth="1" />'
        "</worksheet>"
    )
    return xml.encode("utf-8")


def _ensure_ep_sheet(parts: dict[str, bytes]) -> None:
    workbook = parts["xl/workbook.xml"].decode("utf-8")
    if 'name="EP Details"' not in workbook:
        workbook = workbook.replace(
            '<sheet name="_ChartData"',
            f'<sheet name="EP Details" sheetId="15" state="visible" r:id="{_EP_RID}" />'
            '<sheet name="_ChartData"',
            1,
        )
        parts["xl/workbook.xml"] = workbook.encode("utf-8")
    rels = parts["xl/_rels/workbook.xml.rels"].decode("utf-8")
    if f'Id="{_EP_RID}"' not in rels:
        rels = rels.replace(
            "</Relationships>",
            '<Relationship Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/worksheet" '
            f'Target="/xl/worksheets/sheet15.xml" Id="{_EP_RID}" /></Relationships>',
            1,
        )
        parts["xl/_rels/workbook.xml.rels"] = rels.encode("utf-8")
    content = parts["[Content_Types].xml"].decode("utf-8")
    if "/xl/worksheets/sheet15.xml" not in content:
        content = content.replace(
            "</Types>",
            '<Override PartName="/xl/worksheets/sheet15.xml" '
            'ContentType="application/vnd.openxmlformats-officedocument.spreadsheetml.worksheet+xml"/></Types>',
            1,
        )
        parts["[Content_Types].xml"] = content.encode("utf-8")


def write_bad_spot_advice(src: Path | None = None) -> Path:
    """Graft the findings table onto the six bad-spot sheets without rewriting the file.

    openpyxl save drops the movable circles, so the cells are inserted into the
    worksheet XML. Row heights stay at the default, and the drawings are copied
    through unchanged.
    """

    src = src or REPORT_XLSX
    if not src.exists():
        raise SystemExit(f"Missing workbook: {src}")
    print("Loading drive logs", flush=True)
    idle = load_merged_csv() if MERGED_CSV.exists() else None
    active = load_active_merged() if ACTIVE_MERGED.exists() else None
    if idle is None or active is None:
        raise SystemExit("Merged drive logs are missing; a full report build is required.")
    active = select_active_report(active)
    idle_track = best_server(idle)
    active_track = best_carrier(active)
    tracks = {
        "Bad Spot RSRP": (idle_track, "RSRP"),
        "Bad Spot RSRP_Active": (active_track, "RSRP"),
    }
    advice_for: dict[str, list[dict]] = {}
    for sheet, (frame, kpi) in tracks.items():
        print(f"Advice for {sheet}", flush=True)
        spots = find_bad_spots(frame, kpi)
        if len(spots) > BAD_SPOT_TABLE_MAX:
            spots = spots.head(BAD_SPOT_TABLE_MAX)
        advice_for[sheet] = [bad_spot_advice(frame, kpi, spots.iloc[i]) for i in range(len(spots))]
        kinds: dict[str, int] = {}
        for plan in advice_for[sheet]:
            kinds[plan["kind"]] = kinds.get(plan["kind"], 0) + 1
        print(f"  {len(advice_for[sheet])} spots {kinds}", flush=True)

    with zipfile.ZipFile(src) as zf:
        names = zf.namelist()
        parts = {name: zf.read(name) for name in names}
    drawings_before = {
        name: parts[name] for name in names if name.startswith("xl/drawings/")
    }
    sheet_xml = _worksheet_xml(parts)
    for sheet, pairs in advice_for.items():
        xml = sheet_xml[sheet].decode("utf-8")
        headers = [
            (int(row), int(number))
            for row, number in re.findall(
                r'<c r="A(\d+)"[^>]*>\s*<is><t>Bad Spot (\d+)</t>',
                xml,
            )
        ]
        if len(headers) > len(pairs):
            raise SystemExit(f"{sheet}: {len(headers)} headers, {len(pairs)} advice rows")
        map_plans = pairs[: len(headers)]
        rows: dict[int, str] = {}
        merges: list[str] = []
        xml = _strip_advice_rows(xml, headers)
        for (header, number), plan in zip(headers, map_plans):
            if number < 1:
                raise SystemExit(f"{sheet}: unexpected header {number}")
            added_rows, added_merges = _advice_sheet_rows(header, plan)
            rows.update(added_rows)
            merges.extend(added_merges)
        xml = _insert_sheet_rows(xml, rows, merges)
        xml = _add_solution_columns(xml, pairs)
        xml = _drop_on_sheet_summary(xml)
        for old, new in zip(_ADVICE_NOTE_OLD, _ADVICE_NOTE_NEW):
            if old not in xml:
                if new not in xml:
                    raise SystemExit(f"{sheet}: missing note sentence {old!r}")
                continue
            xml = xml.replace(old, new)
        sheet_xml[sheet] = xml.encode("utf-8")
    # write the updated sheet bytes back onto their zip paths
    main = "{http://schemas.openxmlformats.org/spreadsheetml/2006/main}"
    rel_ns = "{http://schemas.openxmlformats.org/officeDocument/2006/relationships}"
    wb = ET.fromstring(parts["xl/workbook.xml"])
    book_rels = ET.fromstring(parts["xl/_rels/workbook.xml.rels"])
    rid_target = {rel.attrib["Id"]: rel.attrib["Target"].lstrip("/") for rel in book_rels}
    cover_path = None
    for node in wb.find(f"{main}sheets"):
        name = node.attrib["name"]
        path = rid_target[node.attrib[f"{rel_ns}id"]]
        if name == "Cover":
            cover_path = path
        if name in sheet_xml and isinstance(sheet_xml[name], (bytes, bytearray)):
            parts[path] = sheet_xml[name]
    if cover_path is None:
        raise SystemExit("Cover sheet was not found")
    parts[cover_path] = _graft_cover_summary(
        parts[cover_path].decode("utf-8"),
        advice_for["Bad Spot RSRP"],
        advice_for["Bad Spot RSRP_Active"],
    ).encode("utf-8")
    parts[_EP_SHEET] = _ep_sheet_xml(
        _ep_rows("Idle", advice_for["Bad Spot RSRP"])
        + _ep_rows("Active", advice_for["Bad Spot RSRP_Active"])
    )
    _ensure_ep_sheet(parts)
    parts["xl/styles.xml"] = _add_advice_styles(parts["xl/styles.xml"].decode("utf-8")).encode("utf-8")
    for name, body in drawings_before.items():
        if parts.get(name) != body:
            raise SystemExit(f"Drawing changed while adding advice: {name}")
    _drop_workbook_sheets(parts, _DROP_BAD_SPOT_SHEETS)
    tmp = src.with_suffix(".tmp.xlsx")
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as zf:
        for name in names:
            if name in parts:
                zf.writestr(name, parts[name])
        for name, body in parts.items():
            if name not in names:
                zf.writestr(name, body)
    tmp.replace(VERSIONED_XLSX)
    if VERSIONED_XLSX.resolve() != REPORT_XLSX.resolve():
        shutil.copy2(VERSIONED_XLSX, REPORT_XLSX)
    print(f"Wrote {VERSIONED_XLSX}", flush=True)
    return VERSIONED_XLSX


## 22. Web map

Build web/coverage_data.js from the same 50 m bins, attach every sector's cells from the dump, and zip the page.
The page version is the version in the page title. build_web_page writes the bins. add_cell_params and add_site_pies add the cell table. package_web writes the zip.


In [ ]:
# ------------------------------------------------------------------------
# 22. Web map
# Build web/coverage_data.js from the same 50 m bins, attach every sector's cells from the dump, and zip the page.
# The page version is the version in the page title. build_web_page writes the bins. add_cell_params and add_site_pies add the cell table. package_web writes the zip.
# ------------------------------------------------------------------------

WEB_DIR = ROOT / "web"
DATA_JS = WEB_DIR / "coverage_data.js"


def load_cell_directory(path: Path = SITE_DB_PATH) -> tuple[pd.DataFrame, pd.DataFrame]:
    """EUtranCellId names keyed by eNB, cell, and EARFCN."""
    from pyxlsb import open_workbook

    rows: list[list] = []
    cols: list[str] = []
    with open_workbook(str(path)) as wb:
        sheet_name = "4G" if "4G" in wb.sheets else wb.sheets[0]
        sheet = wb.get_sheet(sheet_name)
        for i, row in enumerate(sheet.rows()):
            vals = [c.v for c in row]
            if i == 0:
                cols = [str(v).strip() if v is not None else "" for v in vals]
                continue
            rows.append(vals)
    dump = pd.DataFrame(rows, columns=cols)
    missing = [c for c in ("eNBId", "cellId", "EARFCN", "EUtranCellId") if c not in dump.columns]
    if missing:
        raise SystemExit(f"System dump is missing columns: {missing}")
    dump["eNBId"] = pd.to_numeric(dump["eNBId"], errors="coerce")
    dump["cellId"] = pd.to_numeric(dump["cellId"], errors="coerce")
    dump["EARFCN"] = pd.to_numeric(dump["EARFCN"], errors="coerce")
    dump["EUtranCellId"] = dump["EUtranCellId"].astype("string").str.strip()
    dump = dump.dropna(subset=["eNBId", "cellId", "EUtranCellId"])
    dump = dump[dump["EUtranCellId"].ne("") & dump["EUtranCellId"].ne("<NA>")]
    dump["eNBId"] = dump["eNBId"].astype(np.int64)
    dump["cellId"] = dump["cellId"].astype(np.int64)
    exact = (
        dump.dropna(subset=["EARFCN"])
        .assign(EARFCN=lambda frame: frame["EARFCN"].round().astype(np.int64))
        .drop_duplicates(["eNBId", "cellId", "EARFCN"], keep="first")
        [["eNBId", "cellId", "EARFCN", "EUtranCellId"]]
    )
    pair = (
        dump.groupby(["eNBId", "cellId"], sort=False)["EUtranCellId"]
        .agg(lambda series: series.iloc[0] if series.nunique() == 1 else "")
        .reset_index()
        .rename(columns={"EUtranCellId": "pair_name"})
    )
    pair = pair[pair["pair_name"].ne("")]
    return exact, pair


def _as_int(series: pd.Series) -> pd.Series:
    num = pd.to_numeric(series, errors="coerce")
    return num.round().astype("Int64")


def attach_identity(df: pd.DataFrame, mode: str, exact: pd.DataFrame, pair: pd.DataFrame) -> pd.DataFrame:
    """Add the Cell ID and Cell Name used by the click popup."""
    work = df.copy()
    if mode == "idle":
        enb = _as_int(work["eNB-Id"]) if "eNB-Id" in work.columns else pd.Series(pd.NA, index=work.index, dtype="Int64")
        cell = _as_int(work["Sector Id"]) if "Sector Id" in work.columns else pd.Series(pd.NA, index=work.index, dtype="Int64")
        full = pd.to_numeric(work["Cell Id"], errors="coerce")
        missing = cell.isna() & full.notna()
        if missing.any():
            enb = enb.mask(missing, (full // 256).round().astype("Int64"))
            cell = cell.mask(missing, (full % 256).round().astype("Int64"))
    else:
        full = pd.to_numeric(work["Cell Id"], errors="coerce")
        enb = (full // 256).round().astype("Int64")
        cell = (full % 256).round().astype("Int64")
    work["cell_id"] = cell
    work["_enb"] = enb
    earfcn = pd.to_numeric(work.get("DL EARFCN"), errors="coerce").round().astype("Int64")
    work["_earfcn"] = earfcn
    keyed = work.merge(exact, left_on=["_enb", "cell_id", "_earfcn"], right_on=["eNBId", "cellId", "EARFCN"], how="left")
    keyed = keyed.merge(pair, left_on=["_enb", "cell_id"], right_on=["eNBId", "cellId"], how="left", suffixes=("", "_pair"))
    if len(keyed) != len(work):
        raise RuntimeError(f"Cell name lookup changed the row count ({len(work)} -> {len(keyed)})")
    name = keyed["EUtranCellId"].astype("string")
    name = name.mask(name.isna() | name.eq("") | name.eq("<NA>"), keyed["pair_name"].astype("string"))
    cleaned = name.fillna("").astype(str)
    cleaned = cleaned.where(~cleaned.isin(["<NA>", "nan", "None"]), "")
    work["cell_name"] = cleaned.to_numpy()
    return work


def bin_track(df: pd.DataFrame) -> pd.DataFrame:
    """One spot per 50 m bin: median colour, cell from the sample nearest that median."""
    work = df.dropna(subset=["Longitude", "Latitude"]).copy().reset_index(drop=True)
    work["RSRP"] = pd.to_numeric(work["RSRP"], errors="coerce")
    work["RSRQ"] = pd.to_numeric(work["RSRQ"], errors="coerce")
    work = work[work["RSRP"].notna() | work["RSRQ"].notna()]
    if work.empty:
        return work
    lat = work["Latitude"]
    lon = work["Longitude"]
    dlat, dlon = _bin_steps(lat, MAP_BIN_M)
    work["_bin"] = list(zip(np.floor(lat / dlat).astype(np.int64), np.floor(lon / dlon).astype(np.int64)))
    med = work.groupby("_bin", sort=False)[["RSRP", "RSRQ"]].median()
    work["_med"] = work["_bin"].map(med["RSRP"])
    work["_dist"] = (work["RSRP"] - work["_med"]).abs()
    work["_dist"] = work["_dist"].fillna(1e9)
    rep_idx = work.groupby("_bin", sort=False)["_dist"].idxmin()
    rep = work.loc[rep_idx, ["_bin", "cell_id", "cell_name"]].set_index("_bin")
    pos = work.groupby("_bin", sort=False)[["Latitude", "Longitude"]].mean()
    out = pos.join(med).join(rep)
    out = out.reset_index(drop=True)
    return out


def _records(frame: pd.DataFrame, names: dict[str, int]) -> dict:
    if frame.empty:
        return {"lon": [], "lat": [], "rsrp": [], "rsrq": [], "cell": [], "name": []}
    cell = pd.to_numeric(frame["cell_id"], errors="coerce")
    name_ids = []
    for value in frame["cell_name"].astype(str):
        if value in ("", "nan", "<NA>"):
            name_ids.append(0)
            continue
        if value not in names:
            names[value] = len(names)
        name_ids.append(names[value])
    def series_or_null(col: str, digits: int) -> list:
        num = pd.to_numeric(frame[col], errors="coerce").round(digits)
        return [None if pd.isna(v) else float(v) for v in num]

    cells = [None if pd.isna(v) else int(v) for v in cell]
    return {
        "lon": series_or_null("Longitude", 5),
        "lat": series_or_null("Latitude", 5),
        "rsrp": series_or_null("RSRP", 1),
        "rsrq": series_or_null("RSRQ", 1),
        "cell": cells,
        "name": name_ids,
    }


def build_web_page() -> None:
    if not MERGED_CSV.exists() or not ACTIVE_MERGED.exists():
        raise SystemExit("Merged drive logs are missing.")
    if not SITE_DB_PATH.exists():
        raise SystemExit(f"Missing {SITE_DB_PATH.name}")
    print("Loading system dump", flush=True)
    exact, pair = load_cell_directory()
    print(f"  names by EARFCN {len(exact):,}  unique pairs {len(pair):,}", flush=True)
    print("Loading drive logs", flush=True)
    idle = load_merged_csv()
    active = select_active_report(load_active_merged())
    idle_base = strongest_fix(idle)
    idle_all = best_server(idle_base)
    active_all = strongest_location(active)
    print(f"Idle best server {len(idle_all):,}  Active plotted {len(active_all):,}", flush=True)

    tracks: dict[str, pd.DataFrame] = {
        "idle|all": idle_all,
        "active|all": active_all,
    }
    for layer in REPORT_LAYERS:
        tracks[f"idle|{layer}"] = best_server(filter_by_layer(idle_base, layer))
        tracks[f"active|{layer}"] = filter_by_layer(active_all, layer)

    names: dict[str, int] = {"": 0}
    sets = {}
    for key, frame in tracks.items():
        mode = "idle" if key.startswith("idle") else "active"
        print(f"Binning {key} ({len(frame):,} rows)", flush=True)
        identified = attach_identity(frame, mode, exact, pair)
        binned = bin_track(identified)
        named = int((binned["cell_name"].astype(str).str.len() > 0).sum()) if not binned.empty else 0
        print(f"  {len(binned):,} spots, {named:,} with a Cell Name", flush=True)
        sets[key] = _records(binned, names)

    # names dict was filled in first-seen order starting at 1; index 0 stays blank.
    ordered = [""] * len(names)
    for value, index in names.items():
        ordered[index] = value
    payload = {"names": ordered, "sets": sets}
    WEB_DIR.mkdir(parents=True, exist_ok=True)
    text = "const COVERAGE = " + json.dumps(payload, separators=(",", ":")) + ";\n"
    DATA_JS.write_text(text)
    print(f"Wrote {DATA_JS} ({DATA_JS.stat().st_size / 1e6:.1f} MB)", flush=True)
    add_cell_params()
    add_site_pies()
    package_web()


RF_COLUMNS = (
    "Azimuth", "Height", "E_tilt", "M_tilt", "RS Power(dBm)",
    "dlChannelBandwidth", "noOfTxAntennas", "noOfRxAntennas",
)
RF_DIGITS = (1, 1, 1, 1, 2, 0, 0, 0)


def _rf_number(value, digits: int):
    number = pd.to_numeric(pd.Series([value]), errors="coerce").iloc[0]
    if pd.isna(number):
        return None
    return round(float(number), digits)


def add_cell_params() -> int:
    """Azimuth, height, tilts and RS power for each CellName, from the system dump."""
    from pyxlsb import open_workbook

    lookup: dict[str, list] = {}
    with open_workbook(str(SITE_DB_PATH)) as wb:
        sheet_name = "4G" if "4G" in wb.sheets else wb.sheets[0]
        sheet = wb.get_sheet(sheet_name)
        index: dict[str, int] = {}
        for i, row in enumerate(sheet.rows()):
            vals = [c.v for c in row]
            if i == 0:
                cols = [str(v).strip() if v is not None else "" for v in vals]
                missing = [name for name in ("EUtranCellId",) + RF_COLUMNS if name not in cols]
                if missing:
                    raise SystemExit(f"System dump is missing columns: {missing}")
                index = {name: cols.index(name) for name in ("EUtranCellId",) + RF_COLUMNS}
                continue
            raw_name = vals[index["EUtranCellId"]]
            if raw_name is None:
                continue
            name = str(raw_name).strip()
            if not name or name in lookup:
                continue
            lookup[name] = [
                _rf_number(vals[index[col]], digits)
                for col, digits in zip(RF_COLUMNS, RF_DIGITS)
            ]
    text = DATA_JS.read_text().strip()
    if not text.startswith("const COVERAGE = "):
        raise SystemExit(f"{DATA_JS.name} is not a coverage page")
    payload = json.loads(text[len("const COVERAGE = "):-1])
    cells: list = []
    filled = 0
    for name in payload["names"]:
        if not name:
            cells.append(None)
            continue
        row = lookup.get(name)
        cells.append(row)
        if row is not None:
            filled += 1
    payload["cells"] = cells
    DATA_JS.write_text("const COVERAGE = " + json.dumps(payload, separators=(",", ":")) + ";\n")
    print(f"Cell parameters {filled:,} of {len(payload['names']) - 1:,} names", flush=True)
    return filled


_SECTOR_CELLS: dict[tuple[str, str], list] | None = None


def _grab(value, digits: int):
    if value is None:
        return None
    try:
        if math.isnan(value):
            return None
    except TypeError:
        return None
    if digits == 0:
        return int(value)
    return float(value)


def _sector_cell_index() -> dict[tuple[str, str], list]:
    """Cell parameters for each site and sector, from the system dump."""
    global _SECTOR_CELLS
    if _SECTOR_CELLS is not None:
        return _SECTOR_CELLS

    buckets: dict[tuple[str, str], list] = {}
    frame = _SITE_CELLS
    if frame is None or frame.empty:
        _SECTOR_CELLS = buckets
        return buckets
    work = frame.loc[:, [
        "SiteName", "Sector", "EUtranCellId", "cellId",
        "Azimuth", "Height", "E_tilt", "M_tilt", "RS Power(dBm)",
        "dlChannelBandwidth", "noOfTxAntennas", "noOfRxAntennas",
    ]].copy()
    for col in ("SiteName", "Sector", "EUtranCellId"):
        work[col] = work[col].map(lambda value: "" if value is None else str(value).strip())
    work = work[
        work["SiteName"].ne("")
        & work["Sector"].ne("")
        & work["EUtranCellId"].ne("")
        & work["EUtranCellId"].str.lower().ne("nan")
    ]
    work = work.drop_duplicates(["SiteName", "Sector", "EUtranCellId"], keep="first")
    work = work.rename(columns={
        "SiteName": "site",
        "Sector": "sector",
        "EUtranCellId": "name",
        "cellId": "cid",
        "Azimuth": "az",
        "Height": "ht",
        "E_tilt": "et",
        "M_tilt": "mt",
        "RS Power(dBm)": "pwr",
        "dlChannelBandwidth": "bw",
        "noOfTxAntennas": "tx",
        "noOfRxAntennas": "rx",
    })
    for col, digits in (
        ("cid", 0), ("az", 1), ("ht", 1), ("et", 1), ("mt", 1),
        ("pwr", 2), ("bw", 0), ("tx", 0), ("rx", 0),
    ):
        work[col] = pd.to_numeric(work[col], errors="coerce").round(digits)
    for row in work.itertuples(index=False):
        key = (row.site, row.sector)
        buckets.setdefault(key, []).append([
            row.name,
            _grab(row.cid, 0),
            _grab(row.az, 1),
            _grab(row.ht, 1),
            _grab(row.et, 1),
            _grab(row.mt, 1),
            _grab(row.pwr, 2),
            _grab(row.bw, 0),
            _grab(row.tx, 0),
            _grab(row.rx, 0),
        ])
    for rows in buckets.values():
        rows.sort(key=lambda item: item[0])
    _SECTOR_CELLS = buckets
    return buckets


def _cells_for_sector(site: str, sector: str) -> list:
    return _sector_cell_index().get((site, sector), [])


def add_site_pies(radius_km: float = 3.0) -> int:
    """Every site in the system dump, one centre and every sector arm.

    The page draws a site only while its centre is inside the chart. radius_km
    is unused; it remains so older calls keep working.
    """

    del radius_km
    text = DATA_JS.read_text().strip()
    if not text.startswith("const COVERAGE = "):
        raise SystemExit(f"{DATA_JS.name} is not a coverage page")
    payload = json.loads(text[len("const COVERAGE = "):-1])
    sectors = load_site_sectors()
    sites: list[list] = []
    for _site, part in sectors.groupby("SiteName", sort=False):
        part = part.drop_duplicates(subset=["Sector"]).dropna(subset=["azimuth"])
        if part.empty:
            continue
        arms = []
        site_name = str(_site).strip()
        if site_name.lower() == "nan":
            site_name = ""
        for _, row in part.iterrows():
            name = str(row["Sector"]).strip()
            if not name or name.lower() == "nan":
                continue
            arms.append([
                name,
                round(float(row["azimuth"]) % 360.0, 1),
                _cells_for_sector(site_name, name),
            ])
        if not arms:
            continue
        sites.append([
            round(float(part["lat"].median()), 5),
            round(float(part["lon"].median()), 5),
            site_name,
            arms,
        ])
    payload["sites"] = sites
    DATA_JS.write_text("const COVERAGE = " + json.dumps(payload, separators=(",", ":")) + ";\n")
    print(f"Site pies {len(sites):,} from the system dump", flush=True)
    return len(sites)


def _web_version(html_text: str) -> str:
    match = re.search(r"<h1>Bogura coverage v(\d+(?:\.\d+)+)</h1>", html_text)
    if not match:
        raise SystemExit("web/index.html has no Bogura coverage version")
    return match.group(1)


def package_web() -> Path:
    """One download named with the version in the page title."""

    html_text = (WEB_DIR / "index.html").read_text()
    version = _web_version(html_text)
    marker = f"v{version}"
    if marker not in html_text:
        raise SystemExit(f"web/index.html is not labelled {marker}")
    if not DATA_JS.exists():
        raise SystemExit(f"Missing {DATA_JS.name}; build the coverage data first")
    page = html_text.replace(
        '<script src="coverage_data.js"></script>',
        "<script>\n" + DATA_JS.read_text() + "</script>",
    )
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    dest = OUTPUT_DIR / f"Bogura_DriveTest_Web_v{version}.zip"
    inner = f"Bogura_DriveTest_Web_v{version}.html"
    with zipfile.ZipFile(dest, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        archive.writestr(inner, page)
    print(f"Wrote {dest} ({dest.stat().st_size / 1e6:.1f} MB) containing {inner}", flush=True)
    return dest


## 23. Summary presentation

The current 15-slide deck, v1.65. Three examples of each solution. The slide colours are named PPT_ so they do not replace the Excel colour names.
This deck stays as it is until the next instruction.


In [ ]:
# ------------------------------------------------------------------------
# 23. Summary presentation
# The current 15-slide deck, v1.65. Three examples of each solution. The slide colours are named PPT_ so they do not replace the Excel colour names.
# This deck stays as it is until the next instruction.
# ------------------------------------------------------------------------

OUTPUT = ROOT / "output"
REPORT = OUTPUT / "Bogura_DriveTest_Report_v1.63.xlsx"
PLOTS = OUTPUT / "plots"
DEST = OUTPUT / "Bogura_DriveTest_Summary_v1.65.pptx"

PPT_NAVY = RGBColor(0x1B, 0x4F, 0x72)
INK = RGBColor(0x1A, 0x1A, 0x1A)
MUTED = RGBColor(0x5D, 0x6D, 0x7E)
PPT_WHITE = RGBColor(0xFF, 0xFF, 0xFF)
PAPER = RGBColor(0xF7, 0xF9, 0xFB)
LINE = RGBColor(0xD5, 0xDD, 0xE4)
GOLD = RGBColor(0xB7, 0x86, 0x0B)
ORANGE = RGBColor(0xD3, 0x54, 0x00)
PURPLE = RGBColor(0x6C, 0x34, 0x83)

SOLUTION_COLOR = {
    "Tilt and azimuth": GOLD,
    "4th sector, L900": ORANGE,
    "New 3-sector site": PURPLE,
}

# Three examples of each final solution. The words on the slide come from the workbook.
EXAMPLES = [
    {
        "solution": "Tilt and azimuth",
        "mode": "Idle",
        "n": 1,
        "why": "The sector already points at the red road. Only the tilt changes, from 3° to 1.8°.",
    },
    {
        "solution": "Tilt and azimuth",
        "mode": "Active",
        "n": 97,
        "why": "The tilt is already right for 0.6 km. The azimuth turns 8°, from 140° to 132°. The dashed line leaves GBSDR37A because that cell served the red samples. The plan turns GBSDR37B.",
    },
    {
        "solution": "Tilt and azimuth",
        "mode": "Idle",
        "n": 20,
        "why": "The site is 0.4 km from the spot. Downtilt increases by 2°, the maximum on an existing sector, and the azimuth turns 9°.",
    },
    {
        "solution": "4th sector, L900",
        "mode": "Active",
        "n": 4,
        "why": "The red road is 0.2 km from GBPLS02 and sits in the null between its sectors. A new L900 sector is aimed through that null. Existing sectors stay.",
    },
    {
        "solution": "4th sector, L900",
        "mode": "Idle",
        "n": 12,
        "why": "The existing sector would need more than 2° of extra downtilt. A 4th L900 sector on BOSBG02, 0.2 km away, is used instead.",
    },
    {
        "solution": "4th sector, L900",
        "mode": "Active",
        "n": 1,
        "why": "BOSRK11 is 2.6 km away, inside 3 km, so a 4th L900 sector replaces a new site. It also covers the 1,301 m blank.",
    },
    {
        "solution": "New 3-sector site",
        "mode": "Idle",
        "n": 4,
        "why": "The nearest existing site is 3.3 km away, just past the 3 km line, so the final step is a new site.",
    },
    {
        "solution": "New 3-sector site",
        "mode": "Active",
        "n": 2,
        "why": "This spot includes the long blank. The nearest existing site is 3.9 km away, so the plan is a new site on the spot.",
    },
    {
        "solution": "New 3-sector site",
        "mode": "Active",
        "n": 10,
        "why": "The nearest existing site is 4.8 km away. No existing sector is close enough to cover this spot.",
    },
]


def _cover_rows() -> dict[tuple[str, int], dict[str, str]]:
    ns = {"m": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}
    with zipfile.ZipFile(REPORT) as book:
        root = ET.fromstring(book.read("xl/worksheets/sheet1.xml"))
    found: dict[tuple[str, int], dict[str, str]] = {}
    for row in root.findall("m:sheetData/m:row", ns):
        cells: dict[str, str] = {}
        for cell in row.findall("m:c", ns):
            ref = cell.get("r") or ""
            col = re.match(r"[A-Z]+", ref)
            text = cell.find("m:is/m:t", ns)
            if col and text is not None and text.text:
                cells[col.group(0)] = text.text
        mode = cells.get("A")
        if mode not in ("Idle", "Active"):
            continue
        number = re.search(r"(\d+)", cells.get("B", ""))
        if number is None:
            continue
        found[(mode, int(number.group(1)))] = cells
    return found


def _map_path(mode: str, number: int) -> Path:
    if mode == "Idle":
        return PLOTS / f"excel_bad_spots_rsrp_spot_{number:02d}.png"
    return PLOTS / f"excel_bad_spots_rsrp_spot_active_{number:02d}.png"


def _jpeg(path: Path, width: int = 1400) -> BytesIO:
    image = Image.open(path).convert("RGB")
    if image.width > width:
        height = round(image.height * width / image.width)
        image = image.resize((width, height), Image.Resampling.LANCZOS)
    buf = BytesIO()
    image.save(buf, format="JPEG", quality=80, optimize=True)
    buf.seek(0)
    return buf


def _set_run(run, text: str, size: int, color: RGBColor, bold: bool = False) -> None:
    run.text = text
    run.font.name = "Calibri"
    run.font.size = Pt(size)
    run.font.color.rgb = color
    run.font.bold = bold
    run.font.italic = False


def _box(slide, left, top, width, height, fill: RGBColor, line: RGBColor | None = None):
    shape = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, left, top, width, height)
    shape.fill.solid()
    shape.fill.fore_color.rgb = fill
    if line is None:
        shape.line.fill.background()
    else:
        shape.line.color.rgb = line
        shape.line.width = Pt(1)
    # Drop the shadow that PowerPoint adds on new shapes.
    effect = shape._element.find(qn("a:effectLst"))
    if effect is not None:
        effect.getparent().remove(effect)
    return shape


def _text(slide, left, top, width, height, text: str, size: int, color: RGBColor,
          bold: bool = False, align=PP_ALIGN.LEFT, anchor=MSO_ANCHOR.TOP):
    shape = slide.shapes.add_textbox(left, top, width, height)
    frame = shape.text_frame
    frame.word_wrap = True
    frame.auto_size = None
    frame.margin_left = Emu(0)
    frame.margin_right = Emu(0)
    frame.margin_top = Emu(0)
    frame.margin_bottom = Emu(0)
    try:
        frame._txBody.bodyPr.set("anchor", {MSO_ANCHOR.TOP: "t", MSO_ANCHOR.MIDDLE: "ctr", MSO_ANCHOR.BOTTOM: "b"}[anchor])
    except Exception:
        pass
    paragraph = frame.paragraphs[0]
    paragraph.alignment = align
    paragraph.space_before = Pt(0)
    paragraph.space_after = Pt(0)
    _set_run(paragraph.add_run(), text, size, color, bold)
    return shape


def _footer(slide, page: int, pages: int) -> None:
    _text(
        slide, Inches(0.4), Inches(7.18), Inches(10), Inches(0.24),
        "Bogura drive test  ·  Summary v1.65  ·  Excel v1.63  ·  Web v1.64",
        11, MUTED,
    )
    _text(
        slide, Inches(11.4), Inches(7.18), Inches(1.5), Inches(0.24),
        f"{page}  /  {pages}", 11, MUTED, align=PP_ALIGN.RIGHT,
    )


def _title(prs: Presentation) -> None:
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    _box(slide, 0, 0, prs.slide_width, prs.slide_height, PPT_NAVY)
    _box(slide, 0, 0, Inches(0.18), prs.slide_height, GOLD)
    _text(slide, Inches(0.7), Inches(1.35), Inches(11), Inches(0.35), "BOGURA DRIVE TEST", 16, RGBColor(0xF5, 0xC4, 0x00), True)
    _text(slide, Inches(0.7), Inches(1.8), Inches(11.5), Inches(1.3), "Bad spot summary", 44, PPT_WHITE, True)
    _text(
        slide, Inches(0.7), Inches(3.25), Inches(11), Inches(0.8),
        "Idle and Active. Three examples of each solution:\ntilt and azimuth, a 4th L900 sector, and a new 3-sector site.",
        20, RGBColor(0xD6, 0xE4, 0xF0),
    )
    cards = [
        ("11  +  42", "Tilt and azimuth", GOLD),
        ("12  +  45", "4th sector, L900", ORANGE),
        ("3  +  12", "New 3-sector site", PURPLE),
    ]
    for index, (count, label, color) in enumerate(cards):
        left = Inches(0.7 + index * 3.9)
        _box(slide, left, Inches(4.7), Inches(3.55), Inches(1.45), RGBColor(0x15, 0x3E, 0x5A))
        _box(slide, left, Inches(4.7), Inches(0.1), Inches(1.45), color)
        _text(slide, left + Inches(0.28), Inches(4.85), Inches(3.1), Inches(0.55), count, 26, PPT_WHITE, True)
        _text(slide, left + Inches(0.28), Inches(5.45), Inches(3.1), Inches(0.4), label, 14, RGBColor(0xD6, 0xE4, 0xF0))
    _text(slide, Inches(0.7), Inches(6.6), Inches(11), Inches(0.3), "Idle count  +  Active count.   26 idle bad spots and 99 active bad spots.", 14, RGBColor(0xAE, 0xC6, 0xD6))


def _measured(prs: Presentation, page: int, pages: int) -> None:
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    _box(slide, 0, 0, prs.slide_width, Inches(0.72), PPT_NAVY)
    _text(slide, Inches(0.4), Inches(0.16), Inches(12), Inches(0.42), "What was measured", 26, PPT_WHITE, True)
    rows = [
        ("Idle", "Borga_Idle_V2.0. This drive is unchanged."),
        ("Active", "Borga_Active_mode_11Nov26. 11–21 May 2026. Robi, M-PCell, one band at each location: the strongest RSRP."),
        ("Plotted Active rows", "288,308 of 553,201 Robi M-PCell rows. 2,185,081 rows after the duplicate drop."),
        ("Bad spot", "A 50 m bin is poor when its median RSRP is below −120 dBm. A blank with no plot inside the orange circle is counted with that spot."),
        ("This deck", "The rules, the counts, and three examples of each solution. Every bad spot stays in the Excel workbook."),
    ]
    top = 1.05
    for label, body in rows:
        _box(slide, Inches(0.4), Inches(top), Inches(12.5), Inches(1.05), PAPER, LINE)
        _text(slide, Inches(0.6), Inches(top + 0.12), Inches(2.4), Inches(0.8), label, 16, PPT_NAVY, True, anchor=MSO_ANCHOR.MIDDLE)
        _text(slide, Inches(3.15), Inches(top + 0.14), Inches(9.4), Inches(0.78), body, 16, INK, anchor=MSO_ANCHOR.MIDDLE)
        top += 1.15
    _footer(slide, page, pages)


def _rules(prs: Presentation, page: int, pages: int) -> None:
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    _box(slide, 0, 0, prs.slide_width, Inches(0.72), PPT_NAVY)
    _text(slide, Inches(0.4), Inches(0.16), Inches(12), Inches(0.42), "How a solution is chosen", 26, PPT_WHITE, True)
    cards = [
        (GOLD, "1", "Tilt and azimuth",
         "A sector inside about 2.5 km can be aimed at the spot. Azimuth is turned only when the population on the new bearing still serves the customers. Downtilt of an existing sector does not increase by more than 2°."),
        (ORANGE, "2", "4th sector, L900",
         "Used when the spot sits in the null between sectors, when the required downtilt would be more than 2°, or when an existing site inside 3 km can aim at the spot. Existing sectors are not moved."),
        (PURPLE, "3", "New 3-sector site",
         "Used only when the nearest existing site is farther than 3 km. The site is drawn at the spot, with an azimuth, E_tilt and M_tilt for each of the three L900 sectors."),
    ]
    for index, (color, number, title, body) in enumerate(cards):
        left = Inches(0.35 + index * 4.3)
        _box(slide, left, Inches(1.15), Inches(4.05), Inches(5.55), PAPER, LINE)
        _box(slide, left, Inches(1.15), Inches(4.05), Inches(0.12), color)
        _text(slide, left + Inches(0.25), Inches(1.5), Inches(3.5), Inches(0.4), number, 20, color, True)
        _text(slide, left + Inches(0.25), Inches(2.05), Inches(3.55), Inches(0.8), title, 22, PPT_NAVY, True)
        _text(slide, left + Inches(0.25), Inches(3.05), Inches(3.55), Inches(3.2), body, 16, INK)
    _footer(slide, page, pages)


def _counts(prs: Presentation, page: int, pages: int) -> None:
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    _box(slide, 0, 0, prs.slide_width, Inches(0.72), PPT_NAVY)
    _text(slide, Inches(0.4), Inches(0.16), Inches(12), Inches(0.42), "Bad spot summary across mode", 26, PPT_WHITE, True)
    headers = ["Solution", "Idle", "Active", "Together"]
    data = [
        ("Tilt and azimuth", "11", "42", "53", GOLD),
        ("4th sector, L900", "12", "45", "57", ORANGE),
        ("New 3-sector site", "3", "12", "15", PURPLE),
        ("All bad spots", "26", "99", "125", PPT_NAVY),
    ]
    widths = [4.6, 2.2, 2.2, 2.4]
    left0 = 0.85
    top = 1.45
    x = left0
    for header, width in zip(headers, widths):
        _box(slide, Inches(x), Inches(top), Inches(width), Inches(0.62), PPT_NAVY)
        _text(slide, Inches(x), Inches(top + 0.1), Inches(width), Inches(0.42), header, 16, PPT_WHITE, True, PP_ALIGN.CENTER)
        x += width
    for row_index, (label, idle, active, total, color) in enumerate(data):
        y = top + 0.62 + row_index * 0.95
        values = [label, idle, active, total]
        x = left0
        fill = RGBColor(0xEE, 0xF3, 0xF7) if row_index % 2 == 0 else PPT_WHITE
        if row_index == len(data) - 1:
            fill = RGBColor(0xE8, 0xF0, 0xF6)
        for col, (value, width) in enumerate(zip(values, widths)):
            _box(slide, Inches(x), Inches(y), Inches(width), Inches(0.95), fill, LINE)
            if col == 0:
                _box(slide, Inches(x), Inches(y), Inches(0.1), Inches(0.95), color)
            _text(
                slide, Inches(x + (0.2 if col == 0 else 0)), Inches(y + 0.22), Inches(width), Inches(0.5),
                value, 20, INK, col == 0 or row_index == len(data) - 1, PP_ALIGN.CENTER if col else PP_ALIGN.LEFT,
            )
            x += width
    _text(
        slide, Inches(0.85), Inches(6.35), Inches(11.5), Inches(0.55),
        "The next slides show three examples of each solution. The other 116 bad spots use the same rules and are in the workbook.",
        16, MUTED,
    )
    _footer(slide, page, pages)


def _legend(prs: Presentation, page: int, pages: int) -> None:
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    _box(slide, 0, 0, prs.slide_width, Inches(0.72), PPT_NAVY)
    _text(slide, Inches(0.4), Inches(0.16), Inches(12), Inches(0.42), "How to read an example", 26, PPT_WHITE, True)
    items = [
        ("Red on the road", "The poor samples. The length beside the red is that red stretch."),
        ("Orange circle", "The bad spot. A blank with no plot inside the circle is part of the same spot."),
        ("Yellow dashed line", "The cell that served the red samples. It starts at the end of that sector. It is not the plan."),
        ("Light Y", "The existing sectors of a site in the view. The arms meet at the site centre."),
        ("Orange arm", "A proposed 4th L900 sector, drawn from the existing site toward the spot."),
        ("Purple arms", "A proposed new 3-sector site, drawn at the planned coordinates."),
    ]
    for index, (title, body) in enumerate(items):
        col = index % 2
        row = index // 2
        left = 0.4 + col * 6.45
        top = 1.15 + row * 1.85
        _box(slide, Inches(left), Inches(top), Inches(6.2), Inches(1.65), PAPER, LINE)
        _text(slide, Inches(left + 0.25), Inches(top + 0.2), Inches(5.7), Inches(0.4), title, 20, PPT_NAVY, True)
        _text(slide, Inches(left + 0.25), Inches(top + 0.7), Inches(5.7), Inches(0.75), body, 16, INK)
    _footer(slide, page, pages)


def _example(prs: Presentation, item: dict, record: dict[str, str], ordinal: int, page: int, pages: int) -> None:
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    color = SOLUTION_COLOR[item["solution"]]
    _box(slide, 0, 0, prs.slide_width, Inches(0.62), PPT_NAVY)
    _box(slide, 0, 0, Inches(0.12), Inches(0.62), color)
    _text(
        slide, Inches(0.32), Inches(0.12), Inches(10), Inches(0.4),
        f"{item['solution']}    ·    {ordinal} of 3    ·    {item['mode']} Bad Spot {item['n']}",
        20, PPT_WHITE, True,
    )
    picture = _map_path(item["mode"], item["n"])
    with Image.open(picture) as source:
        aspect = source.width / source.height
    image = _jpeg(picture)
    max_w, max_h = 6.9, 6.28
    width = max_w
    height = width / aspect
    if height > max_h:
        height = max_h
        width = height * aspect
    slide.shapes.add_picture(image, Inches(0.18), Inches(0.76), Inches(width), Inches(height))

    panel_left = 7.35
    _text(slide, Inches(panel_left), Inches(0.76), Inches(5.6), Inches(0.28), item["solution"].upper(), 12, color, True)
    _text(slide, Inches(panel_left), Inches(1.08), Inches(5.6), Inches(1.15), item["why"], 14, INK)

    fields = [
        ("Mode", item["mode"]),
        ("Bad spot", str(item["n"])),
        ("Site", record.get("C", "")),
        ("Sector", record.get("D", "")),
        ("Serving cell", record.get("E", "")),
        ("Population", record.get("F", "")),
    ]
    top = 2.32
    for label, value in fields:
        _text(slide, Inches(panel_left), Inches(top), Inches(1.45), Inches(0.32), label, 12, MUTED, True)
        _text(slide, Inches(panel_left + 1.5), Inches(top), Inches(3.8), Inches(0.32), value, 13, INK)
        top += 0.32
    _text(slide, Inches(panel_left), Inches(top + 0.08), Inches(5.2), Inches(0.28), "FINAL SOLUTION", 12, color, True)
    _box(slide, Inches(panel_left), Inches(top + 0.38), Inches(5.6), Inches(2.35), PAPER, LINE)
    _text(slide, Inches(panel_left + 0.14), Inches(top + 0.48), Inches(5.3), Inches(2.15), record.get("H", ""), 13, INK)
    _footer(slide, page, pages)


def _close(prs: Presentation, page: int, pages: int) -> None:
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    _box(slide, 0, 0, prs.slide_width, prs.slide_height, PPT_NAVY)
    _text(slide, Inches(0.7), Inches(1.5), Inches(12), Inches(0.8), "The full set is in the workbook", 32, PPT_WHITE, True)
    lines = [
        "Excel v1.63 has a map, a suggestion table and a movable circle for all 26 idle spots and all 99 active spots.",
        "The Cover sheet lists every spot, with the sector, the serving cell and the final solution.",
        "EP Details is the sector and cell tracker: existing height, azimuth, E_tilt and M_tilt beside the proposed values.",
        "The web map is v1.64. Site names are white with a dark edge. With KPI set to Blank, Info on a sector or the site centre opens the cell table.",
    ]
    top = 2.6
    for line in lines:
        _text(slide, Inches(0.7), Inches(top), Inches(11.8), Inches(0.7), line, 18, RGBColor(0xD6, 0xE4, 0xF0))
        top += 0.85


def build_summary_deck() -> Path:
    if not REPORT.exists():
        raise SystemExit(f"Missing {REPORT.name}")
    rows = _cover_rows()
    missing = []
    for item in EXAMPLES:
        if (item["mode"], item["n"]) not in rows:
            missing.append(f"{item['mode']} {item['n']}")
        if not _map_path(item["mode"], item["n"]).exists():
            missing.append(f"map {item['mode']} {item['n']}")
    if missing:
        raise SystemExit("Missing example data: " + ", ".join(missing))

    prs = Presentation()
    prs.slide_width = Inches(13.333)
    prs.slide_height = Inches(7.5)
    prs.core_properties.title = "Bogura Drive Test Summary v1.65"
    prs.core_properties.subject = "Three examples of each bad-spot solution"
    pages = 5 + len(EXAMPLES) + 1
    _title(prs)
    _measured(prs, 2, pages)
    _rules(prs, 3, pages)
    _counts(prs, 4, pages)
    _legend(prs, 5, pages)
    seen: dict[str, int] = {}
    for item in EXAMPLES:
        seen[item["solution"]] = seen.get(item["solution"], 0) + 1
        _example(prs, item, rows[(item["mode"], item["n"])], seen[item["solution"]], 5 + seen_total(seen), pages)
    _close(prs, pages, pages)
    OUTPUT.mkdir(parents=True, exist_ok=True)
    prs.save(DEST)
    return DEST


def seen_total(seen: dict[str, int]) -> int:
    return sum(seen.values())


## 24. Build the report

Run every cell above first. Then run this cell.

The workbook is written in three steps because each step rewrites the file. The web map and the summary deck run after the workbook.


In [ ]:
# ------------------------------------------------------------------------
# 24. Build the report
# Step 1 merges the logs and writes the workbook.
# Step 2 writes the solution text.
# Step 3 redraws the bad-spot maps.
# Step 4 builds the web zip. Step 5 builds the summary deck.
# ------------------------------------------------------------------------

if MERGED_CSV.exists() and ACTIVE_MERGED.exists():
    idle = load_merged_csv()
    active = load_active_merged()
else:
    extract_archives()
    idle = merge_parts()
    active = merge_active()

write_summary(idle)
plot_all(idle)
report_path = build_report(idle, active)
check = verify_report(report_path)
print(check)
if check["problems"]:
    raise SystemExit("Excel view checks failed: " + "; ".join(check["problems"]))

write_bad_spot_advice()
refresh_bad_spot_pies()

build_web_page()
add_cell_params()
add_site_pies()
package_web()

summary_path = build_summary_deck()
print("Summary:", summary_path)
